In [1]:
# ==========================================================
# CELL 0 / 환경 설정 + 상수 정의
# 목적: 필요한 패키지 설치/임포트, 사용할 파일 경로와 샘플 크기 설정
# ==========================================================

!pip install -q datasets pandas pyarrow

from datasets import load_dataset
import pandas as pd
import os, sys

# ---- 출력/캐시 경로: Colab이면 /content, 로컬이면 ./outputs ----
# 환경변수 BEAUTY_BASE_DIR로 바꿀 수 있다. 이후 셀은 모두 BASE_DIR 기준으로 저장한다.
IN_COLAB = "google.colab" in sys.modules
BASE_DIR = os.environ.get("BEAUTY_BASE_DIR", "/content" if IN_COLAB else os.path.abspath("outputs"))
os.makedirs(BASE_DIR, exist_ok=True)

# ---- Hugging Face 공개 경로 (Beauty) ----
META_URL = "https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw_meta_All_Beauty/full-00000-of-00001.parquet"
REV_URL  = "https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/review_categories/All_Beauty.jsonl"

# ---- 데모용 샘플 크기 (Colab 안정성 고려해서 작게 시작) ----
N_META_SAMPLE   = 15045     # 상품 메타 최대 n건
N_REVIEW_SAMPLE = 12358    # 리뷰 최대 n건

# ---- pandas 출력 옵션 (보고용 가독성) ----
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_columns", 40)

print("SETUP DONE / META_URL and REV_URL are set for Beauty")

SETUP DONE / META_URL and REV_URL are set for Beauty


In [2]:
# ==========================================================
# CELL 1 / 데이터 로드 + 기본 점검
# 목적: Beauty 메타/리뷰를 HF에서 바로 로드하고, 샘플링 및 컬럼 확인
# 포인트: dataset scripts 사용하지 않고 parquet/jsonl를 직접 지정
# ==========================================================

print("▶ Loading Beauty meta/reviews from Hugging Face ...")

# 1) Beauty 메타(상품 정보) — Parquet
meta_ds = load_dataset(
    "parquet",
    data_files=META_URL,
    split="train",
)
print(f"Meta rows total: {len(meta_ds):,}")

# 2) Beauty 리뷰 — JSONL
reviews_ds = load_dataset(
    "json",
    data_files=REV_URL,
    split="train",
)
print(f"Review rows total: {len(reviews_ds):,}")

# 3) 샘플링 (Colab 메모리 보호용)
meta_ds    = meta_ds.select(range(min(N_META_SAMPLE, len(meta_ds))))
reviews_ds = reviews_ds.select(range(min(N_REVIEW_SAMPLE, len(reviews_ds))))
print(f"Sampled -> Meta: {len(meta_ds):,} / Review: {len(reviews_ds):,}")

# 4) 컬럼 이름 빠르게 확인
print("\n[Meta columns]")
print(meta_ds.column_names)
print("\n[Review columns]")
print(reviews_ds.column_names)

# 5) 일부 레코드 미리보기 (보고용)
print("\n[Meta peek]")
print(meta_ds.select(range(min(3, len(meta_ds)))).to_pandas())

print("\n[Review peek]")
print(reviews_ds.select(range(min(3, len(reviews_ds)))).to_pandas()[["parent_asin","user_id","rating","title","text"]])


▶ Loading Beauty meta/reviews from Hugging Face ...
Meta rows total: 112,590
Review rows total: 701,528
Sampled -> Meta: 15,045 / Review: 12,358

[Meta columns]
['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together', 'subtitle', 'author']

[Review columns]
['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']

[Meta peek]
  main_category  \
0    All Beauty   
1    All Beauty   
2    All Beauty   

                                                                                                                                             title  \
0                                                                                              Howard LC0008 Leather Conditioner, 8-Ounce (4-Pack)   
1  Yes to Tomatoes Detoxifying Charcoal Cleanser (Pack of 2) with Charcoal Powder, Tomato Fruit Extra

In [3]:
# ==========================================================
# CELL 2 / 컬럼 매핑 + pandas 변환 / Column mapping + to-pandas
# 목적: HF Dataset(meta_ds/reviews_ds)을 표준 스키마로 매핑해 pandas DataFrame으로 변환
# 입력: meta_ds, reviews_ds (CELL 1에서 로드/샘플링 완료 상태)
# 출력: meta_df, rev_df (표준 컬럼만 포함한 DataFrame)
# ==========================================================

# --- 0) 선행 체크 / Pre-check ---
if "meta_ds" not in locals() or "reviews_ds" not in locals():
    raise RuntimeError("CELL 1이 실행되어 meta_ds/reviews_ds가 메모리에 있어야 합니다. 먼저 CELL 1을 실행하세요.")

# --- 1) 표준 스키마 정의 / Standard schema with candidate names ---
# 주: 일부 카테고리/파일에서 필드명이 다를 수 있어 후보명을 여러 개 둡니다.
META_SPECS = {
    "parent_asin":   ["parent_asin", "parent-id", "parentId"],
    "title":         ["title", "item_name", "name"],
    "brand":         ["brand", "by", "manufacturer"],
    "main_category": ["main_category", "main_cat", "category"],
    "categories":    ["categories", "category_path"],
    "details":       ["details"],
    "feature":       ["feature", "features"],
    "description":   ["description", "desc"],
    "price":         ["price"],
    "store":         ["store", "seller"]
}

REV_SPECS = {
    "parent_asin":   ["parent_asin", "parent-id"],
    "user_id":       ["user_id", "user", "reviewer_id"],
    "rating":        ["rating", "stars", "overall"],
    "title":         ["title", "review_title"],
    "text":          ["text", "review_text", "content"],
    "images":        ["images"],
    "timestamp":     ["timestamp", "unixReviewTime", "time"],
    "verified":      ["verified", "verified_purchase"],
    "helpful_votes": ["helpful_vote", "helpful_votes", "helpful"]
}

def resolve_actual_columns(df_columns, spec_dict):
    """
    각 표준 컬럼(std_name)에 대해 실제 존재하는 첫 후보 컬럼명을 매핑합니다.
    존재하지 않으면 None을 매핑해 이후에 빈 컬럼을 채웁니다.
    """
    actual = {}
    for std_name, candidates in spec_dict.items():
        chosen = None
        for c in candidates:
            if c in df_columns:
                chosen = c
                break
        actual[std_name] = chosen
    return actual

def build_standard_dataframe(ds, spec_dict):
    """
    HF datasets.Dataset → pandas.DataFrame 변환 후 표준 스키마로 재구성
    - 없는 컬럼은 NaN으로 생성
    - 반환: 표준 컬럼 순서가 유지된 DataFrame
    """
    df = ds.to_pandas()  # CELL 1에서 이미 샘플링했으므로 to_pandas 안전
    actual = resolve_actual_columns(df.columns, spec_dict)

    out = {}
    for std_name, real_name in actual.items():
        if real_name is None:
            # 후보명 어느 것도 없으면 빈 컬럼 생성
            out[std_name] = pd.Series([None] * len(df), index=df.index)
        else:
            out[std_name] = df[real_name]
    return pd.DataFrame(out)

# --- 2) 변환 실행 / Convert datasets to standardized DataFrames ---
print("▶ Converting datasets to pandas with standardized columns ...")
meta_df = build_standard_dataframe(meta_ds, META_SPECS)
rev_df  = build_standard_dataframe(reviews_ds, REV_SPECS)

# --- 3) 기본 점검 / Quick sanity checks ---
print(f"meta_df shape: {meta_df.shape} / rev_df shape: {rev_df.shape}")
print("\n[meta_df columns]", list(meta_df.columns))
print("[rev_df columns]",  list(rev_df.columns))

# 샘플 확인 / Peek few rows (보고용)
print("\n[meta_df.sample(3)]")
display(meta_df.sample(min(3, len(meta_df))) if len(meta_df) else meta_df.head(0))

print("\n[rev_df.sample(3)]")
show_cols = [c for c in ["parent_asin","user_id","rating","title","text"] if c in rev_df.columns]
display(rev_df[show_cols].sample(min(3, len(rev_df))) if len(rev_df) else rev_df.head(0))

# --- 4) 캐시 목적 선택적 저장 (선택사항) / Optional: cache to disk for later cells ---
# (원하지 않으면 주석 처리하세요)
# meta_df.to_parquet("/mnt/data/_cache_meta_df.parquet", index=False)
# rev_df.to_parquet("/mnt/data/_cache_rev_df.parquet", index=False)
# print("Cached: /mnt/data/_cache_meta_df.parquet / _cache_rev_df.parquet")


▶ Converting datasets to pandas with standardized columns ...
meta_df shape: (15045, 10) / rev_df shape: (12358, 9)

[meta_df columns] ['parent_asin', 'title', 'brand', 'main_category', 'categories', 'details', 'feature', 'description', 'price', 'store']
[rev_df columns] ['parent_asin', 'user_id', 'rating', 'title', 'text', 'images', 'timestamp', 'verified', 'helpful_votes']

[meta_df.sample(3)]


,parent_asin,title,brand,main_category,categories,details,feature,description,price,store
949,B01LF8FVQS,Linha Humor Natura - Agua de Colonia Masculina Humor a Dois (Humor 2) 75 Ml - (Natura Mood Collection - Mood For Two (Mood 2) Eau De Cologne For Men 2.53 Fl...,None,All Beauty,[],"{""Brand"": ""Natura"", ""Item Volume"": ""75 Milliliters"", ""Age Range (Description)"": ""Adult"", ""Style"": ""Modern"", ""Item Weight"": ""0.17 Pounds"", ""Is Discontinued B...","[Olfactive Family: Spicy Woody (Família olfativa : Amadeirada - Especiada), Top Notes: Black Pepper, Sicilian Lemon and Bergamot (Notas de Saída: Pimenta pr...","[Espalhe Seu Humor. O Provocante Do Picante Envolvidos Por Notas Amadeiradas Com Um Toque De Breu Branco E Priprioca, Ingredientes Naturais Da Biodiversidad...",37.7,natura
14746,B08HQWBDF5,SEGO Micro Link Hair Extensions Human Hair MicroBead Hair Extensions Highlight Pre Bonded Micro Ring Human Hair Extensions Straight Real Hairpiece 50g 100 S...,None,All Beauty,[],"{""Brand"": ""SEGO"", ""Material"": ""Human Hair"", ""Hair Type"": ""Straight"", ""Installation Type"": ""Fusion"", ""Package Dimensions"": ""12.28 x 7.95 x 1.18 inches; 3.53 ...",[],[],None,SEGO
9614,B07KMCJZVR,100% Remy Human Hair Silk Base Top Hairpieces Replacement Clip in Topper For Women Crown Top Piece Short 10''/10inch #27 Dark Blonde 20g,None,All Beauty,[],"{""Package Dimensions"": ""10.63 x 7.64 x 0.24 inches; 1.16 Ounces""}",[],[],None,None



[rev_df.sample(3)]


,parent_asin,user_id,rating,title,text
4794,B07S9H98QF,AGXHFUENM3G65MN4BGET2DNU4IAQ,5.0,Great remover,Great polish remover. I used it to remove three coats of polish and it took it off with one swipe. Recommended
2475,B00AR59PZ4,AHHENT2VWJDJ6PDVOJQJWYWYOGDA,1.0,Didn't see boosting effects at all. It doesn't even ...,"Didn't see boosting effects at all. It doesn't even spray, the liquid just dribbles out the sprayer and runs down the side of the bottle. Having a sprayer i..."
3342,B08HPSMC2C,AHVBPSAR5WEXUGWURWTF6YAGALMQ,5.0,During the mask up lock down days,"If you are going to wear a mask, better to match with your outfit. I always go in style, never look like a slob in public"


In [4]:
# ==========================================================
# CELL 2.5 / 서브카테고리 프리필터 (Beauty 핵심만 남기기)
# 목적: meta_df에서 Beauty 핵심 영역(Skin/Hair/Makeup/Fragrance/Body 등)만 남기고
#       문신/공구성 품목(tattoo/cartridge/plunger/needle 등)과 잡음을 사전 제거.
# 입력: CELL 2에서 생성된 meta_df, rev_df
# 출력: (덮어쓰기) meta_df, rev_df  -> 필터링된 데이터셋
#       (보고용) 필터 통계/예시 출력
# ==========================================================

# ---------- 0) 선행 체크 ----------
if "meta_df" not in locals() or "rev_df" not in locals():
    raise RuntimeError("CELL 2가 먼저 실행되어 meta_df/rev_df가 있어야 합니다.")

import os
import re
import pandas as pd
import numpy as np

# ---------- 1) 키워드 정책(필요 시 프로젝트 상황에 맞게 조정) ----------
# 허용(포함) 서브카테고리/키워드(카테고리 경로와 텍스트에서 찾음)
ALLOW_SUBCATS = [
    # 상위 영역
    "skin", "hair", "makeup", "cosmetic", "fragrance", "perfume", "body", "face", "nail",
    # 대표 제품유형 (누락 방지용 보강)
    "shampoo", "conditioner", "cleanser", "face wash", "toner", "serum", "essence", "ampoule",
    "moisturizer", "lotion", "cream", "gel", "mask", "sunscreen", "oil", "body wash", "body lotion",
    # (제출 후 추가) 단어 경계 매칭으로 바꾸면서 "skin"/"hair"가 더는 잡지 못하는 붙여 쓴 복합어
    "skincare", "haircare",
    # (제출 후 추가) 단어 경계 매칭으로 바꾼 뒤 새로 제외된 215개(prefilter_match_diff.csv)를 검토해,
    # 예전에는 부분 문자열로 우연히 남아 있던 뷰티 품목명을 추가 (향수 / 메이크업·네일 / 헤어 도구·액세서리)
    # 파우치, 칫솔, 면도기 헤드, 발톱 도구 등은 핵심 뷰티 범위가 아니라서 추가하지 않는다.
    "eau de toilette", "eau de parfum", "parfum", "cologne", "edp", "edt",
    "mascara", "concealer", "lipstick", "eyeliner", "eyeshadow", "foundation", "blush",
    "lip", "lash", "eyelash", "nail polish", "gel polish", "scrub",
    "hairband", "headband", "hairpiece", "hairpin", "hairclip", "hairspray", "haircolor", "comb", "brush",
]

# 차단(제외) 키워드(문신/공구/산업용 등 Beauty 시연에 부적합한 품목)
BLOCK_TERMS = [
    "tattoo", "cartridge", "plunger", "needle", "grip", "vice", "autoclave",
    "stainless steel", "brass tip", "industrial", "solder", "welding",
    # 가죽·가구 관리 용품 (제출 후 추가: 제출본에서는 별도 셀에 있어 실제로 적용되지 않았음)
    "upholstery", "furniture", "sofa", "car seat", "shoe polish", "saddle",
]
# 제출본 별도 셀에 있던 "leather", "vinyl", "restorer", "boot"는 일부러 넣지 않는다.
#  - leather : 가죽향 향수 등 뷰티 상품까지 제외되고, 아래 3-1) 맥락 규칙을 무의미하게 만든다
#  - vinyl   : 미용·위생용 vinyl 장갑이 제외된다
#  - restorer: "hair restorer" 같은 헤어 제품이 제외된다
#  - boot    : 부분 문자열 비교에서 "booty" 등에도 걸린다

# main_category가 Beauty가 맞는지 보조 점검(데이터에 따라 표기가 다를 수 있어 느슨하게)
def is_main_beauty(x: str) -> bool:
    if x is None:
        return False
    s = str(x).strip().lower()
    return ("beaut" in s)  # "beauty", "all beauty" 등 포괄

# 안전한 문자열화(리스트/딕셔너리 포함)
def to_safe_str(x):
    if x is None:
        return ""
    if isinstance(x, (list, tuple, set, np.ndarray, pd.Series)):
        try:
            return " / ".join(map(str, x))
        except Exception:
            return " / ".join([str(e) for e in list(x)])
    if isinstance(x, dict):
        try:
            return " ; ".join([f"{k}: {v}" for k, v in x.items()])
        except Exception:
            return str(x)
    return str(x)

def lower_str(x):
    return to_safe_str(x).lower()

def contains_any_substr(text: str, keywords) -> bool:
    """제출본 방식: 부분 문자열 비교. 아래 비교 리포트용으로만 남겨 둔다.
    "vice"가 device/service/advice에, "oil"이 toilet/foil에 걸리는 문제가 있었다."""
    return any(kw in text for kw in keywords)

def build_kw_regex(keywords):
    """(제출 후 수정) 단어 경계 기준 매칭.
    - 복수형 s/es 허용 (creams, lotions, needles)
    - 여러 단어 키워드는 사이 공백 1개 이상 허용 (face  wash)"""
    alts = sorted({re.escape(k).replace(r"\ ", r"\s+") for k in keywords}, key=len, reverse=True)
    return re.compile(r"\b(?:" + "|".join(alts) + r")(?:s|es)?\b")

ALLOW_RE = build_kw_regex(ALLOW_SUBCATS)
BLOCK_RE = build_kw_regex(BLOCK_TERMS)

def contains_any(text: str, pattern) -> bool:
    return bool(pattern.search(text))

# ---------- 2) 카테고리/텍스트 기반 필터 신호 만들기 ----------
df = meta_df.copy()

# 카테고리 문자열(중첩 리스트를 문자열로 평탄화)
cat_str = df.get("categories")
cat_str = cat_str.apply(lower_str) if cat_str is not None else pd.Series([""] * len(df))

# 메인카테고리 검사
main_ok = df.get("main_category").apply(is_main_beauty) if "main_category" in df.columns else pd.Series([True]*len(df))

# 간이 텍스트(제목/설명/피처/디테일)에서 조기 판별용 문자열 만들기
title_str = df.get("title").apply(lower_str) if "title" in df.columns else pd.Series([""] * len(df))
desc_str  = df.get("description").apply(lower_str) if "description" in df.columns else pd.Series([""] * len(df))
feat_str  = df.get("feature").apply(lower_str) if "feature" in df.columns else pd.Series([""] * len(df))
det_str   = df.get("details").apply(lower_str) if "details" in df.columns else pd.Series([""] * len(df))

quick_text = (title_str + " / " + desc_str + " / " + feat_str + " / " + det_str)

# 허용/차단 신호
allow_hit_cat   = cat_str.apply(lambda s: contains_any(s, ALLOW_RE))
allow_hit_text  = quick_text.apply(lambda s: contains_any(s, ALLOW_RE))
block_hit_text  = quick_text.apply(lambda s: contains_any(s, BLOCK_RE))

# 제출본(부분 문자열) 방식 신호: 매칭 방식 변경의 영향을 비교하는 용도
allow_hit_old = (cat_str.apply(lambda s: contains_any_substr(s, ALLOW_SUBCATS))
                 | quick_text.apply(lambda s: contains_any_substr(s, ALLOW_SUBCATS)))
block_hit_old = quick_text.apply(lambda s: contains_any_substr(s, BLOCK_TERMS))

# ---------- 3) 최종 보존 규칙 ----------
# 규칙 요약:
#  - main_category가 Beauty 계열 AND
#  - (카테고리에서 허용키워드 OR 텍스트에서 허용키워드) AND
#  - 텍스트에서 차단키워드가 없어야 함
keep_mask = (main_ok) & ((allow_hit_cat) | (allow_hit_text)) & (~block_hit_text)

# ---------- 3-1) 가죽 컨디셔너 맥락 규칙 ----------
# (제출 후 수정) 제출본에서는 이 규칙이 별도 셀에서 keep_mask만 다시 계산하고
# meta_df에 적용되지 않아, 가죽 관리 제품이 최종 CSV까지 남았다.
# "conditioner"는 헤어 제품과 가죽 관리 제품 양쪽에 쓰이므로,
# 'leather conditioner'가 있고 헤어 맥락(hair/scalp 등)이 없을 때만 제외한다.
LEATHER_COND_RE = re.compile(
    r"(?:\bleather\b[^\w]{0,3}\bcondit(?:ioner)?\b|\bcondit(?:ioner)?\b[^\w]{0,3}\bleather\b)",
    re.I
)
HAIR_CTX_RE = re.compile(r"\b(?:hair|scalp|shampoo|rinse|split\s*ends|leave[-\s]?in)\b", re.I)

leather_cond = quick_text.str.contains(LEATHER_COND_RE)
hair_ctx     = quick_text.str.contains(HAIR_CTX_RE)
leather_drop = leather_cond & ~hair_ctx
keep_mask    = keep_mask & ~leather_drop

# ---------- 4) 결과 적용 및 리뷰 동기화 ----------
before_n = len(df)
df_filt = df[keep_mask].reset_index(drop=True)
after_n  = len(df_filt)
removed  = before_n - after_n

# 리뷰도 parent_asin 기준으로 동기 필터
if "parent_asin" in df_filt.columns and "parent_asin" in rev_df.columns:
    keep_asins = set(df_filt["parent_asin"].dropna().astype(str))
    rev_df = rev_df[rev_df["parent_asin"].astype(str).isin(keep_asins)].reset_index(drop=True)

# 덮어쓰기(이후 셀에서 meta_df 사용)
meta_df = df_filt

# ---------- 5) 리포트 ----------
print("▶ CELL 2.5 서브카테고리 프리필터 적용")
print(f"  - 총 상품 수(이전): {before_n:,}")
print(f"  - 총 상품 수(이후): {after_n:,}")
print(f"  - 제거 수: {removed:,} ({(removed/max(1,before_n)*100):.1f}%)")
print("  - 제거 사유별 (한 상품이 여러 사유에 걸릴 수 있음):")
print(f"      main_category 불일치 : {int((~main_ok).sum()):,}")
print(f"      허용어 없음          : {int((~(allow_hit_cat | allow_hit_text)).sum()):,}")
print(f"      차단어 포함          : {int(block_hit_text.sum()):,}")
print(f"      가죽 컨디셔너 규칙   : {int(leather_drop.sum()):,}")

# 매칭 방식 변경 영향 (같은 키워드 목록으로 부분 문자열 vs 단어 경계 비교)
keep_old      = main_ok & allow_hit_old & ~block_hit_old & ~leather_drop
newly_kept    = keep_mask & ~keep_old
newly_removed = keep_old & ~keep_mask
print("\n[매칭 방식 변경 영향: 부분 문자열 → 단어 경계]")
print(f"  - 새로 남은 상품   : {int(newly_kept.sum()):,}  (예: 'vice'가 'device'에 걸려 잘못 제외되던 경우)")
print(f"  - 새로 제외된 상품 : {int(newly_removed.sum()):,}  (예: 'oil'이 'toilet'에 걸려 잘못 남던 경우)")
for label, m in [("새로 남은 예시", newly_kept), ("새로 제외된 예시", newly_removed)]:
    ex = title_str[m].head(5).tolist()
    if ex:
        print(f"  [{label}]")
        for t in ex:
            print(f"    - {t[:120]}")

# 판정이 바뀐 상품 전체 목록을 파일로 남긴다 (허용어/차단어 보강의 근거 자료)
_diff = pd.DataFrame({
    "parent_asin": df["parent_asin"] if "parent_asin" in df.columns else "",
    "change": np.where(newly_kept, "newly_kept", np.where(newly_removed, "newly_removed", "")),
    "title": title_str,
})
_diff = _diff[_diff["change"] != ""]
_diff_path = os.path.join(BASE_DIR, "prefilter_match_diff.csv")
_diff.to_csv(_diff_path, index=False)
print(f"  - 전체 목록 저장: {_diff_path} ({len(_diff):,}행)")
if "parent_asin" in meta_df.columns and "parent_asin" in rev_df.columns:
    print(f"  - 연동된 리뷰 수(이후): {len(rev_df):,}")

# 허용/차단 신호 비율(참고)
print("\n[허용/차단 신호 비율]")
print(f"  - allow_hit_cat  : {(allow_hit_cat.mean()*100):.1f}%")
print(f"  - allow_hit_text : {(allow_hit_text.mean()*100):.1f}%")
print(f"  - block_hit_text : {(block_hit_text.mean()*100):.1f}%")

# 카테고리 상위 토큰(간단 뷰)
print("\n[카테고리 문자열 상위 예시 3건]")
print(cat_str.head(3).tolist())

# 차단으로 제거된 대표 예시(최대 5건) — 디버깅용
try:
    removed_examples = df[~keep_mask].copy()
    if len(removed_examples):
        cols = [c for c in ["title","categories","details","feature","description"] if c in removed_examples.columns]
        print("\n[제거 예시 3건]")
        display(removed_examples[cols].head(3))
except Exception:
    pass


▶ CELL 2.5 서브카테고리 프리필터 적용
  - 총 상품 수(이전): 15,045
  - 총 상품 수(이후): 12,402
  - 제거 수: 2,643 (17.6%)
  - 제거 사유별 (한 상품이 여러 사유에 걸릴 수 있음):
      main_category 불일치 : 0
      허용어 없음          : 2,050
      차단어 포함          : 790
      가죽 컨디셔너 규칙   : 3

[매칭 방식 변경 영향: 부분 문자열 → 단어 경계]
  - 새로 남은 상품   : 203  (예: 'vice'가 'device'에 걸려 잘못 제외되던 경우)
  - 새로 제외된 상품 : 266  (예: 'oil'이 'toilet'에 걸려 잘못 남던 경우)
  [새로 남은 예시]
    - false nails for kids 24 pcs press on girls fake nails tips watermenlon pattern
    - nook no yellow"the service color" silver shampoo 300 ml.
    - kayline chemical service station in black fold-a-way service tray (model: ft59)
    - african traditional wax print head wrap scarf, women multi-color headband (tj3)
    - tape in hair extensions 20 inch, 100% remy human hair, natural seamless long silk straight #1 jet black 50g 20pcs hair e
  [새로 제외된 예시]
    - small money clip leather wallet id bag cash holder credit card cover case pouch
    - kosher complete lulav and esrog etrog set
    - j

,title,categories,details,feature,description
0,"Howard LC0008 Leather Conditioner, 8-Ounce (4-Pack)",[],"{""Package Dimensions"": ""7.1 x 5.5 x 3 inches; 2.38 Pounds"", ""UPC"": ""617390882781""}",[],[]
2,Eye Patch Black Adult with Tie Band (6 Per Pack),[],"{""Manufacturer"": ""Levine Health Products""}",[],[]
3,"Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4D Imitation Eyebrow Tattoos, 4D Hair-like Authentic Eyebrows Waterproof Long Lasting for Woman & Man Makeup Tool",[],"{""Brand"": ""Cherioll"", ""Item Form"": ""Powder"", ""Finish Type"": ""Natural"", ""Product Benefits"": ""Long Lasting"", ""Skin Type"": ""All"", ""Package Dimensions"": ""8.43 x...",[],[]


In [5]:
# ==========================================================
# CELL 3 / 전처리 1단계 (숫자화·정규화)
# 목적: meta_df/rev_df의 핵심 필드를 일관된 타입으로 정리
#      - details/feature JSON 파싱
#      - price 숫자화
#      - 리뷰 timestamp/verified/helpful 정규화
# 입력: (CELL 2.5 이후) meta_df, rev_df
# 출력: meta_df_clean, rev_df_clean
# ==========================================================

# ---------- 0) 선행 체크 ----------
if "meta_df" not in locals() or "rev_df" not in locals():
    raise RuntimeError("meta_df / rev_df가 없습니다. CELL 2.5까지 먼저 실행하세요.")

import json, math
import pandas as pd
import numpy as np

# ---------- 1) 유틸 함수 ----------
def parse_maybe_json(x):
    """문자열이 JSON 형태면 파싱, dict/list면 그대로, 그 외는 원본 유지"""
    if x is None or isinstance(x, (dict, list)):
        return x
    if isinstance(x, str):
        s = x.strip()
        if (s.startswith("{") and s.endswith("}")) or (s.startswith("[") and s.endswith("]")):
            try:
                return json.loads(s)
            except Exception:
                return x
    return x

def to_float_or_none(x):
    """'$12.99', '12,345', 'None' 등 혼재값 → float 또는 None"""
    if x is None:
        return None
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)
    s = str(x).strip()
    if s.lower() in {"none", "nan", ""}:
        return None
    s = s.replace("$", "").replace(",", "")
    try:
        return float(s)
    except Exception:
        return None

def to_int_or_none(x):
    """정수 변환(실패 시 None)"""
    try:
        if x is None or (isinstance(x, float) and math.isnan(x)):
            return None
        return int(float(x))
    except Exception:
        return None

def normalize_timestamp_seconds(x):
    """리뷰 timestamp를 초 단위 정수로 통일(ms로 크면 /1000)"""
    ts = to_int_or_none(x)
    if ts is None:
        return None
    if ts >= 10**12:  # ms로 보이는 경우
        ts //= 1000
    return ts

def to_bool_or_none(x):
    """'true/1/yes'/'false/0/no' 등 불리언 정규화"""
    if isinstance(x, bool):
        return x
    if x is None:
        return None
    s = str(x).strip().lower()
    if s in {"true", "1", "yes", "y"}:
        return True
    if s in {"false", "0", "no", "n"}:
        return False
    return None

def to_int_nonneg(x):
    """도움됨 수 등 음수 방지 정수(없거나 음수면 0)"""
    v = to_int_or_none(x)
    if v is None or v < 0:
        return 0
    return v

def valid_ratio(series):
    n = series.notna().sum()
    t = len(series)
    return f"{n}/{t} ({(n/t*100 if t else 0):.1f}%)"

# ---------- 2) 메타 전처리 ----------
meta_df_clean = meta_df.copy()

# (a) details/feature JSON 파싱(행 단위)
if "details" in meta_df_clean.columns:
    meta_df_clean["details"] = meta_df_clean["details"].apply(parse_maybe_json)
if "feature" in meta_df_clean.columns:
    meta_df_clean["feature"] = meta_df_clean["feature"].apply(parse_maybe_json)

# (b) price → price_num(float)
if "price" in meta_df_clean.columns:
    meta_df_clean["price_num"] = meta_df_clean["price"].apply(to_float_or_none)
else:
    meta_df_clean["price_num"] = None

# (c) 문자열 컬럼 기본 정리
for col in ["title", "brand", "description", "store", "main_category"]:
    if col in meta_df_clean.columns:
        meta_df_clean[col] = meta_df_clean[col].fillna("").astype(str).str.strip()

# ---------- 3) 리뷰 전처리 ----------
rev_df_clean = rev_df.copy()

# (a) 평점 숫자화
if "rating" in rev_df_clean.columns:
    rev_df_clean["rating_num"] = rev_df_clean["rating"].apply(to_float_or_none)
else:
    rev_df_clean["rating_num"] = None

# (b) timestamp 초 단위 + datetime 생성
if "timestamp" in rev_df_clean.columns:
    rev_df_clean["timestamp_sec"] = rev_df_clean["timestamp"].apply(normalize_timestamp_seconds)
    rev_df_clean["ts_datetime"]   = pd.to_datetime(rev_df_clean["timestamp_sec"], unit="s", errors="coerce")
else:
    rev_df_clean["timestamp_sec"] = None
    rev_df_clean["ts_datetime"]   = None

# (c) verified 정규화
if "verified" in rev_df_clean.columns:
    rev_df_clean["verified_bool"] = rev_df_clean["verified"].apply(to_bool_or_none)
else:
    rev_df_clean["verified_bool"] = None

# (d) helpful_votes 정리
if "helpful_votes" in rev_df_clean.columns:
    rev_df_clean["helpful_votes_int"] = rev_df_clean["helpful_votes"].apply(to_int_nonneg)
else:
    rev_df_clean["helpful_votes_int"] = 0

# (e) 리뷰 제목/본문 문자열 정리
for col in ["title", "text"]:
    if col in rev_df_clean.columns:
        rev_df_clean[col] = rev_df_clean[col].fillna("").astype(str).str.strip()

# ---------- 4) 요약 리포트 ----------
print("▶ meta_df_clean:", meta_df_clean.shape)
if "price_num" in meta_df_clean.columns:
    print("  - price_num 유효비율:", valid_ratio(meta_df_clean["price_num"]))
if "details" in meta_df_clean.columns:
    types = meta_df_clean["details"].map(lambda v: type(v).__name__)
    print("  - details 타입 분포:", dict(types.value_counts().head(3)))

print("\n▶ rev_df_clean:", rev_df_clean.shape)
print("  - rating_num 유효비율:", valid_ratio(rev_df_clean["rating_num"]))
print("  - timestamp_sec 유효비율:", valid_ratio(rev_df_clean["timestamp_sec"]))
print("  - verified_bool 유효비율:", valid_ratio(rev_df_clean["verified_bool"]))
print("  - helpful_votes 합계:", int(rev_df_clean["helpful_votes_int"].sum() if len(rev_df_clean) else 0))

# ---------- 5) 미리보기 ----------
try:
    cols_meta_peek = [c for c in ["parent_asin","title","brand","price","price_num","main_category"] if c in meta_df_clean.columns]
    print("\n[meta_df_clean preview]")
    display(meta_df_clean[cols_meta_peek].head(3))
except Exception:
    pass

try:
    cols_rev_peek = [c for c in ["parent_asin","user_id","rating","rating_num","timestamp","timestamp_sec","verified","verified_bool","helpful_votes","helpful_votes_int","title","text"] if c in rev_df_clean.columns]
    print("\n[rev_df_clean preview]")
    display(rev_df_clean[cols_rev_peek].head(3))
except Exception:
    pass

# (선택) 캐시 저장
# meta_df_clean.to_parquet("/mnt/data/_cache_meta_df_clean.parquet", index=False)
# rev_df_clean.to_parquet("/mnt/data/_cache_rev_df_clean.parquet", index=False)
# print("Saved cache: _cache_meta_df_clean.parquet / _cache_rev_df_clean.parquet")


▶ meta_df_clean: (12402, 11)
  - price_num 유효비율: 2249/12402 (18.1%)
  - details 타입 분포: {'dict': np.int64(12402)}

▶ rev_df_clean: (1570, 14)
  - rating_num 유효비율: 1570/1570 (100.0%)
  - timestamp_sec 유효비율: 1570/1570 (100.0%)
  - verified_bool 유효비율: 1570/1570 (100.0%)
  - helpful_votes 합계: 1643

[meta_df_clean preview]


,parent_asin,title,brand,price,price_num,main_category
0,B076WQZGPM,"Yes to Tomatoes Detoxifying Charcoal Cleanser (Pack of 2) with Charcoal Powder, Tomato Fruit Extract, and Gingko Biloba Leaf Extract, 5 fl. oz.",,None,NaN,All Beauty
1,B07G9GWFSM,Lurrose 100Pcs Full Cover Fake Toenails Artificial Transparent Nail Tips Nail Art for DIY,,6.99,6.99,All Beauty
2,B08DNQTTQK,50 Pieces False Eyelash Packaging Box Empty Eyelash Box Plastic Eyelash Storage Case with Glitter Paper and Clear Tray for Women Girls Eyelash (Holographic),,None,NaN,All Beauty



[rev_df_clean preview]


,parent_asin,user_id,rating,rating_num,timestamp,timestamp_sec,verified,verified_bool,helpful_votes,helpful_votes_int,title,text
0,B08BBQ29N5,AEYORY2AVPMCPDV57CE337YU5LXA,3.0,3.0,1634275259292,1634275259,True,True,0,0,Meh,These were lightweight and soft but much too small for my liking. I would have preferred two of these together to make one loc. For that reason I will not b...
1,B07KG1TWP5,AFSKPY37N3C43SOI5IEXEK5JSIYA,5.0,5.0,1596473351088,1596473351,False,False,0,0,Great all natural ear swabs!,"I really like these ear swabs. First they come in a large, handy box and are easy to store (will last a long time). Second, they are all organic and good fo..."
2,B07W397QG4,AFSKPY37N3C43SOI5IEXEK5JSIYA,5.0,5.0,1593352422858,1593352422,False,False,1,1,Wonderful overnight cream!,"To be honest, I rarely have used an overnight cream. Typically, my skin care routine is the same morning and night (wash, apply hyalauronic serum, and some ..."


In [6]:
# ==========================================================
# CELL 4 / raw_text 통합 생성 (title+feature+description+details)
# 목적: 텍스트 기반 속성추출·임베딩에 사용할 'raw_text' 필드 생성
# 입력: meta_df_clean (CELL 3 결과)
# 출력: meta_df_text (추가 컬럼: feature_str/details_str/raw_text/raw_text_len/title_norm/brand_norm)
# ==========================================================

# ---------- 0) 선행 체크 ----------
if "meta_df_clean" not in locals():
    raise RuntimeError("meta_df_clean이 없습니다. CELL 3까지 먼저 실행하세요.")

import re, json
import numpy as np
import pandas as pd

# ---------- 1) 유틸 함수: HTML/nbsp 제거 ----------
def strip_html_and_nbsp(s: str) -> str:
    if not isinstance(s, str):
        s = "" if s is None else str(s)
    s = s.replace("\xa0", " ")
    s = re.sub(r"<br\s*/?>", " ", s, flags=re.IGNORECASE)
    s = re.sub(r"<[^>]+>", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# ---------- 2) 유틸 함수: dict → "k: v" 문자열 ----------
def dict_to_kv_string(d: dict) -> str:
    parts = []
    for k, v in d.items():
        ks = str(k).strip()
        if not ks or ks.isdigit():             # 0/1/2 같은 인덱스 키는 제외
            continue
        if v in (None, "", [], {}):
            continue
        if isinstance(v, (list, tuple, set, np.ndarray, pd.Series)):
            vv = [str(t).strip() for t in list(v) if str(t).strip()]
            if vv:
                parts.append(f"{ks}: {', '.join(vv)}")
        else:
            parts.append(f"{ks}: {str(v).strip()}")
    return "; ".join(parts)

# ---------- 3) 유틸 함수: 행 단위 안전 문자열화 ----------
def stringify_for_text(obj):
    """
    행(row) 단위로 호출됨.
    - dict  : "k: v; k2: v2"
    - 시퀀스: 각 원소를 문자열화 후 '; '로 연결
    - 기타  : str() 후 HTML/nbsp/공백 정리
    """
    if obj is None:
        return ""
    if isinstance(obj, str):
        return strip_html_and_nbsp(obj)

    if isinstance(obj, pd.Series):
        obj = obj.tolist()
    if isinstance(obj, np.ndarray):
        obj = obj.tolist()

    if isinstance(obj, dict):
        return strip_html_and_nbsp(dict_to_kv_string(obj))

    if isinstance(obj, (list, tuple, set)):
        items = []
        for x in obj:
            if x in (None, "", [], {}):
                continue
            if isinstance(x, dict):
                s = dict_to_kv_string(x)
            elif isinstance(x, (list, tuple, set, np.ndarray, pd.Series)):
                flat = list(x) if not isinstance(x, (np.ndarray, pd.Series)) else list(x.tolist())
                s = ", ".join([str(t).strip() for t in flat if str(t).strip()])
            else:
                s = str(x).strip()
            s = strip_html_and_nbsp(s)
            if s:
                items.append(s)
        return "; ".join(items)

    return strip_html_and_nbsp(str(obj))

# ---------- 4) 유틸 함수: 대괄호/인덱스 노이즈 정리 ----------
def cleanup_bracket_noise_series(s: pd.Series) -> pd.Series:
    return (
        s
        .str.replace(r"\b\d+\s*\[\s*\]", " ", regex=True)  # "0 []" 등 제거
        .str.replace(r"\[\s*\]", " ", regex=True)          # 빈 대괄호 제거
        .str.replace(r"\s*;\s*", "; ", regex=True)         # 세미콜론 주변 공백 정리
        .str.replace(r"\s{2,}", " ", regex=True)           # 다중 공백 정리
        .str.strip()
    )

# --- 5) 원본 복사 및 기본 문자열 컬럼 정리 (수정) ---
meta_df_text = meta_df_clean.copy()

# title은 문자열 캐스팅, description은 행단위 안전 문자열화로 변경
meta_df_text["title"] = meta_df_text.get("title", "").fillna("").astype(str)
if "description" in meta_df_text.columns:
    meta_df_text["description"] = meta_df_text["description"].apply(stringify_for_text)
else:
    meta_df_text["description"] = ""

# --- 6) feature/details를 '행 단위'로 문자열화 (동일) ---
meta_df_text["feature_str"] = (
    meta_df_text["feature"].apply(stringify_for_text) if "feature" in meta_df_text.columns else ""
)
meta_df_text["details_str"] = (
    meta_df_text["details"].apply(stringify_for_text) if "details" in meta_df_text.columns else ""
)

# --- 7) raw_text 구성: 빈 세그먼트는 자동 제외 + 소문자화 + 노이즈 정리 (수정) ---
def join_nonempty(parts):
    parts = [p.strip() for p in parts if isinstance(p, str) and p.strip()]
    return " / ".join(parts)

meta_df_text["raw_text"] = meta_df_text.apply(
    lambda r: join_nonempty([r["title"], r["feature_str"], r["description"], r["details_str"]]),
    axis=1
).str.lower()

# 헤더 토큰(‘description/feature/package including’) 제거
meta_df_text["raw_text"] = meta_df_text["raw_text"].str.replace(
    r"\b(description|feature|package including)\b\s*[:\-]?\s*", "", regex=True
)

# 대괄호·중복 공백 정리
meta_df_text["raw_text"] = cleanup_bracket_noise_series(meta_df_text["raw_text"])

# 길이/보조 컬럼
MAX_LEN = 4000
meta_df_text["raw_text"] = meta_df_text["raw_text"].str.slice(0, MAX_LEN)
meta_df_text["raw_text_len"] = meta_df_text["raw_text"].str.len()

# --- 8) 보조 정규화 컬럼(브랜드/제목) (동일) ---
meta_df_text["title_norm"] = meta_df_text["title"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
# (제출 후 수정) 이 데이터셋 메타에는 brand 열이 없다(CELL 2의 후보 brand/by/manufacturer가 모두 없음).
# 그래서 제출본에서는 brand가 전부 비어 최종 CSV의 brand 커버리지가 0%였다.
# 비어 있으면 details의 "Brand" 키 → store 열 → details의 "Manufacturer" 키 순서로 채운다.
BRAND_EMPTY_VALUES = {"", "generic", "unknown", "none", "nan", "n/a", "na", "unbranded", "no brand"}

def _clean_brand_value(v) -> str:
    s = "" if v is None else str(v).strip()
    return "" if s.lower() in BRAND_EMPTY_VALUES else s

def _detail_value(d, key) -> str:
    return _clean_brand_value(d.get(key)) if isinstance(d, dict) else ""

_n = len(meta_df_text)
if "brand" not in meta_df_text.columns:
    meta_df_text["brand"] = ""
meta_df_text["brand"] = meta_df_text["brand"].map(_clean_brand_value)
meta_df_text["brand_source"] = np.where(meta_df_text["brand"] != "", "meta_brand", "")

_details = meta_df_text["details"] if "details" in meta_df_text.columns else pd.Series([{}] * _n, index=meta_df_text.index)
_store   = meta_df_text["store"].map(_clean_brand_value) if "store" in meta_df_text.columns else pd.Series([""] * _n, index=meta_df_text.index)
for _src, _vals in [("details.Brand",        _details.map(lambda d: _detail_value(d, "Brand"))),
                    ("store",                _store),
                    ("details.Manufacturer", _details.map(lambda d: _detail_value(d, "Manufacturer")))]:
    _fill = (meta_df_text["brand"] == "") & (_vals != "")
    meta_df_text.loc[_fill, "brand"] = _vals[_fill]
    meta_df_text.loc[_fill, "brand_source"] = _src

meta_df_text["brand_norm"] = meta_df_text["brand"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
print("- brand 출처별 상품 수:", meta_df_text["brand_source"].replace("", "(없음)").value_counts().to_dict())

# 품질 점검
with pd.option_context("display.max_colwidth", None):
    display(meta_df_text[["parent_asin","raw_text_len","raw_text"]].head(3))


# ---------- 9) 품질 점검 출력 ----------
def pct(n, d):
    return f"{(n/d*100):.1f}%" if d else "0.0%"

n_total = len(meta_df_text)
n_empty_raw = int((meta_df_text["raw_text"].str.len() == 0).sum())
print("▶ raw_text 생성 완료")
print(f"  - rows: {n_total:,}")
print(f"  - empty raw_text: {n_empty_raw:,} ({pct(n_empty_raw, n_total)})")
print(f"  - avg raw_text_len: {meta_df_text['raw_text_len'].mean():.1f}")

# 미리보기(상위 3행, 내용 생략 없이 보기)
with pd.option_context("display.max_colwidth", None):
    cols_peek = [c for c in ["parent_asin","title","brand","price_num","raw_text_len","raw_text"] if c in meta_df_text.columns]
    print("\n[meta_df_text preview]")
    display(meta_df_text[cols_peek].head(3))


- brand 출처별 상품 수: {'details.Brand': 9158, 'store': 2112, '(없음)': 1120, 'details.Manufacturer': 12}


,parent_asin,raw_text_len,raw_text
0,B076WQZGPM,370,"yes to tomatoes detoxifying charcoal cleanser (pack of 2) with charcoal powder, tomato fruit extract, and gingko biloba leaf extract, 5 fl. oz. / / item form: powder; skin type: acne prone; brand: yes to; age range (): adult; unit count: 10 fl oz; is discontinued by manufacturer: no; item model number: sg_b076wqzgpm_us; upc: 653801351125; manufacturer: yes to tomatoes"
1,B07G9GWFSM,1414,"lurrose 100pcs full cover fake toenails artificial transparent nail tips nail art for diy / the false toenails are durable with perfect length. you have the option to wear them long or clip them short, easy to trim and file them to in any length and shape you like.; abs is kind of green enviromental material, and makes the nails durable, breathable, light even no pressure on your own nails.; fit well to your natural toenails. non toxic, no smell, no harm to your health.; wonderful as gift for girlfriend, family and friends.; the easiest and most efficient way to do your toenail tips for manicures or nail art designs. it's fashion, creative, a useful accessory brighten up your look, also as a gift. / ['' 'the false toenails are durable with perfect length. you have the option to wear them long or clip them short, easy to trim and file them to in any length and shape you like. plus, abs is kind of green enviromental material, and makes the nails durable, breathable, light even no pressure on your own toenails. fit well to your natural toenails. non toxic, no smell, no harm to your health.' '' '- color: as shown.- material: abs.- size: 14.3 x 7.2 x 1cm.' '' '100 x pieces fake toenails'] / color: as shown; size: large; material: acrylonitrile butadiene styrene (abs); brand: lurrose; style: french; product dimensions: 5.63 x 2.83 x 0.39 inches; 1.9 ounces; upc: 799768026253; manufacturer: lurrose"
2,B08DNQTTQK,221,50 pieces false eyelash packaging box empty eyelash box plastic eyelash storage case with glitter paper and clear tray for women girls eyelash (holographic) / / package dimensions: 14.49 x 11.26 x 2.36 inches; 2.65 pounds


▶ raw_text 생성 완료
  - rows: 12,402
  - empty raw_text: 0 (0.0%)
  - avg raw_text_len: 464.2

[meta_df_text preview]


,parent_asin,title,brand,price_num,raw_text_len,raw_text
0,B076WQZGPM,"Yes to Tomatoes Detoxifying Charcoal Cleanser (Pack of 2) with Charcoal Powder, Tomato Fruit Extract, and Gingko Biloba Leaf Extract, 5 fl. oz.",Yes To,NaN,370,"yes to tomatoes detoxifying charcoal cleanser (pack of 2) with charcoal powder, tomato fruit extract, and gingko biloba leaf extract, 5 fl. oz. / / item form: powder; skin type: acne prone; brand: yes to; age range (): adult; unit count: 10 fl oz; is discontinued by manufacturer: no; item model number: sg_b076wqzgpm_us; upc: 653801351125; manufacturer: yes to tomatoes"
1,B07G9GWFSM,Lurrose 100Pcs Full Cover Fake Toenails Artificial Transparent Nail Tips Nail Art for DIY,Lurrose,6.99,1414,"lurrose 100pcs full cover fake toenails artificial transparent nail tips nail art for diy / the false toenails are durable with perfect length. you have the option to wear them long or clip them short, easy to trim and file them to in any length and shape you like.; abs is kind of green enviromental material, and makes the nails durable, breathable, light even no pressure on your own nails.; fit well to your natural toenails. non toxic, no smell, no harm to your health.; wonderful as gift for girlfriend, family and friends.; the easiest and most efficient way to do your toenail tips for manicures or nail art designs. it's fashion, creative, a useful accessory brighten up your look, also as a gift. / ['' 'the false toenails are durable with perfect length. you have the option to wear them long or clip them short, easy to trim and file them to in any length and shape you like. plus, abs is kind of green enviromental material, and makes the nails durable, breathable, light even no pressure on your own toenails. fit well to your natural toenails. non toxic, no smell, no harm to your health.' '' '- color: as shown.- material: abs.- size: 14.3 x 7.2 x 1cm.' '' '100 x pieces fake toenails'] / color: as shown; size: large; material: acrylonitrile butadiene styrene (abs); brand: lurrose; style: french; product dimensions: 5.63 x 2.83 x 0.39 inches; 1.9 ounces; upc: 799768026253; manufacturer: lurrose"
2,B08DNQTTQK,50 Pieces False Eyelash Packaging Box Empty Eyelash Box Plastic Eyelash Storage Case with Glitter Paper and Clear Tray for Women Girls Eyelash (Holographic),Maitys,NaN,221,50 pieces false eyelash packaging box empty eyelash box plastic eyelash storage case with glitter paper and clear tray for women girls eyelash (holographic) / / package dimensions: 14.49 x 11.26 x 2.36 inches; 2.65 pounds


In [7]:
# ==========================================================
# CELL 4.3 / raw_text 추가 클린업 (이중 슬래시 + 배열표기 문자열 제거)
# ==========================================================
import ast
import pandas as pd
import numpy as np
import re

def fix_listish_string(s: str) -> str:
    """문자열이 ['a' 'b'] 같은 '배열처럼 보이는' 형태면 리스트로 해석해 '; '로 합침"""
    if not isinstance(s, str):
        return s
    txt = s.strip()
    # 대괄호로 감싸진 '리스트처럼 보이는' 문자열만 시도
    if not (txt.startswith("[") and txt.endswith("]")):
        return s
    # 1) 먼저 공백 구분 ndarray 표기 → 콤마 삽입
    #    예: ['a' 'b' '' 'c'] -> ['a','b','','c']
    fixed = re.sub(r"'\s+'", "', '", txt)
    try:
        obj = ast.literal_eval(fixed)  # 파이썬 리터럴로 안전 파싱
        if isinstance(obj, (list, tuple)):
            items = [str(x).strip() for x in obj if str(x).strip()]
            return "; ".join(items)
        return s
    except Exception:
        return s

# feature_str / description 에 남은 배열표기 문자열 정리
meta_df_text["feature_str"] = meta_df_text["feature_str"].apply(fix_listish_string)
meta_df_text["description"] = meta_df_text["description"].apply(fix_listish_string)

# 빈 파트 제거하여 다시 조립
def join_nonempty(parts):
    parts = [p.strip() for p in parts if isinstance(p, str) and p.strip()]
    return " / ".join(parts)

meta_df_text["raw_text"] = meta_df_text.apply(
    lambda r: join_nonempty([r.get("title",""), r.get("feature_str",""), r.get("description",""), r.get("details_str","")]),
    axis=1
).str.lower()

# 헤더 토큰 제거 후 남은 이중 슬래시/끝단 슬래시 정리
meta_df_text["raw_text"] = meta_df_text["raw_text"].str.replace(
    r"\b(description|feature|package including)\b\s*[:\-]?\s*", "", regex=True
)
meta_df_text["raw_text"] = meta_df_text["raw_text"]\
    .str.replace(r"(\s*/\s*){2,}", " / ", regex=True)\
    .str.replace(r"^\s*/\s*|\s*/\s*$", "", regex=True)\
    .str.replace(r"\s{2,}", " ", regex=True)\
    .str.strip()

# 길이 갱신 & 미리보기
meta_df_text["raw_text_len"] = meta_df_text["raw_text"].str.len()
with pd.option_context("display.max_colwidth", None):
    display(meta_df_text[["parent_asin","raw_text_len","raw_text"]].head(3))


,parent_asin,raw_text_len,raw_text
0,B076WQZGPM,368,"yes to tomatoes detoxifying charcoal cleanser (pack of 2) with charcoal powder, tomato fruit extract, and gingko biloba leaf extract, 5 fl. oz. / item form: powder; skin type: acne prone; brand: yes to; age range (): adult; unit count: 10 fl oz; is discontinued by manufacturer: no; item model number: sg_b076wqzgpm_us; upc: 653801351125; manufacturer: yes to tomatoes"
1,B07G9GWFSM,1405,"lurrose 100pcs full cover fake toenails artificial transparent nail tips nail art for diy / the false toenails are durable with perfect length. you have the option to wear them long or clip them short, easy to trim and file them to in any length and shape you like.; abs is kind of green enviromental material, and makes the nails durable, breathable, light even no pressure on your own nails.; fit well to your natural toenails. non toxic, no smell, no harm to your health.; wonderful as gift for girlfriend, family and friends.; the easiest and most efficient way to do your toenail tips for manicures or nail art designs. it's fashion, creative, a useful accessory brighten up your look, also as a gift. / ; the false toenails are durable with perfect length. you have the option to wear them long or clip them short, easy to trim and file them to in any length and shape you like. plus, abs is kind of green enviromental material, and makes the nails durable, breathable, light even no pressure on your own toenails. fit well to your natural toenails. non toxic, no smell, no harm to your health.; ; - color: as shown.- material: abs.- size: 14.3 x 7.2 x 1cm.; ; 100 x pieces fake toenails / color: as shown; size: large; material: acrylonitrile butadiene styrene (abs); brand: lurrose; style: french; product dimensions: 5.63 x 2.83 x 0.39 inches; 1.9 ounces; upc: 799768026253; manufacturer: lurrose"
2,B08DNQTTQK,219,50 pieces false eyelash packaging box empty eyelash box plastic eyelash storage case with glitter paper and clear tray for women girls eyelash (holographic) / package dimensions: 14.49 x 11.26 x 2.36 inches; 2.65 pounds


In [8]:
# ==========================================================
# CELL 4.4 / raw_text 세미콜론/중복 문장 클린업
# ==========================================================
import re

def dedupe_phrases(s: str) -> str:
    if not isinstance(s, str) or not s.strip():
        return s
    # 1) 불필요한 구분자 정리
    s = (s.replace("/ ;", "/ ")
           .replace("; /", "/")
           .replace(";;", ";"))
    s = re.sub(r"\s*;\s*;\s*", "; ", s)
    s = re.sub(r"\(\s*\)", "", s)              # 빈 괄호 제거
    s = re.sub(r"\s{2,}", " ", s).strip()

    # 2) 세미콜론 기준으로 중복 문장 제거(짧은 토큰은 유지)
    parts = [p.strip() for p in s.split(";")]
    seen, out = set(), []
    for p in parts:
        if len(p) >= 8:        # 너무 짧은 토큰은 그냥 둠
            if p.lower() in seen:
                continue
            seen.add(p.lower())
        out.append(p)
    s2 = "; ".join([p for p in out if p])
    # 마지막으로 슬래시 주변 공백/중복 슬래시 정리
    s2 = re.sub(r"(\s*/\s*){2,}", " / ", s2).strip(" /")
    return s2

meta_df_text["raw_text"] = meta_df_text["raw_text"].apply(dedupe_phrases)
meta_df_text["raw_text_len"] = meta_df_text["raw_text"].str.len()

with pd.option_context("display.max_colwidth", None):
    display(meta_df_text[["parent_asin","raw_text_len","raw_text"]].head(3))


,parent_asin,raw_text_len,raw_text
0,B076WQZGPM,366,"yes to tomatoes detoxifying charcoal cleanser (pack of 2) with charcoal powder, tomato fruit extract, and gingko biloba leaf extract, 5 fl. oz. / item form: powder; skin type: acne prone; brand: yes to; age range : adult; unit count: 10 fl oz; is discontinued by manufacturer: no; item model number: sg_b076wqzgpm_us; upc: 653801351125; manufacturer: yes to tomatoes"
1,B07G9GWFSM,1399,"lurrose 100pcs full cover fake toenails artificial transparent nail tips nail art for diy / the false toenails are durable with perfect length. you have the option to wear them long or clip them short, easy to trim and file them to in any length and shape you like.; abs is kind of green enviromental material, and makes the nails durable, breathable, light even no pressure on your own nails.; fit well to your natural toenails. non toxic, no smell, no harm to your health.; wonderful as gift for girlfriend, family and friends.; the easiest and most efficient way to do your toenail tips for manicures or nail art designs. it's fashion, creative, a useful accessory brighten up your look, also as a gift. / the false toenails are durable with perfect length. you have the option to wear them long or clip them short, easy to trim and file them to in any length and shape you like. plus, abs is kind of green enviromental material, and makes the nails durable, breathable, light even no pressure on your own toenails. fit well to your natural toenails. non toxic, no smell, no harm to your health.; - color: as shown.- material: abs.- size: 14.3 x 7.2 x 1cm.; 100 x pieces fake toenails / color: as shown; size: large; material: acrylonitrile butadiene styrene (abs); brand: lurrose; style: french; product dimensions: 5.63 x 2.83 x 0.39 inches; 1.9 ounces; upc: 799768026253; manufacturer: lurrose"
2,B08DNQTTQK,219,50 pieces false eyelash packaging box empty eyelash box plastic eyelash storage case with glitter paper and clear tray for women girls eyelash (holographic) / package dimensions: 14.49 x 11.26 x 2.36 inches; 2.65 pounds


In [9]:
# ==========================================================
# CELL 4.5 / raw_text 교차-세그먼트 중복 제거( '/'·';'·'.' 기준 )
# ==========================================================
import re

def _norm(p: str) -> str:
    p = re.sub(r"\(\s*\)", "", p)       # 빈 괄호 제거
    p = re.sub(r"\s{2,}", " ", p).strip()
    return p

def dedupe_cross_segment(s: str) -> str:
    if not isinstance(s, str) or not s.strip():
        return s
    segs = [seg.strip() for seg in s.split("/") if seg.strip()]
    seen = set()
    out_segs = []
    for seg in segs:
        parts = re.split(r"[;\.]", seg)  # 세미콜론·마침표 기준 분해
        uniq = []
        for part in parts:
            ph = _norm(part)
            if not ph:
                continue
            key = ph.lower()
            # 8자 미만은 너무 자잘하니 dedupe 완화
            if len(ph) >= 8:
                if key in seen:
                    continue
                seen.add(key)
            uniq.append(ph)
        seg_clean = "; ".join(uniq).strip(" ;")
        if seg_clean:
            out_segs.append(seg_clean)
    s2 = " / ".join(out_segs)
    s2 = re.sub(r"(\s*/\s*){2,}", " / ", s2)
    s2 = re.sub(r"\s*;\s*;\s*", "; ", s2)
    s2 = re.sub(r"\(\s*\)", "", s2)
    return s2

meta_df_text["raw_text"] = meta_df_text["raw_text"].apply(dedupe_cross_segment)
meta_df_text["raw_text_len"] = meta_df_text["raw_text"].str.len()

with pd.option_context("display.max_colwidth", None):
    display(meta_df_text[["parent_asin","raw_text_len","raw_text"]].head(3))


,parent_asin,raw_text_len,raw_text
0,B076WQZGPM,365,"yes to tomatoes detoxifying charcoal cleanser (pack of 2) with charcoal powder, tomato fruit extract, and gingko biloba leaf extract, 5 fl; oz / item form: powder; skin type: acne prone; brand: yes to; age range : adult; unit count: 10 fl oz; is discontinued by manufacturer: no; item model number: sg_b076wqzgpm_us; upc: 653801351125; manufacturer: yes to tomatoes"
1,B07G9GWFSM,1146,"lurrose 100pcs full cover fake toenails artificial transparent nail tips nail art for diy / the false toenails are durable with perfect length; you have the option to wear them long or clip them short, easy to trim and file them to in any length and shape you like; abs is kind of green enviromental material, and makes the nails durable, breathable, light even no pressure on your own nails; fit well to your natural toenails; non toxic, no smell, no harm to your health; wonderful as gift for girlfriend, family and friends; the easiest and most efficient way to do your toenail tips for manicures or nail art designs; it's fashion, creative, a useful accessory brighten up your look, also as a gift / plus, abs is kind of green enviromental material, and makes the nails durable, breathable, light even no pressure on your own toenails; - color: as shown; - material: abs; - size: 14; 3 x 7; 2 x 1cm; 100 x pieces fake toenails / color: as shown; size: large; material: acrylonitrile butadiene styrene (abs); brand: lurrose; style: french; product dimensions: 5; 63 x 2; 83 x 0; 39 inches; 1; 9 ounces; upc: 799768026253; manufacturer: lurrose"
2,B08DNQTTQK,223,50 pieces false eyelash packaging box empty eyelash box plastic eyelash storage case with glitter paper and clear tray for women girls eyelash (holographic) / package dimensions: 14; 49 x 11; 26 x 2; 36 inches; 2; 65 pounds


In [10]:
# ==========================================================
# CELL 5 / Beauty 핵심 속성 온톨로지 정의 (한글 주석)
# 목적: 이후 추출/필터/랭킹에서 사용할 "속성 스키마"와 토큰·정규식을 한 곳에 정의
# 출력: ATTR_ONTOLOGY, ATTR_TOKENS, ATTR_REGEX, SOURCE_WEIGHTS, CONFIDENCE_THRESHOLD_HARD, attr_def_df
# 참고: 실제 추출 로직은 CELL 6, 브랜드 보강 사전은 CELL 5.1에서 추가 예정
# ==========================================================

import pandas as pd
from pprint import pprint

# --------------------------
# 5-0) 버전/스코프 표기
# --------------------------
EXTRACTOR_VERSION = "beauty_v2"
CATEGORY_SCOPE    = "Beauty"

# --------------------------
# 5-1) 출처별 신뢰 가중치(셀6에서 사용)
# --------------------------
# details > feature > title > description 순으로 신뢰
SOURCE_WEIGHTS = {
    "details_str": 1.00,
    "feature_str": 0.85,
    "title":       0.75,
    "description": 0.60,
}

# --------------------------
# 5-2) 하드필터 임계치(기본값) — 셀 6B에서 속성별로 조정 가능
# --------------------------
CONFIDENCE_THRESHOLD_HARD = 0.80

# --------------------------
# 5-3) 도메인 속성 정의(스키마)
# --------------------------
# value_type: "number" | "enum" | "multilabel" | "string" | "boolean"
# unit      : 수치 단위(ml/usd 등), 비수치면 ""
# allowed_values / canonical_map: enum/멀티라벨의 허용값/동의어→표준값 맵
PRODUCT_TYPES = [
    # 스킨/헤어/바디 대표 타입
    "cleanser","face wash","toner","serum","essence","ampoule",
    "moisturizer","lotion","cream","gel","mask","sunscreen",
    "shampoo","conditioner","hair oil","oil","body wash","body lotion",
    "balm"
]
# 표기 변형을 표준값으로 통일
PRODUCT_TYPE_CANON = {
    "face cleanser": "cleanser",
    "cleaning foam": "cleanser",
    "sun screen":    "sunscreen",
    "sunblock":      "sunscreen",
    "hair mask":     "mask",
}

TARGET_AUDIENCE = [
    # 성별/연령
    "men","women","unisex","baby","kids","teens",
    # 피부/모발 타입
    "sensitive","dry","oily","normal","combination",
    "acne","acne-prone","mature","aging",
    "curly","color-treated","damaged"
]
TARGET_CANON = {
    "colored":        "color-treated",
    "acne prone":     "acne-prone",
    "ageing":         "aging",
    "teen":           "teens",
    "kid":            "kids",
}

SCENT_VALUES = [
    "unscented","lavender","rose","floral","citrus","lemon","orange","grapefruit",
    "vanilla","coconut","jasmine","tea tree","green tea","peppermint","eucalyptus",
    "almond","berry","apple"
]
SCENT_CANON = {
    "fragrance free": "unscented",
    "fragrance-free": "unscented",
    "no fragrance":   "unscented",
    "no scent":       "unscented",
    "scent free":     "unscented",
}

VOLUME_UNIT = "ml"  # 용량은 최종적으로 ml로 환산(CELL 6에서 oz→ml 변환)

ATTR_ONTOLOGY = [
    {
        "attr_id":         "brand",
        "display_name_kr": "브랜드",
        "value_type":      "string",
        "unit":            "",
        "allowed_values":  [],
        "canonical_map":   {},
        "notes":           "메타 brand/brand_norm 우선. 셀 5.1에서 사전화 보강"
    },
    {
        "attr_id":         "product_type",
        "display_name_kr": "제품유형",
        "value_type":      "enum",
        "unit":            "",
        "allowed_values":  PRODUCT_TYPES,
        "canonical_map":   PRODUCT_TYPE_CANON,
        "notes":           "title/feature/description 키워드 매칭"
    },
    {
        "attr_id":         "target_audience",
        "display_name_kr": "대상/피부·모발타입",
        "value_type":      "multilabel",
        "unit":            "",
        "allowed_values":  TARGET_AUDIENCE,
        "canonical_map":   TARGET_CANON,
        "notes":           "men/women/unisex/baby/kids/teens/sensitive/dry/oily/…"
    },
    {
        "attr_id":         "volume_ml",
        "display_name_kr": "용량(ml)",
        "value_type":      "number",
        "unit":            VOLUME_UNIT,
        "allowed_values":  [],
        "canonical_map":   {},
        "notes":           "ml 또는 fl oz를 ml로 변환해 대표 용량 선택"
    },
    {
        "attr_id":         "scent",
        "display_name_kr": "향",
        "value_type":      "enum",
        "unit":            "",
        "allowed_values":  SCENT_VALUES,
        "canonical_map":   SCENT_CANON,
        "notes":           "unscented(=fragrance-free/no fragrance 등) 정규화"
    },
    {
        "attr_id":         "price_num",
        "display_name_kr": "가격(USD)",
        "value_type":      "number",
        "unit":            "usd",
        "allowed_values":  [],
        "canonical_map":   {},
        "notes":           "CELL 3에서 숫자화된 가격"
    },
]

# --------------------------
# 5-4) 속성 토큰/사전(셀 6에서 재사용)
# --------------------------
ATTR_TOKENS = {
    "product_type": {
        "tokens": PRODUCT_TYPES + list(PRODUCT_TYPE_CANON.keys()),
        "canonical_map": PRODUCT_TYPE_CANON
    },
    "target_audience": {
        "tokens": TARGET_AUDIENCE + list(TARGET_CANON.keys()),
        "canonical_map": TARGET_CANON
    },
    "scent": {
        "tokens": SCENT_VALUES + list(SCENT_CANON.keys()),
        "canonical_map": SCENT_CANON
    },
}

# --------------------------
# 5-5) 정규식 패턴(셀 6에서 컴파일)
# --------------------------
ATTR_REGEX = {
    # 용량: ml / fl. oz / oz 변형 포괄
    "volume_patterns": [
        r"(\d+(?:\.\d+)?)\s?(ml|mL)",
        r"(\d+(?:\.\d+)?)\s?(fl\.?\s?oz|fluid ounce|fl oz|oz)"
    ],
    # 일반 노이즈 패턴(필요 시 재사용)
    "noise_brackets": [
        r"\b\d+\s*\[\s*\]",  # "0 []" 같은 패턴
        r"\[\s*\]"           # 빈 대괄호
    ],
}

# --------------------------
# 5-6) 정의 표 출력(보고용)
# --------------------------
attr_def_df = pd.DataFrame(ATTR_ONTOLOGY)
print("▶ ATTR_ONTOLOGY (정의 요약)")
display(attr_def_df)

print("\n▶ ATTR_TOKENS keys:", list(ATTR_TOKENS.keys()))
for k, v in ATTR_TOKENS.items():
    print(f"- {k}: tokens={len(v['tokens'])}, canonical_map={len(v.get('canonical_map', {}))}")

print("\n▶ SOURCE_WEIGHTS (출처 가중치):")
pprint(SOURCE_WEIGHTS)

print("\n▶ CONFIDENCE_THRESHOLD_HARD (기본):", CONFIDENCE_THRESHOLD_HARD)

# 주: 브랜드 사전화 맵(BRAND_CANON)은 CELL 5.1에서 별도 정의/저장합니다.


▶ ATTR_ONTOLOGY (정의 요약)


,attr_id,display_name_kr,value_type,unit,allowed_values,canonical_map,notes
0,brand,브랜드,string,,[],{},메타 brand/brand_norm 우선. 셀 5.1에서 사전화 보강
1,product_type,제품유형,enum,,"[cleanser, face wash, toner, serum, essence, ampoule, moisturizer, lotion, cream, gel, mask, sunscreen, shampoo, conditioner, hair oil, oil, body wash, body...","{'face cleanser': 'cleanser', 'cleaning foam': 'cleanser', 'sun screen': 'sunscreen', 'sunblock': 'sunscreen', 'hair mask': 'mask'}",title/feature/description 키워드 매칭
2,target_audience,대상/피부·모발타입,multilabel,,"[men, women, unisex, baby, kids, teens, sensitive, dry, oily, normal, combination, acne, acne-prone, mature, aging, curly, color-treated, damaged]","{'colored': 'color-treated', 'acne prone': 'acne-prone', 'ageing': 'aging', 'teen': 'teens', 'kid': 'kids'}",men/women/unisex/baby/kids/teens/sensitive/dry/oily/…
3,volume_ml,용량(ml),number,ml,[],{},ml 또는 fl oz를 ml로 변환해 대표 용량 선택
4,scent,향,enum,,"[unscented, lavender, rose, floral, citrus, lemon, orange, grapefruit, vanilla, coconut, jasmine, tea tree, green tea, peppermint, eucalyptus, almond, berry...","{'fragrance free': 'unscented', 'fragrance-free': 'unscented', 'no fragrance': 'unscented', 'no scent': 'unscented', 'scent free': 'unscented'}",unscented(=fragrance-free/no fragrance 등) 정규화
5,price_num,가격(USD),number,usd,[],{},CELL 3에서 숫자화된 가격



▶ ATTR_TOKENS keys: ['product_type', 'target_audience', 'scent']
- product_type: tokens=24, canonical_map=5
- target_audience: tokens=23, canonical_map=5
- scent: tokens=23, canonical_map=5

▶ SOURCE_WEIGHTS (출처 가중치):
{'description': 0.6, 'details_str': 1.0, 'feature_str': 0.85, 'title': 0.75}

▶ CONFIDENCE_THRESHOLD_HARD (기본): 0.8


In [11]:
# ==========================================================
# CELL 5.1 (fixed) / 브랜드 토큰 정규화 유틸
# - 목적: 브랜드 문자열을 비교/매칭하기 쉬운 형태로 정규화
# - 산출: clean_brand_token(), normalize_brand() 유틸 함수 등록
# ==========================================================
import re
import unicodedata
import numpy as np
from typing import Optional

def _strip_accents(text: str) -> str:
    """악센트 제거(NFKD). 예: L'Oréal -> L'Oreal -> loreal 처리 전 단계"""
    text = unicodedata.normalize("NFKD", text)
    return "".join(ch for ch in text if not unicodedata.combining(ch))

def clean_brand_token(s: Optional[str]) -> str:
    """
    브랜드 매칭용 '토큰' 정규화:
    - None/NaN 안전 처리
    - 유니코드 정규화 + 악센트 제거
    - 소문자화
    - & -> and 치환
    - 법인/상호 불용어 제거(co/inc/ltd/llc/corp/company/official/store/shop/beauty/cosmetics 등)
    - 아포스트로피/기호 제거(영숫자/공백만 유지, +는 유지)
    - 다중 공백 정리
    - 특례: 'l oreal' -> 'loreal', 'kiehl s' -> 'kiehls', '3 m' -> '3m'
    """
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return ""
    t = str(s).strip()
    if not t:
        return ""

    # 1) 악센트 제거 + 소문자
    t = _strip_accents(t).lower()

    # 2) & -> and
    t = re.sub(r"&", " and ", t)

    # 3) 불용어 제거(단어 경계 기준)
    stop = (
        r"\b(co|co\.|inc|inc\.|ltd|ltd\.|llc|corp|corporation|company|"
        r"official|store|shop|beauty|cosmetics|skincare|care|usa|us|korea|japan|brand)\b"
    )
    t = re.sub(stop, " ", t)

    # 4) 아포스트로피/특수문자 제거(영숫자/공백/플러스만 유지)
    t = t.replace("’", "").replace("'", "")
    t = re.sub(r"[^a-z0-9\s\+]", " ", t)

    # 5) 공백 정리
    t = re.sub(r"\s+", " ", t).strip()

    # 6) 특례 맵
    special = {"l oreal": "loreal", "kiehl s": "kiehls", "3 m": "3m"}
    t = special.get(t, t)
    return t

def normalize_brand(s: Optional[str]) -> str:
    """
    표시용 정규화(프런트 노출용):
    - clean_brand_token 을 기반으로 단어별 Title-Case 적용
    - 공백/빈값이면 빈 문자열 반환
    """
    tok = clean_brand_token(s)
    if not tok:
        return ""
    # 'and'는 소문자 유지, 나머지는 Title
    parts = []
    for w in tok.split():
        parts.append("and" if w == "and" else w.title())
    return " ".join(parts)

# 간단 자체 테스트 (원하는 경우 주석 해제)
# tests = ["L'Oréal Paris", "Kiehl's", "Dove Inc.", "GARNIER Official Store", "3M", "Yes To, Inc."]
# for x in tests:
#     print(x, "=>", clean_brand_token(x), "||", normalize_brand(x))

print("[OK] CELL 5.1(fixed): clean_brand_token / normalize_brand 등록 완료")


[OK] CELL 5.1(fixed): clean_brand_token / normalize_brand 등록 완료


## CELL 6 / 규칙 기반 1차 속성 추출

In [12]:
# ==========================================================
# CELL 6 / 규칙 기반 1차 속성 추출 + confidence 계산 (한글 주석)
# 목적: meta_df_text에서 Beauty 핵심 속성(brand/product_type/target_audience/
#       scent/volume_ml/price_num)을 간단 규칙으로 추출하고 신뢰도(confidence) 부여
# 입력: meta_df_text (CELL 4.x), ATTR_ONTOLOGY/ATTR_TOKENS/ATTR_REGEX/
#      SOURCE_WEIGHTS/CONFIDENCE_THRESHOLD_HARD (CELL 5)
# 출력: item_attr_long, attr_coverage_report
# ==========================================================

# ---------- 0) 선행 체크 ----------
req_vars = ["meta_df_text", "ATTR_ONTOLOGY", "ATTR_TOKENS", "ATTR_REGEX",
            "SOURCE_WEIGHTS", "CONFIDENCE_THRESHOLD_HARD"]
missing = [v for v in req_vars if v not in globals()]
if missing:
    raise RuntimeError(f"이전 셀 결과가 없습니다: {missing}")

import re, math
import pandas as pd
import numpy as np

# ---------- 1) 상수/패턴 컴파일 ----------
# 1 fl oz ≈ 29.5735 mL (미국 액량온스→ml 변환 상수)
OZ_TO_ML = 29.5735

# 용량 정규식 컴파일
vol_patterns = [re.compile(p, flags=re.IGNORECASE) for p in ATTR_REGEX.get("volume_patterns", [])]

# 토큰들을 단어경계 고려 패턴으로 변환
def compile_token_patterns(tokens):
    pats = {}
    for tok in tokens:
        esc = re.escape(tok)
        if re.search(r"[A-Za-z0-9]", tok):
            pats[tok] = re.compile(rf"(?<![A-Za-z0-9]){esc}(?![A-Za-z0-9])", flags=re.IGNORECASE)
        else:
            pats[tok] = re.compile(esc, flags=re.IGNORECASE)
    return pats

PT_TOKENS   = ATTR_TOKENS["product_type"]["tokens"]
PT_CANON    = ATTR_TOKENS["product_type"]["canonical_map"]
PT_PATTERNS = compile_token_patterns(PT_TOKENS)

AUD_TOKENS   = ATTR_TOKENS["target_audience"]["tokens"]
AUD_CANON    = ATTR_TOKENS["target_audience"]["canonical_map"]
AUD_PATTERNS = compile_token_patterns(AUD_TOKENS)

SC_TOKENS   = ATTR_TOKENS["scent"]["tokens"]
SC_CANON    = ATTR_TOKENS["scent"]["canonical_map"]
SC_PATTERNS = compile_token_patterns(SC_TOKENS)

# 출처별 가중치/우선순위
SOURCE_ORDER = [
    ("details_str", SOURCE_WEIGHTS.get("details_str", 1.00)),
    ("feature_str", SOURCE_WEIGHTS.get("feature_str", 0.85)),
    ("title",       SOURCE_WEIGHTS.get("title",       0.75)),
    ("description", SOURCE_WEIGHTS.get("description", 0.60)),
]
TOTAL_SRC_WEIGHT = sum(w for _, w in SOURCE_ORDER)

# ---------- 2) 보조 함수 ----------
def canon_value(val, canon_map):
    """동의어를 표준값으로 치환"""
    if val is None:
        return None
    v = val.strip().lower()
    return canon_map.get(v, v)

def extract_volume_ml_from_text(text):
    """텍스트에서 용량 수치 추출 → (ml값, 기본신뢰도, 적중여부)"""
    if not text:
        return None, 0.0, False
    best_ml = None
    for pat in vol_patterns:
        for m in pat.finditer(text):
            try:
                val = float(m.group(1))
            except Exception:
                continue
            unit = m.group(2).lower()
            ml = val if "ml" in unit else val * OZ_TO_ML
            if best_ml is None or ml > best_ml:
                best_ml = ml
    if best_ml is not None:
        return round(best_ml, 2), 0.8, True  # 숫자 직추출 기본 신뢰도
    return None, 0.0, False

def detect_enum_single(text, patterns, canon_map):
    """단일라벨(enum) 탐지 → (값 또는 None, 적중여부)"""
    if not text:
        return None, False
    scores = {}
    for tok, pat in patterns.items():
        if pat.search(text):
            key = canon_value(tok, canon_map)
            scores[key] = scores.get(key, 0) + 1
    if not scores:
        return None, False
    return max(scores.items(), key=lambda kv: kv[1])[0], True

def detect_multilabel(text, patterns, canon_map):
    """다중라벨 탐지 → (값집합, 적중여부)"""
    if not text:
        return set(), False
    hits = set()
    for tok, pat in patterns.items():
        if pat.search(text):
            hits.add(canon_value(tok, canon_map))
    if "acne prone" in hits:  # 흔한 변형 정리
        hits.discard("acne prone"); hits.add("acne-prone")
    return hits, bool(hits)

# ---------- 3) 행 단위 추출 ----------
def extract_row(row):
    """
    각 상품 행에서 여러 출처 텍스트를 훑어 속성값을 추출하고 신뢰도를 계산.
    - details_str > feature_str > title > description 순으로 가중 합산
    - volume_ml은 가장 큰 값을 대표로 선택(멀티팩/세트 표기 완화)
    """
    results = []
    parent_asin = row.get("parent_asin")

    # 출처 텍스트 모으기
    src_texts = []
    for col, w in SOURCE_ORDER:
        t = row.get(col, "")
        t = "" if t is None else str(t)
        src_texts.append((col, w, t.lower()))

    # 브랜드: brand_norm 우선 사용(5.1/6A에서 더 보강 예정)
    # (제출 후 수정) CELL 4에서 brand를 채운 출처에 따라 신뢰도를 다르게 준다.
    #  - details.Brand / meta_brand: 브랜드 전용 필드 → 0.95
    #  - store: 판매처 이름이라 리셀러일 수 있음 → 0.85
    #  - details.Manufacturer: 제조사(모회사)일 수 있음 → 0.80 (brand hard 임계값과 같음)
    BRAND_CONF_BY_SOURCE = {"meta_brand": 0.95, "details.Brand": 0.95, "store": 0.85, "details.Manufacturer": 0.80}
    if "brand_norm" in row and isinstance(row["brand_norm"], str) and row["brand_norm"].strip():
        _bsrc = row.get("brand_source") or "meta_brand"
        results.append({
            "parent_asin": parent_asin, "attr_id": "brand",
            "value_text": row["brand_norm"].strip(), "value_num": None, "unit": "",
            "confidence": BRAND_CONF_BY_SOURCE.get(_bsrc, 0.80), "support_sources": _bsrc
        })

    # 가격: price_num 그대로 사용
    if "price_num" in row and pd.notna(row["price_num"]):
        results.append({
            "parent_asin": parent_asin, "attr_id": "price_num",
            "value_text": None, "value_num": float(row["price_num"]), "unit": "usd",
            "confidence": 0.95, "support_sources": "meta_price"
        })

    # 제품유형 product_type
    pt_weights, pt_sources = {}, {}
    for col, w, txt in src_texts:
        val, hit = detect_enum_single(txt, PT_PATTERNS, PT_CANON)
        if hit and val:
            pt_weights[val] = pt_weights.get(val, 0.0) + w
            pt_sources.setdefault(val, []).append(col)
    if pt_weights:
        best_val = max(pt_weights.items(), key=lambda kv: kv[1])[0]
        conf = min(1.0, 0.6 + 0.4 * (pt_weights[best_val] / TOTAL_SRC_WEIGHT))
        results.append({
            "parent_asin": parent_asin, "attr_id": "product_type",
            "value_text": best_val, "value_num": None, "unit": "",
            "confidence": conf, "support_sources": ",".join(sorted(set(pt_sources.get(best_val, []))))
        })

    # 대상 target_audience (다중라벨)
    aud_weights, aud_sources = {}, {}
    for col, w, txt in src_texts:
        vals, hit = detect_multilabel(txt, AUD_PATTERNS, AUD_CANON)
        if hit:
            for v in vals:
                aud_weights[v] = aud_weights.get(v, 0.0) + w
                aud_sources.setdefault(v, []).append(col)
    for v, w_sum in sorted(aud_weights.items(), key=lambda kv: -kv[1]):
        conf = min(1.0, 0.55 + 0.45 * (w_sum / TOTAL_SRC_WEIGHT))
        if conf >= 0.6:  # 너무 약한 적중은 제외
            results.append({
                "parent_asin": parent_asin, "attr_id": "target_audience",
                "value_text": v, "value_num": None, "unit": "",
                "confidence": conf, "support_sources": ",".join(sorted(set(aud_sources.get(v, []))))
            })

    # 향 scent
    sc_weights, sc_sources = {}, {}
    for col, w, txt in src_texts:
        val, hit = detect_enum_single(txt, SC_PATTERNS, SC_CANON)
        if hit and val:
            sc_weights[val] = sc_weights.get(val, 0.0) + w
            sc_sources.setdefault(val, []).append(col)
    if sc_weights:
        best_val = max(sc_weights.items(), key=lambda kv: kv[1])[0]
        conf = min(1.0, 0.6 + 0.4 * (sc_weights[best_val] / TOTAL_SRC_WEIGHT))
        results.append({
            "parent_asin": parent_asin, "attr_id": "scent",
            "value_text": best_val, "value_num": None, "unit": "",
            "confidence": conf, "support_sources": ",".join(sorted(set(sc_sources.get(best_val, []))))
        })

    # 용량 volume_ml (수치)
    vol_best, best_conf, best_srcs = None, 0.0, []
    for col, w, txt in src_texts:
        ml, base_conf, hit = extract_volume_ml_from_text(txt)
        if not hit:
            continue
        conf = min(1.0, base_conf * (0.7 + 0.3 * (w / max(1e-6, TOTAL_SRC_WEIGHT))))
        if (vol_best is None) or (ml > vol_best) or (math.isclose(ml, vol_best) and conf > best_conf):
            vol_best, best_conf, best_srcs = ml, conf, [col]
        elif vol_best is not None and abs(ml - vol_best) <= 5.0:
            # 근사치가 여러 출처에서 일치하면 신뢰도 소폭 가산
            best_conf = min(1.0, best_conf + 0.1 * (w / TOTAL_SRC_WEIGHT))
            best_srcs.append(col)
    if vol_best is not None:
        results.append({
            "parent_asin": parent_asin, "attr_id": "volume_ml",
            "value_text": None, "value_num": float(round(vol_best, 2)), "unit": "ml",
            "confidence": best_conf, "support_sources": ",".join(sorted(set(best_srcs)))
        })

    return results

# ---------- 4) 전 데이터에 적용 ----------
rows = []
for _, r in meta_df_text.iterrows():
    rows.extend(extract_row(r))

item_attr_long = pd.DataFrame(rows, columns=[
    "parent_asin","attr_id","value_text","value_num","unit","confidence","support_sources"
])

print("▶ item_attr_long 생성 완료")
print("  - 행 수:", len(item_attr_long))
print("  - 추출 속성:", sorted(item_attr_long["attr_id"].unique().tolist()))

# ---------- 5) 커버리지 요약 ----------
def coverage_report(df):
    total_items = meta_df_text["parent_asin"].nunique() if "parent_asin" in meta_df_text.columns else len(meta_df_text)
    rep = []
    for attr_id, g in df.groupby("attr_id"):
        items_with_attr = g["parent_asin"].nunique()
        uniq_text = g["value_text"].nunique()
        uniq_num  = g.loc[g["value_num"].notna(), "value_num"].nunique()
        hard_items = g.loc[g["confidence"] >= CONFIDENCE_THRESHOLD_HARD, "parent_asin"].nunique()
        rep.append({
            "attr_id": attr_id,
            "items_with_attr": items_with_attr,
            "coverage_pct": round(100.0 * items_with_attr / max(1, total_items), 1),
            "unique_text_values": int(uniq_text),
            "unique_num_values": int(uniq_num),
            "hard_ready_items": int(hard_items),
            "hard_ready_pct": round(100.0 * hard_items / max(1, total_items), 1),
        })
    return pd.DataFrame(rep).sort_values(["coverage_pct","attr_id"], ascending=[False, True])

attr_coverage_report = coverage_report(item_attr_long)

print("\n▶ 속성 커버리지 미리보기")
display(attr_coverage_report)

print("\n▶ item_attr_long 미리보기 (head)")
display(item_attr_long.head(10))

# (선택) 캐시
# item_attr_long.to_parquet("/mnt/data/beauty_item_attr_long.parquet", index=False)
# attr_coverage_report.to_csv("/mnt/data/beauty_attr_coverage.csv", index=False)


▶ item_attr_long 생성 완료
  - 행 수: 30011
  - 추출 속성: ['brand', 'price_num', 'product_type', 'scent', 'target_audience', 'volume_ml']

▶ 속성 커버리지 미리보기


,attr_id,items_with_attr,coverage_pct,unique_text_values,unique_num_values,hard_ready_items,hard_ready_pct
0,brand,11282,91.0,6701,0,11282,91.0
4,target_audience,5419,43.7,18,0,246,2.0
2,product_type,3870,31.2,18,0,1375,11.1
5,volume_ml,3030,24.4,0,419,0,0.0
1,price_num,2249,18.1,0,952,2249,18.1
3,scent,1433,11.6,18,0,475,3.8



▶ item_attr_long 미리보기 (head)


,parent_asin,attr_id,value_text,value_num,unit,confidence,support_sources
0,B076WQZGPM,brand,yes to,NaN,,0.950000,details.Brand
1,B076WQZGPM,product_type,cleanser,NaN,,0.693750,title
2,B076WQZGPM,target_audience,acne-prone,NaN,,0.690625,details_str
3,B076WQZGPM,target_audience,acne,NaN,,0.690625,details_str
4,B076WQZGPM,volume_ml,None,295.74,ml,0.635000,details_str
5,B07G9GWFSM,brand,lurrose,NaN,,0.950000,details.Brand
6,B07G9GWFSM,price_num,None,6.99,usd,0.950000,meta_price
7,B08DNQTTQK,brand,maitys,NaN,,0.850000,store
8,B08DNQTTQK,target_audience,women,NaN,,0.655469,title
9,B01ERJEGS6,brand,balmain,NaN,,0.950000,details.Brand


In [13]:
# ==========================================================
# CELL 6A (self-contained) / 브랜드 캐논 구축 & 정규식 생성
# - 선행 객체가 없어도 동작(소스 프레임 자동탐지 + 브랜드 시드 리스트 내장)
# - 산출: BRAND_CANON(dict), BRAND_REGEX(re.Pattern), guess_brand_from_title()
# ==========================================================
import re, unicodedata
import numpy as np
import pandas as pd
from typing import Optional
from IPython.display import display

# ---------- A) 유틸: 정규화 함수 (없으면 자동 정의) ----------
def _strip_accents(text: str) -> str:
    text = unicodedata.normalize("NFKD", text)
    return "".join(ch for ch in text if not unicodedata.combining(ch))

def _safe_defined(name: str)->bool:
    try:
        globals()[name]
        return True
    except KeyError:
        return False

if not _safe_defined("clean_brand_token"):
    def clean_brand_token(s: Optional[str]) -> str:
        if s is None or (isinstance(s, float) and np.isnan(s)): return ""
        t = str(s).strip()
        if not t: return ""
        t = _strip_accents(t).lower()
        t = re.sub(r"&", " and ", t)
        stop = (r"\b(co|co\.|inc|inc\.|ltd|ltd\.|llc|corp|corporation|company|"
                r"official|store|shop|beauty|cosmetics|skincare|care|usa|us|korea|japan|brand)\b")
        t = re.sub(stop, " ", t)
        t = t.replace("’","").replace("'","")
        t = re.sub(r"[^a-z0-9\s\+]", " ", t)
        t = re.sub(r"\s+", " ", t).strip()
        special = {"l oreal":"loreal", "kiehl s":"kiehls", "3 m":"3m"}
        return special.get(t, t)

if not _safe_defined("normalize_brand"):
    def normalize_brand(s: Optional[str]) -> str:
        tok = clean_brand_token(s)
        if not tok: return ""
        parts = []
        for w in tok.split():
            parts.append("and" if w=="and" else w.title())
        return " ".join(parts)

# ---------- B) 소스 프레임 자동 선택 ----------
SRC = None
src_name = None
for name in ["item_features_step3", "item_table_enh", "item_table", "meta_df_text"]:
    if name in globals():
        SRC = globals()[name].copy()
        src_name = name
        break

if SRC is None:
    # 소스 프레임이 전혀 없을 때도 동작하도록 빈 프레임 생성
    SRC = pd.DataFrame(columns=["title"])
    src_name = "(no frame; seed list only)"

# ---------- C) 브랜드 후보 시리즈 수집 ----------
brand_cols = [c for c in ["brand_view","brand","details.brand","details.manufacturer"] if c in SRC.columns]
series_list = []
if brand_cols:
    series_list += [SRC[c].astype(str) for c in brand_cols]
# title은 heuristic 탐지용으로도 쓰지만, 토큰 소스로 직접 쓰진 않음(노이즈 큼)

brand_series = pd.concat(series_list, axis=0) if series_list else pd.Series([], dtype="object")

# ---------- D) 시드 리스트 (브랜드 컬럼 없거나 비어 있을 때 사용) ----------
SEED_BRANDS = [
    # 스킨케어/메이크업 대중 브랜드
    "L'Oreal Paris","Maybelline","Garnier","L.A. Girl","NYX","Revlon","Covergirl",
    "Neutrogena","Aveeno","CeraVe","Cetaphil","La Roche-Posay","Eucerin","Olay",
    "Nivea","The Ordinary","COSRX","Paula's Choice","Bioderma","Avene","Vichy",
    "Kiehl's","Clinique","Estee Lauder","Lancome","Shiseido","SK-II","Innisfree","Etude",
    "Hada Labo","Rohto","Biore","Banila Co","Missha","Dr. Jart+","Sulwhasoo","Belif",
    # 헤어/바디
    "Head & Shoulders","Pantene","Herbal Essences","TRESemme","Garnier Fructis",
    "Dove","OGX","Aussie","Redken","L'Oreal Elvive","Maui Moisture","SheaMoisture",
    "Old Spice","Nivea Men","Vaseline","Palmers","Bio-Oil",
    # 남성/향
    "Nautica","Calvin Klein","Versace","Davidoff","Burberry","Dolce & Gabbana",
    # 치약/기타 뷰티 주변
    "Colgate","Oral-B","Crest","Sensodyne","Listerine",
]
if brand_series.empty or brand_series.dropna().eq("").all():
    brand_series = pd.Series(SEED_BRANDS, dtype="object")

# ---------- E) 토큰화 & 캐논 생성 ----------
tok_series = brand_series.map(clean_brand_token)
tok_series = tok_series[tok_series.astype(bool)]
tok_counts = tok_series.value_counts()

def _canonical_for_token(token: str) -> str:
    # 원본 브랜드에서 가져올 수 없으면 토큰을 Title-Case로
    if series_list:
        mask = brand_series.map(lambda x: clean_brand_token(x) == token if isinstance(x, str) else False)
        originals = brand_series[mask]
        if not originals.empty:
            canons = originals.map(normalize_brand).value_counts()
            if len(canons): return canons.index[0]
    return normalize_brand(token)

BRAND_MAX_PATTERNS = 2000
top_tokens = list(tok_counts.index[:BRAND_MAX_PATTERNS])
BRAND_CANON = {t: _canonical_for_token(t) for t in top_tokens if t}

# ---------- F) 제목 탐지용 정규식 ----------
def _token_to_regex_alt(t: str) -> str:
    t = t.replace("+", r"\+")
    t = re.sub(r"\s+", r"\\s+", t)
    return t

tokens_sorted = sorted(BRAND_CANON.keys(), key=len, reverse=True)
alts = [_token_to_regex_alt(t) for t in tokens_sorted if t]

if alts:
    pattern_str = r"(?i)\b(?:%s)\b" % ("|".join(alts))
    try:
        BRAND_REGEX = re.compile(pattern_str)
    except re.error:
        # 너무 길면 상한을 더 낮춰 재시도
        BRAND_MAX_PATTERNS = 1000
        tokens_sorted = sorted(list(tok_counts.index[:BRAND_MAX_PATTERNS]), key=len, reverse=True)
        BRAND_CANON = {t: _canonical_for_token(t) for t in tokens_sorted}
        alts = [_token_to_regex_alt(t) for t in tokens_sorted if t]
        BRAND_REGEX = re.compile(r"(?i)\b(?:%s)\b" % ("|".join(alts)))
else:
    # 매칭 없음: 절대 매칭되지 않는 패턴
    BRAND_REGEX = re.compile(r"a^")

# ---------- G) 헬퍼 ----------
def guess_brand_from_title(title: str) -> str:
    if not isinstance(title, str) or not title.strip(): return ""
    m = BRAND_REGEX.search(title.lower())
    if not m: return ""
    raw = m.group(0)
    tok = clean_brand_token(raw)
    return BRAND_CANON.get(tok, normalize_brand(tok))

# ---------- H) 리포트 ----------
print("=== CELL 6A REPORT — BRAND CANON/REGEX 구축 (self-contained) ===")
print(f"- 소스 프레임: {src_name}")
print(f"- 관측 브랜드 토큰(유니크): {tok_counts.size:,}")
print(f"- BRAND_CANON 크기: {len(BRAND_CANON):,}")
print(f"- BRAND_REGEX 길이: {len(BRAND_REGEX.pattern):,} chars")
# 샘플 5건
titles = None
if "title" in SRC.columns and not SRC["title"].isna().all():
    titles = SRC["title"].dropna().astype(str).head(5)
else:
    titles = pd.Series([
        "L'Oreal Paris Revitalift Serum 1.5% HA",
        "CeraVe Hydrating Cleanser 16 fl oz",
        "Kiehl's Ultra Facial Cream",
        "Dove Men+Care Body Wash",
        "La Roche-Posay Anthelios Sunscreen SPF 50"
    ])
sample_df = pd.DataFrame({"title": titles})
sample_df["brand_guess_from_title"] = sample_df["title"].map(guess_brand_from_title)
display(sample_df)


=== CELL 6A REPORT — BRAND CANON/REGEX 구축 (self-contained) ===
- 소스 프레임: meta_df_text
- 관측 브랜드 토큰(유니크): 6,661
- BRAND_CANON 크기: 2,000
- BRAND_REGEX 길이: 20,858 chars


,title,brand_guess_from_title
0,"Yes to Tomatoes Detoxifying Charcoal Cleanser (Pack of 2) with Charcoal Powder, Tomato Fruit Extract, and Gingko Biloba Leaf Extract, 5 fl. oz.",Yes To
1,Lurrose 100Pcs Full Cover Fake Toenails Artificial Transparent Nail Tips Nail Art for DIY,Lurrose
2,50 Pieces False Eyelash Packaging Box Empty Eyelash Box Plastic Eyelash Storage Case with Glitter Paper and Clear Tray for Women Girls Eyelash (Holographic),
3,Gold extatic Musk EDT 90ml,
4,4 Pieces Satin Bonnet Adjustable Sleep Cap Double Layer Printed Hair Bonnet Large Reversible Sleeping Silk Night Hair Cap for Women Natural Curly Hair (Flor...,Double


In [14]:
# ==========================================================
# CELL 6B / 신뢰도 정책 조정(속성별 하드임계치 + 합의 보너스)  [한글 주석]
# 목적: 속성마다 'hard 통과' 기준을 다르게 두고, 다중 출처 합의 시 신뢰도 가산
# 입력: item_attr_long (CELL 6), meta_df_text (원문 텍스트는 합의 판단 참고 X)
# 출력: item_attr_long (confidence_adj, hard_pass 컬럼 추가), hard 기준 리포트(전/후)
# ==========================================================

import pandas as pd

if "item_attr_long" not in locals():
    raise RuntimeError("item_attr_long이 없습니다. CELL 6을 먼저 실행하세요.")

# 1) 속성별 하드 임계치(프로젝트 상황에 맞게 조정 가능)
THRESH_BY_ATTR = {
    "brand":          0.80,   # 0.85 -> 0.80
    "product_type":   0.75,
    "target_audience":0.70,
    "scent":          0.75,
    "volume_ml":      0.65,
    "price_num":      0.90,
}


# 2) 합의(지원 출처) 보너스: 같은 값이 2개 이상 출처에서 잡히면 +0.15, 3개 이상이면 +0.20 (상한 1.0)
def consensus_bonus(sources: str) -> float:
    if not isinstance(sources, str) or not sources.strip():
        return 0.0
    n_src = len(set([p.strip() for p in sources.split(",") if p.strip()]))
    if n_src >= 3:
        return 0.20
    if n_src >= 2:
        return 0.15
    return 0.0

df0 = item_attr_long.copy()
df0["confidence_adj"] = df0["confidence"] + df0["support_sources"].fillna("").apply(consensus_bonus)
df0["confidence_adj"] = df0["confidence_adj"].clip(upper=1.0)

# 3) 속성별 하드 패스 계산
def hard_pass(row):
    thr = THRESH_BY_ATTR.get(row["attr_id"], CONFIDENCE_THRESHOLD_HARD if "CONFIDENCE_THRESHOLD_HARD" in globals() else 0.8)
    return float(row["confidence_adj"]) >= float(thr)

df0["hard_pass"] = df0.apply(hard_pass, axis=1)

# 4) 전/후 하드 통과율 비교 리포트
def hard_coverage_table(df_before, df_after):
    rows = []
    attrs = sorted(df_after["attr_id"].unique().tolist())
    for a in attrs:
        b = df_before[df_before["attr_id"]==a]
        c = df_after[df_after["attr_id"]==a]
        total_items = meta_df_text["parent_asin"].nunique()
        before_hard = b[b["confidence"] >= THRESH_BY_ATTR.get(a, 0.8)]["parent_asin"].nunique()
        after_hard  = c[c["hard_pass"]==True]["parent_asin"].nunique()
        rows.append({
            "attr_id": a,
            "thr": THRESH_BY_ATTR.get(a, 0.8),
            "before_hard_items": int(before_hard),
            "after_hard_items": int(after_hard),
            "before_hard_pct": round(100*before_hard/max(1,total_items),1),
            "after_hard_pct":  round(100*after_hard/max(1,total_items),1),
        })
    return pd.DataFrame(rows).sort_values(["after_hard_pct","attr_id"], ascending=[False, True])

hard_report_6B = hard_coverage_table(item_attr_long, df0)

# 결과 반영
item_attr_long = df0

print("▶ CELL 6B 완료 — 속성별 임계치/합의 보너스 적용")
display(hard_report_6B)


▶ CELL 6B 완료 — 속성별 임계치/합의 보너스 적용


,attr_id,thr,before_hard_items,after_hard_items,before_hard_pct,after_hard_pct
0,brand,0.80,11282,11282,91.0,91.0
1,price_num,0.90,2249,2249,18.1,18.1
2,product_type,0.75,1577,1577,12.7,12.7
4,target_audience,0.70,1271,1271,10.2,10.2
5,volume_ml,0.65,537,750,4.3,6.0
3,scent,0.75,548,548,4.4,4.4


In [15]:
# ==========================================================
# CELL 6C / 보정(Fallback) + 직접파싱(Direct) 규칙 전체 재작성  [한글 주석]
# 목적:
#   1) details에서 scent/target_audience 직접 파싱(고신뢰)
#   2) 누락된 product_type/scent/target_audience를 카테고리/문장 패턴으로 보완
#   3) confidence_adj/하드패스 재계산 및 커버리지 리포트
# 입력: meta_df_text, item_attr_long, ATTR_TOKENS, TARGET_AUDIENCE/TARGET_CANON,
#      SCENT 토큰 및 정규화 맵(SC_CANON), THRESH_BY_ATTR (CELL 6B)
# 출력: item_attr_long(업데이트), 커버리지 요약 cov_after_6C
# ==========================================================

import re
import pandas as pd

# ---------- 0) 선행 체크 ----------
need_vars = ["meta_df_text", "item_attr_long", "ATTR_TOKENS"]
missing = [v for v in need_vars if v not in globals()]
if missing:
    raise RuntimeError(f"필요 객체가 없습니다: {missing}")

# TARGET/SCENT 관련 맵/토큰 확보
SC_TOKENS = ATTR_TOKENS["scent"]["tokens"]
SC_CANON  = ATTR_TOKENS["scent"]["canonical_map"]
TARGET_AUDIENCE = ATTR_TOKENS["target_audience"]["tokens"]
TARGET_CANON    = ATTR_TOKENS["target_audience"]["canonical_map"]

# 하드 임계치(6B에서 정의되어 있으면 사용, 없으면 기본)
if "THRESH_BY_ATTR" not in locals():
    THRESH_BY_ATTR = {
        "brand":          0.80,
        "product_type":   0.75,
        "target_audience":0.70,
        "scent":          0.75,
        "volume_ml":      0.65,
        "price_num":      0.90,
    }

# 합의 보너스(6B에서 정의되어 있으면 사용, 없으면 기본 정의)
if "consensus_bonus" not in locals():
    def consensus_bonus(sources: str) -> float:
        if not isinstance(sources, str) or not sources.strip():
            return 0.0
        n_src = len(set([p.strip() for p in sources.split(",") if p.strip()]))
        if n_src >= 3:
            return 0.20
        if n_src >= 2:
            return 0.15
        return 0.0

total_items = meta_df_text["parent_asin"].nunique()

# ---------- 1) details에서 scent 직접 파싱(고신뢰) ----------
EXT_SCENT_CANON = {
    "fragrance free":"unscented","fragrance-free":"unscented","no fragrance":"unscented",
    "no scent":"unscented","scent free":"unscented",
    "mint":"peppermint","spearmint":"peppermint",
    "peach":"peach","musk":"musk","shea":"shea","cocoa":"cocoa","cherry":"berry","cucumber":"cucumber"
}
def canon_scent(v: str):
    if not v: return None
    x = str(v).strip().lower()
    x = SC_CANON.get(x, EXT_SCENT_CANON.get(x, x))
    # 허용 토큰 집합에 있거나, 동의어 사전에서 변환 성공한 경우만 채택
    if x in SC_TOKENS:
        return x
    return SC_CANON.get(x, EXT_SCENT_CANON.get(x))

rows_direct_scent = []
if "details" in meta_df_text.columns:
    for _, r in meta_df_text.iterrows():
        asin = r.get("parent_asin")
        d = r.get("details", {})
        if not isinstance(d, dict):
            continue
        chosen = None
        for k, val in d.items():
            kl = str(k).lower()
            if ("scent" in kl) or ("fragrance" in kl):
                vals = val if isinstance(val, (list,tuple,set)) else [val]
                for vv in vals:
                    c = canon_scent(vv)
                    if c:
                        chosen = c
                        break
            if chosen:
                rows_direct_scent.append({
                    "parent_asin": asin, "attr_id":"scent",
                    "value_text": chosen, "value_num": None, "unit":"",
                    "confidence": 0.95, "support_sources": f"details.{kl}"
                })
                break

# ---------- 2) details에서 target_audience 직접 파싱(고신뢰) ----------
A_KEYS = ["skin type","hair type","age range","age range (description)"]
def canon_aud(v: str):
    if not v: return None
    x = str(v).strip().lower()
    x = TARGET_CANON.get(x, x)
    return x if x in TARGET_AUDIENCE else None

rows_direct_aud = []
if "details" in meta_df_text.columns:
    for _, r in meta_df_text.iterrows():
        asin = r.get("parent_asin")
        d = r.get("details", {})
        if not isinstance(d, dict):
            continue
        hit_vals = set()
        for k, v in d.items():
            kl = str(k).lower()
            if any(ak in kl for ak in A_KEYS):
                vals = v if isinstance(v, (list,tuple,set)) else [v]
                for vv in vals:
                    lab = canon_aud(vv)
                    if lab:
                        hit_vals.add(lab)
        for lab in hit_vals:
            rows_direct_aud.append({
                "parent_asin": asin, "attr_id":"target_audience",
                "value_text": lab, "value_num": None, "unit":"",
                "confidence": 0.90, "support_sources": "details.audience"
            })

# ---------- 3) product_type Fallback (categories/title 기반, 신뢰도 0.78) ----------
PT_FALLBACK_MAP = {
    "shampoo":       ["shampoo"],
    "conditioner":   ["conditioner"],
    "cleanser":      ["cleanser","face wash","washing"],
    "toner":         ["toner"],
    "serum":         ["serum","essence","ampoule"],
    "moisturizer":   ["moisturizer","lotion","cream","gel"],
    "mask":          ["mask","sheet mask"],
    "sunscreen":     ["sunscreen","sun screen","sunblock"],
    "body wash":     ["body wash"],
    "body lotion":   ["body lotion","body moisturizer"],
    "hair oil":      ["hair oil"],
    "oil":           ["facial oil","face oil"],
}
def guess_pt_from_text(s):
    s = (s or "").lower()
    for canon, keys in PT_FALLBACK_MAP.items():
        for k in keys:
            if k in s:
                return canon
    return None

have_pt_asins = set(item_attr_long.loc[item_attr_long["attr_id"]=="product_type","parent_asin"])
need_pt_asins = set(meta_df_text["parent_asin"]) - have_pt_asins
rows_fb_pt = []
if need_pt_asins:
    cats = meta_df_text.set_index("parent_asin").get("categories")
    titles = meta_df_text.set_index("parent_asin").get("title")
    for asin in need_pt_asins:
        cat_str = str(cats.get(asin,"")).lower() if cats is not None else ""
        tit_str = str(titles.get(asin,"")).lower() if titles is not None else ""
        pt = guess_pt_from_text(cat_str) or guess_pt_from_text(tit_str)
        if pt:
            rows_fb_pt.append({
                "parent_asin": asin, "attr_id": "product_type",
                "value_text": pt, "value_num": None, "unit": "",
                "confidence": 0.78,  # 상향
                "support_sources": "fallback.categories/title"
            })

# ---------- 4) scent Fallback — 'unscented' 문구 매핑 ----------
UNSCENT_RE = re.compile(r"\b(fragrance\s*free|fragrance\-free|no\s*fragrance|no\s*scent|scent\s*free)\b", re.I)
have_sc_asins = set(item_attr_long.loc[item_attr_long["attr_id"]=="scent","parent_asin"])
need_sc_asins = set(meta_df_text["parent_asin"]) - have_sc_asins
raw_map = meta_df_text.set_index("parent_asin")["raw_text"]
rows_fb_scent = []
for asin in need_sc_asins:
    txt = str(raw_map.get(asin, "")).lower()
    if UNSCENT_RE.search(txt):
        rows_fb_scent.append({
            "parent_asin": asin, "attr_id": "scent",
            "value_text": "unscented", "value_num": None, "unit": "",
            "confidence": 0.76, "support_sources": "fallback.raw_text"
        })

# ---------- 5) target_audience Fallback — 문장 패턴 ----------
AUD_PATTERNS_FALLBACK = [
    (re.compile(r"\bfor\s+men\b|\bmen's\b", re.I), "men"),
    (re.compile(r"\bfor\s+women\b|\bwomen's\b|\bfor\s+her\b", re.I), "women"),
    (re.compile(r"\bunisex\b", re.I), "unisex"),
    (re.compile(r"\bbaby\b", re.I), "baby"),
    (re.compile(r"\bkids?\b", re.I), "kids"),
    (re.compile(r"\bteens?\b", re.I), "teens"),
    (re.compile(r"\bsensitive\s+skin\b", re.I), "sensitive"),
    (re.compile(r"\bdry\s+skin\b", re.I), "dry"),
    (re.compile(r"\boily\s+skin\b", re.I), "oily"),
    (re.compile(r"\bcombination\s+skin\b", re.I), "combination"),
    (re.compile(r"\bmature\s+skin|aging\s+skin\b", re.I), "aging"),
    (re.compile(r"\bcurly\s+hair\b", re.I), "curly"),
    (re.compile(r"\bcolor[-\s]*treated\s+hair\b", re.I), "color-treated"),
    (re.compile(r"\bdamaged\s+hair\b", re.I), "damaged"),
]
have_aud_asins = set(item_attr_long.loc[item_attr_long["attr_id"]=="target_audience","parent_asin"])
need_aud_asins = set(meta_df_text["parent_asin"]) - have_aud_asins
rows_fb_aud = []
for asin in need_aud_asins:
    txt = str(raw_map.get(asin, "")).lower()
    vals = set()
    for pat, lab in AUD_PATTERNS_FALLBACK:
        if pat.search(txt):
            vals.add(lab)
    for v in vals:
        rows_fb_aud.append({
            "parent_asin": asin, "attr_id": "target_audience",
            "value_text": v, "value_num": None, "unit": "",
            "confidence": 0.72, "support_sources": "fallback.raw_text"
        })

# ---------- 6) 결합 + 중복 정리 ----------
new_chunks = []
if rows_direct_scent: new_chunks.append(pd.DataFrame(rows_direct_scent))
if rows_direct_aud:   new_chunks.append(pd.DataFrame(rows_direct_aud))
if rows_fb_pt:        new_chunks.append(pd.DataFrame(rows_fb_pt))
if rows_fb_scent:     new_chunks.append(pd.DataFrame(rows_fb_scent))
if rows_fb_aud:       new_chunks.append(pd.DataFrame(rows_fb_aud))

if new_chunks:
    add_df = pd.concat(new_chunks, ignore_index=True)
    # 기존 테이블과 결합
    item_attr_long = pd.concat([item_attr_long, add_df], ignore_index=True)
    # (parent_asin, attr_id, value_text) 기준으로 confidence 높은 것 남기기
    # confidence_adj는 이후 계산하므로 일단 confidence 우선
    item_attr_long.sort_values(
        ["parent_asin","attr_id","value_text","confidence"],
        ascending=[True, True, True, False],
        inplace=True, na_position="last"
    )
    item_attr_long = item_attr_long.drop_duplicates(
        subset=["parent_asin","attr_id","value_text"], keep="first"
    ).reset_index(drop=True)

# ---------- 7) confidence_adj / hard_pass 재계산 ----------
item_attr_long["confidence_adj"] = item_attr_long["confidence"] + item_attr_long["support_sources"].fillna("").apply(consensus_bonus)
item_attr_long["confidence_adj"] = item_attr_long["confidence_adj"].clip(upper=1.0)

def _hard_pass(row):
    thr = THRESH_BY_ATTR.get(row["attr_id"], 0.8)
    try:
        return float(row["confidence_adj"]) >= float(thr)
    except Exception:
        return False

item_attr_long["hard_pass"] = item_attr_long.apply(_hard_pass, axis=1)

# ---------- 8) 커버리지 리포트 ----------
def coverage_v2(df):
    rows=[]
    total = meta_df_text["parent_asin"].nunique()
    for a, g in df.groupby("attr_id"):
        rows.append({
            "attr_id": a,
            "items_with_attr": g["parent_asin"].nunique(),
            "coverage_pct": round(100*g['parent_asin'].nunique()/max(1,total),1),
            "hard_ready_items": g.loc[g["hard_pass"],"parent_asin"].nunique(),
            "hard_ready_pct": round(100*g.loc[g["hard_pass"],"parent_asin"].nunique()/max(1,total),1),
            "unique_text_values": int(g["value_text"].nunique()),
            "unique_num_values": int(g["value_num"].dropna().nunique()),
            "avg_conf_adj": round(float(g["confidence_adj"].mean()),3),
        })
    return pd.DataFrame(rows).sort_values(["hard_ready_pct","coverage_pct","attr_id"], ascending=[False,False,True])

cov_after_6C = coverage_v2(item_attr_long)

print("▶ CELL 6C 완료 — Direct + Fallback 적용 및 하드패스 재계산")
display(cov_after_6C)


/tmp/ipykernel_18119/186462359.py:224: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  item_attr_long = pd.concat([item_attr_long, add_df], ignore_index=True)


▶ CELL 6C 완료 — Direct + Fallback 적용 및 하드패스 재계산


,attr_id,items_with_attr,coverage_pct,hard_ready_items,hard_ready_pct,unique_text_values,unique_num_values,avg_conf_adj
0,brand,11282,91.0,11282,91.0,6701,0,0.931
1,price_num,2249,18.1,2249,18.1,0,952,0.950
4,target_audience,5430,43.8,2091,16.9,18,0,0.733
2,product_type,3972,32.0,1679,13.5,18,0,0.809
5,volume_ml,3030,24.4,750,6.0,0,419,0.667
3,scent,1452,11.7,712,5.7,22,0,0.824


In [16]:
# ==========================================================
# CELL 6D / 커버리지 리포트 v2 + 품질 점검 출력  [한글 주석]
# 목적: (6B/6C 적용 후) 속성별 커버리지/하드 통과율/유니크 값 개수 요약
#       + 샘플 레코드로 최종 추출 품질 점검
# 입력: item_attr_long, meta_df_text
# 출력: attr_coverage_report_v2, 샘플 미리보기
# ==========================================================

import pandas as pd

if "item_attr_long" not in locals():
    raise RuntimeError("item_attr_long이 없습니다.")

def attr_report_v2(df):
    total = meta_df_text["parent_asin"].nunique()
    rows=[]
    for a, g in df.groupby("attr_id"):
        rows.append({
            "attr_id": a,
            "items_with_attr": g["parent_asin"].nunique(),
            "coverage_pct": round(100*g['parent_asin'].nunique()/max(1,total),1),
            "hard_ready_items": g.loc[g["hard_pass"],"parent_asin"].nunique(),
            "hard_ready_pct": round(100*g.loc[g["hard_pass"],"parent_asin"].nunique()/max(1,total),1),
            "unique_text_values": int(g["value_text"].nunique()),
            "unique_num_values": int(g["value_num"].dropna().nunique()),
            "avg_conf_adj": round(float(g["confidence_adj"].mean()),3),
        })
    return pd.DataFrame(rows).sort_values(["hard_ready_pct","coverage_pct","attr_id"], ascending=[False,False,True])

attr_coverage_report_v2 = attr_report_v2(item_attr_long)

print("▶ CELL 6D 완료 — 커버리지 리포트 v2")
display(attr_coverage_report_v2)

print("\n[샘플 — 브랜드/제품유형/용량/향/대상/가격 각 3건]")
def show_samples(aid, k=3):
    sub = item_attr_long[item_attr_long["attr_id"]==aid].sort_values(["hard_pass","confidence_adj"], ascending=[False,False]).head(k)
    cols = ["parent_asin","attr_id","value_text","value_num","unit","confidence","confidence_adj","hard_pass","support_sources"]
    display(sub[cols])

for aid in ["brand","product_type","volume_ml","scent","target_audience","price_num"]:
    if aid in item_attr_long["attr_id"].unique():
        print(f"\n— {aid} —")
        show_samples(aid, 3)


▶ CELL 6D 완료 — 커버리지 리포트 v2


,attr_id,items_with_attr,coverage_pct,hard_ready_items,hard_ready_pct,unique_text_values,unique_num_values,avg_conf_adj
0,brand,11282,91.0,11282,91.0,6701,0,0.931
1,price_num,2249,18.1,2249,18.1,0,952,0.950
4,target_audience,5430,43.8,2091,16.9,18,0,0.733
2,product_type,3972,32.0,1679,13.5,18,0,0.809
5,volume_ml,3030,24.4,750,6.0,0,419,0.667
3,scent,1452,11.7,712,5.7,22,0,0.824



[샘플 — 브랜드/제품유형/용량/향/대상/가격 각 3건]

— brand —


,parent_asin,attr_id,value_text,value_num,unit,confidence,confidence_adj,hard_pass,support_sources
0,0005946468,brand,patanjali,NaN,,0.95,0.95,True,details.Brand
2,1625684320,brand,tata harper,NaN,,0.95,0.95,True,details.Brand
5,7458592737,brand,yardley,NaN,,0.95,0.95,True,details.Brand



— product_type —


,parent_asin,attr_id,value_text,value_num,unit,confidence,confidence_adj,hard_pass,support_sources
15,B000052WYR,product_type,lotion,NaN,,0.89375,1.0,True,"description,details_str,title"
23,B0000ZLF18,product_type,oil,NaN,,1.00000,1.0,True,"description,details_str,feature_str,title"
53,B00028NLKG,product_type,cream,NaN,,0.87500,1.0,True,"description,feature_str,title"


... (4 outputs left out)


— target_audience —


,parent_asin,attr_id,value_text,value_num,unit,confidence,confidence_adj,hard_pass,support_sources
141,B000COKYTG,target_audience,dry,NaN,,0.894531,1.0,True,"description,details_str,feature_str"
142,B000COKYTG,target_audience,normal,NaN,,0.894531,1.0,True,"description,details_str,feature_str"
433,B001CDGVWA,target_audience,men,NaN,,0.894531,1.0,True,"description,details_str,feature_str"



— price_num —


,parent_asin,attr_id,value_text,value_num,unit,confidence,confidence_adj,hard_pass,support_sources
10,979077530X,price_num,None,51.99,usd,0.95,0.95,True,meta_price
22,B0000ZLF18,price_num,None,9.81,usd,0.95,0.95,True,meta_price
28,B000143M46,price_num,None,7.95,usd,0.95,0.95,True,meta_price


In [17]:
# ==========================================================
# CELL 7 / 리뷰 집계 + 품질 리포트 (JSONL 직로딩, trust_remote_code 미사용) [완전 실행형]
# - 리뷰는 Parquet가 아니라: raw/review_categories/<Category>.jsonl
# - 로컬로 1개 JSONL만 받아서 집계/리포트
# - 전제: meta_df_text 존재(CELL 4까지 완료)
# ==========================================================
import os, sys, subprocess
import pandas as pd, numpy as np

# ---------- 0) 선행 체크 & 파라미터 ----------
if "meta_df_text" not in globals():
    raise RuntimeError("meta_df_text가 없습니다. CELL 4까지 먼저 실행하세요.")

CATEGORY_REVIEWS   = "All_Beauty"     # 예: All_Beauty / Beauty_and_Personal_Care 등 (HF 파일명 그대로)
REPO_ID            = "McAuley-Lab/Amazon-Reviews-2023"
LOCAL_DIR          = os.path.join(BASE_DIR, "amazon_reviews_2023")
# (제출 후 수정) 제출본은 N_REVIEW_SAMPLE(앞에서부터 N건)을 여기서도 썼다. 이제는 상품으로 먼저 거르고,
# 그 뒤에만 상한을 적용한다. None=남은 상품에 연결된 리뷰 전부 사용
REVIEW_LINKED_CAP  = None

os.makedirs(LOCAL_DIR, exist_ok=True)
PARENT_SET   = set(meta_df_text["parent_asin"].astype(str))
TOTAL_ITEMS  = len(PARENT_SET)

# ---------- 1) 의존성 보장 ----------
def _pip(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
for p in ["huggingface_hub>=0.24.0", "datasets>=2.20.0", "pyarrow>=14.0.1", "fsspec>=2023.10.0"]:
    try:
        __import__(p.split(">=")[0].replace("-", "_"))
    except Exception:
        _pip(p)

from huggingface_hub import hf_hub_download
from datasets import load_dataset

# ---------- 2) JSONL 단일 파일 다운로드 ----------
jsonl_rel = f"raw/review_categories/{CATEGORY_REVIEWS}.jsonl"
local_jsonl = hf_hub_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    filename=jsonl_rel,
    local_dir=LOCAL_DIR,
    local_dir_use_symlinks=False,
)
if not os.path.exists(local_jsonl):
    raise FileNotFoundError(f"리뷰 JSONL이 보이지 않습니다: {local_jsonl}")

# ---------- 3) JSONL 로드 (datasets/json) ----------
ds = load_dataset("json", data_files=local_jsonl, split="train")  # trust_remote_code 사용 안 함
n_total_reviews = len(ds)

# (제출 후 수정) 제출본은 리뷰를 앞에서부터 N건 자른 뒤 남은 상품으로 걸러서,
# 리뷰 12,358건 중 1,424건(상품 967개, 8.6%)만 연결됐다.
# 남은 상품(PARENT_SET)의 리뷰를 먼저 고르고, 상한은 그다음에 적용한다.
_pcol = "parent_asin" if "parent_asin" in ds.column_names else "parent"
ds = ds.filter(lambda b: [str(p) in PARENT_SET for p in b[_pcol]], batched=True)
n_linked_reviews = len(ds)
if isinstance(REVIEW_LINKED_CAP, int) and 0 < REVIEW_LINKED_CAP < n_linked_reviews:
    ds = ds.select(range(REVIEW_LINKED_CAP))
print(f"- 리뷰 전체 {n_total_reviews:,}건 → 남은 상품에 연결된 리뷰 {n_linked_reviews:,}건 → 사용 {len(ds):,}건")

df = ds.to_pandas()  # All_Beauty는 ~수백 MB 수준 → Colab 가능

# ---------- 4) 컬럼 정리 ----------
def pick_col(cols, cands, default=None):
    for c in cands:
        if c in cols: return c
    return default

rating_col   = pick_col(df.columns, ["rating","star_rating","overall"])
parent_col   = pick_col(df.columns, ["parent_asin","parent"])
verified_col = pick_col(df.columns, ["verified_purchase","verified","is_verified"])
helpful_col  = pick_col(df.columns, ["helpful_vote","helpful_votes","vote","votes"])
ts_cols_try  = ["review_date","reviewTime","date","timestamp","review_timestamp","time","unixReviewTime","unix_time"]

if rating_col is None or parent_col is None:
    raise RuntimeError(f"필수 컬럼 누락: rating={rating_col} / parent_asin={parent_col}")

use = [c for c in [rating_col,parent_col,verified_col,helpful_col] if c is not None]
use += [c for c in ts_cols_try if c in df.columns]
df = df[use].copy()
df.rename(columns={rating_col:"rating", parent_col:"parent_asin"}, inplace=True)

# 타입/값 보정
df["parent_asin"] = df["parent_asin"].astype(str)
df = df[df["parent_asin"].isin(PARENT_SET)]

df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
df = df[(df["rating"]>=1) & (df["rating"]<=5)]

if verified_col:
    x = df[verified_col].astype(str).str.lower().str.strip()
    df["verified_flag"] = x.isin(["1","y","yes","true","t"])
else:
    df["verified_flag"] = pd.NA

if helpful_col:
    hv = pd.to_numeric(df[helpful_col], errors="coerce").fillna(0)
    hv[hv<0] = 0
    df["helpful_vote"] = hv
else:
    df["helpful_vote"] = 0

# 날짜 정규화(문자/epoch 혼재 대응)
def coerce_dt_any(dfx):
    for c in ["review_date","reviewTime","date"]:
        if c in dfx.columns:
            dt = pd.to_datetime(dfx[c], errors="coerce", utc=True)
            if dt.notna().any(): return dt
    for c in ["timestamp","review_timestamp","time","unixReviewTime","unix_time"]:
        if c in dfx.columns:
            s = pd.to_numeric(dfx[c], errors="coerce")
            if s.notna().sum()==0:
                continue
            mx = s.max()
            unit = "ms" if (pd.notna(mx) and mx>10**12) else "s"
            dt = pd.to_datetime(s, unit=unit, errors="coerce", utc=True)
            if dt.notna().any(): return dt
    return pd.Series([pd.NaT]*len(dfx), dtype="datetime64[ns, UTC]")

df["review_dt"] = coerce_dt_any(df)

# ---------- 5) 집계 ----------
def safe_ratio(num, den):
    den = den if pd.notnull(den) and den>0 else np.nan
    return float(num)/float(den) if pd.notnull(den) else np.nan

g = df.groupby("parent_asin", as_index=False)
review_agg = g.agg(
    avg_rating=("rating","mean"),
    rating_cnt=("rating","size"),
    helpful_sum=("helpful_vote","sum"),
    first_review_date=("review_dt","min"),
    last_review_date=("review_dt","max"),
    verified_sum=("verified_flag","sum"),
)
review_agg["verified_ratio"] = review_agg.apply(lambda r: safe_ratio(r["verified_sum"], r["rating_cnt"]), axis=1)
review_agg.drop(columns=["verified_sum"], inplace=True)

today_utc = pd.Timestamp.utcnow().normalize()
review_agg["days_since_last_review"] = (today_utc - review_agg["last_review_date"]).dt.days
review_agg.sort_values("rating_cnt", ascending=False, inplace=True, ignore_index=True)

# ---------- 6) 품질 리포트 ----------
items_with_reviews = review_agg["parent_asin"].nunique()
coverage_pct = round(100*items_with_reviews/max(1,TOTAL_ITEMS), 1)
rating_hist = df["rating"].value_counts(normalize=True).sort_index()
rating_hist_str = " / ".join([f"{int(k)}: {round(v*100,1)}%" for k,v in rating_hist.items()])

if df["verified_flag"].isna().all():
    verified_cov = 0.0
    verified_mean = np.nan
    verified_med  = np.nan
else:
    verified_cov  = round(100*df["verified_flag"].notna().mean(), 1)
    verified_mean = round(review_agg["verified_ratio"].mean(skipna=True), 3)
    verified_med  = round(review_agg["verified_ratio"].median(skipna=True), 3)

helpful_pos_share = round(100*(df["helpful_vote"]>0).mean(), 1)
helpful_mean = round(df["helpful_vote"].mean(), 2)
helpful_sum_total = int(df["helpful_vote"].sum())

def pct_recent(days):
    sub = review_agg["days_since_last_review"].dropna()
    return round(100*(sub<=days).mean(), 1) if len(sub)>0 else 0.0
pct_30, pct_90, pct_180, pct_365 = pct_recent(30), pct_recent(90), pct_recent(180), pct_recent(365)

bins   = [1,2,5,10,25,50,100,1_000_000]
labels = ["1","2-4","5-9","10-24","25-49","50-99","100+"]
review_agg["cnt_bin"] = pd.cut(review_agg["rating_cnt"], bins=bins, labels=labels, right=False, include_lowest=True)
bin_dist = review_agg["cnt_bin"].value_counts(normalize=True).reindex(labels).fillna(0.0)
bin_dist_str = " / ".join([f"{lab}: {round(p*100,1)}%" for lab,p in bin_dist.items()])

low_bad   = review_agg[(review_agg["avg_rating"]<2.0) & (review_agg["rating_cnt"]>=20)].shape[0]
high_susp = review_agg[(review_agg["avg_rating"]>4.8) & (review_agg["rating_cnt"]>=20)].shape[0]

report_lines = []
report_lines.append("=== REVIEW QUALITY REPORT / CELL7 (JSONL) ===")
report_lines.append(f"- category file: {CATEGORY_REVIEWS}.jsonl")
report_lines.append(f"- meta items: {TOTAL_ITEMS:,}")
report_lines.append(f"- items with reviews: {items_with_reviews:,} ({coverage_pct}%)")
report_lines.append(f"- loaded rows: {len(df):,} (source: {jsonl_rel})")
report_lines.append("")
report_lines.append("[rating distribution]")
report_lines.append(f"- 1/2/3/4/5 share: {rating_hist_str}")
report_lines.append("")
report_lines.append("[verified purchase]")
report_lines.append(f"- verified_flag availability: {verified_cov}%")
report_lines.append(f"- verified_ratio mean/median: {verified_mean} / {verified_med}")
report_lines.append("")
report_lines.append("[helpful votes]")
report_lines.append(f"- helpful>0 share: {helpful_pos_share}% / mean helpful: {helpful_mean} / total: {helpful_sum_total:,}")
report_lines.append("")
report_lines.append("[recency]")
report_lines.append(f"- last review within 30/90/180/365 days: {pct_30}% / {pct_90}% / {pct_180}% / {pct_365}%")
report_lines.append("")
report_lines.append("[rating count bins]")
report_lines.append(f"- {bin_dist_str}")
report_lines.append("")
report_lines.append("[outliers]")
report_lines.append(f"- avg_rating<2.0 and rating_cnt>=20: {low_bad}")
report_lines.append(f"- avg_rating>4.8 and rating_cnt>=20: {high_susp}")
print("\n".join(report_lines))

print("\n[Top by rating_cnt / head 5]")
display(review_agg[["parent_asin","avg_rating","rating_cnt","verified_ratio","helpful_sum","last_review_date"]].head(5))

print("\n[Top by helpful_sum / head 5]")
display(review_agg.sort_values("helpful_sum", ascending=False)[["parent_asin","avg_rating","rating_cnt","verified_ratio","helpful_sum","last_review_date"]].head(5))

# 결과 객체 유지
reviews_df = df


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Filter:   0%|          | 0/701528 [00:00<?, ? examples/s]

- 리뷰 전체 701,528건 → 남은 상품에 연결된 리뷰 91,631건 → 사용 91,631건
=== REVIEW QUALITY REPORT / CELL7 (JSONL) ===
- category file: All_Beauty.jsonl
- meta items: 12,402
- items with reviews: 12,402 (100.0%)
- loaded rows: 91,631 (source: raw/review_categories/All_Beauty.jsonl)

[rating distribution]
- 1/2/3/4/5 share: 1: 14.0% / 2: 6.1% / 3: 8.0% / 4: 10.9% / 5: 61.0%

[verified purchase]
- verified_flag availability: 100.0%
- verified_ratio mean/median: 0.905 / 1.0

[helpful votes]
- helpful>0 share: 27.1% / mean helpful: 0.94 / total: 86,226

[recency]
- last review within 30/90/180/365 days: 0.0% / 0.0% / 0.0% / 0.0%

[rating count bins]
- 1: 38.1% / 2-4: 34.5% / 5-9: 13.7% / 10-24: 8.8% / 25-49: 2.7% / 50-99: 1.3% / 100+: 1.0%

[outliers]
- avg_rating<2.0 and rating_cnt>=20: 8
- avg_rating>4.8 and rating_cnt>=20: 21

[Top by rating_cnt / head 5]


,parent_asin,avg_rating,rating_cnt,verified_ratio,helpful_sum,last_review_date
0,B09X9BG4FC,4.496361,1374,0.958515,588,2023-08-08 23:45:23.135000+00:00
1,B01N4885PD,3.432980,1037,0.949855,353,2023-08-24 16:25:30.872000+00:00
2,B09C2ZW2GL,4.441084,959,0.980188,465,2023-07-14 11:37:07.204000+00:00
3,B09LHS6RGF,3.988649,881,0.976163,1575,2023-07-27 23:35:31.219000+00:00
4,B08F4Q8ZBR,4.509804,561,0.985740,643,2023-08-06 09:51:20.591000+00:00



[Top by helpful_sum / head 5]


,parent_asin,avg_rating,rating_cnt,verified_ratio,helpful_sum,last_review_date
89,B09K9JHDNX,4.582609,115,0.721739,2430,2023-05-23 01:01:57.181000+00:00
14,B07R1K1SBD,4.432277,347,0.979827,2142,2023-08-11 14:15:57.365000+00:00
3,B09LHS6RGF,3.988649,881,0.976163,1575,2023-07-27 23:35:31.219000+00:00
69,B01C7532DM,4.226027,146,0.849315,1268,2021-11-01 23:01:40.491000+00:00
72,B07KXVGHTV,3.908451,142,0.978873,1218,2022-03-02 20:35:55.154000+00:00


In [18]:
# ==========================================================
# CELL 7.1 / Review-derived features (Bayes/Wilson/Popularity/Recency) [완전 실행형]
# 입력: reviews_df, review_agg (CELL 7에서 생성됨)
# 출력: review_agg_feats (후속 조인/랭킹용), 콘솔 미니 리포트
# ==========================================================
import numpy as np
import pandas as pd

# --- 선행 체크 ---
need = ["reviews_df", "review_agg"]
missing = [v for v in need if v not in globals()]
if missing:
    raise RuntimeError(f"CELL 7.1 선행 객체 없음: {missing}  (CELL 7 먼저 실행)")

# --- 파라미터 (가중치는 데모용 기본값; CELL 11에서 재조정 가능) ---
BAYES_PRIOR_WEIGHT = 10          # α: 베이즈 보정 강도(10~20 권장)
POS_THRESH         = 4.0         # 긍정 기준 별점(4~5)
W = {
    "bayes":   0.50,             # 베이즈 평균 가중치
    "wilson":  0.20,             # 윌슨 하한(긍정비율) 가중치
    "pop":     0.15,             # 인기도(log1p(review_cnt)) 가중치
    "helpful": 0.10,             # 도움됨 정규화 가중치
    "recency": 0.05,             # 상대 최신성 가중치(절대 최신성은 0이므로 내부 상대값 사용)
}

# --- 1) 전역 평균 μ(평균 별점) ---
mu = (reviews_df["rating"].mean()).clip(1.0, 5.0)

# --- 2) 상품별 긍정 리뷰수(>=POS_THRESH) 계산 ---
pos_map = (
    reviews_df.assign(is_pos = reviews_df["rating"] >= POS_THRESH)
              .groupby("parent_asin")["is_pos"].sum()
              .astype("Int64")
              .rename("pos_cnt")
)
agg = review_agg.merge(pos_map, on="parent_asin", how="left")
agg["pos_cnt"] = agg["pos_cnt"].fillna(0).astype("Int64")

# --- 3) 베이즈 평균 (μ·α + Σrating) / (α + n) ---
agg["sum_rating"]  = agg["avg_rating"] * agg["rating_cnt"]
agg["bayes_mean"]  = (mu * BAYES_PRIOR_WEIGHT + agg["sum_rating"]) / (BAYES_PRIOR_WEIGHT + agg["rating_cnt"].clip(lower=0.0))

# --- 4) 윌슨 하한 (긍정비율의 95% 하한) ---
def wilson_lower(p, n, z=1.96):
    if n <= 0 or pd.isna(p):
        return np.nan
    denom = 1 + z*z/n
    centre = p + z*z/(2*n)
    adj = z * np.sqrt((p*(1-p) + z*z/(4*n))/n)
    return (centre - adj) / denom

agg["pos_ratio"]   = agg.apply(lambda r: float(r["pos_cnt"])/float(r["rating_cnt"]) if r["rating_cnt"]>0 else np.nan, axis=1)
agg["wilson_lb"]   = agg.apply(lambda r: wilson_lower(r["pos_ratio"], r["rating_cnt"], 1.96), axis=1)

# --- 5) 인기도·도움됨·상대 최신성 ---
agg["pop_log"]     = np.log1p(agg["rating_cnt"])
agg["helpful_per"] = (agg["helpful_sum"] / agg["rating_cnt"]).replace([np.inf, -np.inf], np.nan)

# 내부 상대 최신성 (데이터 내부 min/max 기준 정규화)
min_dt = agg["first_review_date"].min()
max_dt = agg["last_review_date"].max()
if pd.isna(min_dt) or pd.isna(max_dt) or min_dt == max_dt:
    agg["recency_rel"] = 0.0
else:
    # 최근일수 작을수록 큰 값이 되도록: 1 - (max - last) / (max - min)
    age = (max_dt - agg["last_review_date"]).dt.total_seconds()
    span = (max_dt - min_dt).total_seconds()
    agg["recency_rel"] = 1.0 - (age / span)
    agg.loc[agg["recency_rel"].isna(), "recency_rel"] = 0.0
    agg["recency_rel"] = agg["recency_rel"].clip(0, 1)

# --- 6) 0~1 정규화 (min-max, 안전 처리) ---
def mm_norm(s):
    s = s.astype(float)
    lo, hi = np.nanmin(s), np.nanmax(s)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - lo) / (hi - lo)

agg["bayes_norm"]   = mm_norm(agg["bayes_mean"])
agg["wilson_norm"]  = mm_norm(agg["wilson_lb"])
agg["pop_norm"]     = mm_norm(agg["pop_log"])
agg["helpful_norm"] = mm_norm(agg["helpful_per"])

# --- 7) 종합 리뷰 신호 (가중합) ---
agg["review_signal"] = (
    W["bayes"]   * agg["bayes_norm"]  +
    W["wilson"]  * agg["wilson_norm"] +
    W["pop"]     * agg["pop_norm"]    +
    W["helpful"] * agg["helpful_norm"]+
    W["recency"] * agg["recency_rel"]
)

# --- 8) 정리 ---
cols = [
    "parent_asin","avg_rating","rating_cnt","helpful_sum","verified_ratio",
    "bayes_mean","wilson_lb","pop_log","helpful_per","recency_rel",
    "bayes_norm","wilson_norm","pop_norm","helpful_norm","review_signal",
    "first_review_date","last_review_date","days_since_last_review"
]
review_agg_feats = agg[cols].sort_values("review_signal", ascending=False).reset_index(drop=True)

# --- 9) 미니 리포트 ---
print("=== CELL 7.1 REVIEW SIGNAL SUMMARY ===")
print(f"- global mean rating μ: {mu:.3f} / prior α: {BAYES_PRIOR_WEIGHT}")
print(f"- recency window (internal): {str(min_dt)} → {str(max_dt)}")
print(f"- top review_signal head(5):")
display(review_agg_feats[["parent_asin","review_signal","bayes_mean","wilson_lb","pop_log","helpful_per","recency_rel"]].head(5))

print("\n[분포 요약 (0~1 정규화 기준)]")
for k in ["bayes_norm","wilson_norm","pop_norm","helpful_norm","recency_rel","review_signal"]:
    s = review_agg_feats[k].dropna()
    if len(s)==0:
        print(f"- {k}: n=0")
        continue
    print(f"- {k}: n={len(s)} / mean={s.mean():.3f} / p50={s.median():.3f} / p90={s.quantile(0.9):.3f} / max={s.max():.3f}")


=== CELL 7.1 REVIEW SIGNAL SUMMARY ===
- global mean rating μ: 3.989 / prior α: 10
- recency window (internal): 2003-12-28 01:11:13+00:00 → 2023-09-02 19:29:15.060000+00:00
- top review_signal head(5):


,parent_asin,review_signal,bayes_mean,wilson_lb,pop_log,helpful_per,recency_rel
0,B09FFQT1KK,0.875785,4.781755,0.921352,6.259581,0.291188,0.979114
1,B0888TZ9W6,0.853176,4.751526,0.907486,5.517453,2.020161,0.995311
2,B09F3LF24J,0.835845,4.688089,0.902092,5.459586,0.303419,0.956459
3,B011KJ084M,0.832764,4.721277,0.896031,5.398163,0.286364,0.834328
4,B09X9BG4FC,0.832493,4.492698,0.846285,7.226209,0.427948,0.996547



[분포 요약 (0~1 정규화 기준)]
- bayes_norm: n=12402 / mean=0.715 / p50=0.747 / p90=0.804 / max=1.000
- wilson_norm: n=12402 / mean=0.275 / p50=0.224 / p90=0.584 / max=1.000
- pop_norm: n=12402 / mean=0.110 / p50=0.062 / p90=0.298 / max=1.000
- helpful_norm: n=12402 / mean=0.007 / p50=0.000 / p90=0.016 / max=1.000
- recency_rel: n=12402 / mean=0.829 / p50=0.855 / p90=0.948 / max=1.000
- review_signal: n=12402 / mean=0.471 / p50=0.463 / p90=0.591 / max=0.876


In [19]:
# ==========================================================
# CELL 8 / 메타+속성+리뷰 조인 & 파생 bin 생성  [완전 실행형]
# 목적:
#   1) item_attr_long에서 속성별 "최적 1개" 후보를 선별(하드패스/신뢰도/근거수)
#   2) meta_df_text(타이틀) + 선택 속성 + review_agg_feats(리뷰신호) 조인
#   3) 가격/용량 bin 생성(도메인 프리셋 또는 분위수)
#   4) 품질 리포트(커버리지/분포) 출력
# 산출:
#   item_table : 추천/필터/랭킹에 바로 쓰는 단일 테이블
# ==========================================================
import pandas as pd
import numpy as np

# ----------------------- 선행 체크 -----------------------
need = ["meta_df_text", "item_attr_long", "review_agg_feats"]
missing = [n for n in need if n not in globals()]
if missing:
    raise RuntimeError(f"CELL 8 선행 객체 없음: {missing}  (CELL 4, 6, 7.1 먼저 실행)")

# ------------------ 파라미터(필요시 조정) ------------------
# bin 스킴: "domain_beauty" 또는 "quantile"
PRICE_BIN_SCHEME  = "domain_beauty"
VOLUME_BIN_SCHEME = "domain_beauty"

# 분위수 bin 갯수(quantile 사용 시)
PRICE_Q = 5
VOL_Q   = 5

# 선별 우선순위 가중치(하드패스/신뢰도/근거수/텍스트길이/숫자유무)
W_HARD     = 10000
W_CONF     = 1000
W_SUPPORT  = 10
W_TXTLEN   = 1
W_NUMPREF  = 500   # 숫자값(value_num) 보유 항목에 보너스(용량/가격 속성에만 적용)

# 조인 후 보존할 리뷰컬럼
REVIEW_KEEP = [
    "review_signal","avg_rating","rating_cnt","verified_ratio","helpful_sum",
    "first_review_date","last_review_date","days_since_last_review"
]

# ------------------ 1) 속성 최적값 선별 ------------------
df = item_attr_long.copy()

# 안전 캐스팅
for c in ["attr_id","value_text","unit","support_sources"]:
    if c in df.columns:
        df[c] = df[c].astype(str)
for c in ["value_num","confidence","confidence_adj"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df["is_hard"]   = df["hard_pass"] if "hard_pass" in df.columns else False
df["conf_adj"]  = df["confidence_adj"] if "confidence_adj" in df.columns else df.get("confidence", np.nan)
df["support_n"] = df["support_sources"].fillna("").str.split(",").apply(lambda L: len([x for x in L if x.strip()])) if "support_sources" in df.columns else 0

wanted = ["brand","product_type","target_audience","scent","volume_ml","price_num"]
df = df[df["attr_id"].isin(wanted)].copy()

def _pick_best_attr(group: pd.DataFrame) -> pd.Series:
    attr = group.name[1]  # (parent_asin, attr_id)
    g = group.copy()

    # 기본 점수
    base = (g["is_hard"].astype(int)*W_HARD) + (g["conf_adj"].fillna(0)*W_CONF) + (g["support_n"].fillna(0)*W_SUPPORT)
    # 텍스트 길이(브랜드/타입/향/대상) 보너스(간단한 타이브레이커)
    txtlen = g["value_text"].fillna("").str.len() if "value_text" in g.columns else 0
    base = base + (txtlen * W_TXTLEN)

    # 숫자 속성에선 value_num 보유 가산
    if attr in ["price_num","volume_ml"]:
        base = base + (g["value_num"].notna().astype(int)*W_NUMPREF)

    # 최고 점수 1건 선택
    idx = base.idxmax()
    row = g.loc[idx]

    # 반환: 숫자/문자 열 모두 리턴(나중에 attr별로 골라씀)
    return pd.Series({
        "chosen_text": row.get("value_text", np.nan),
        "chosen_num":  row.get("value_num", np.nan),
        "unit":        row.get("unit", ""),
        "is_hard":     bool(row.get("is_hard", False)),
        "conf_used":   row.get("conf_adj", row.get("confidence", np.nan)),
        "support_n":   row.get("support_n", 0)
    })

picked = (
    df.groupby(["parent_asin","attr_id"], as_index=False)
      .apply(_pick_best_attr)
      .reset_index()  # groupby apply 멀티인덱스 정리
)

# wide 변환
def _colmap(attr, kind):  # kind: text/num
    return f"{attr}" if kind=="text" else f"{attr}"

wide_text = picked.pivot(index="parent_asin", columns="attr_id", values="chosen_text")
wide_num  = picked.pivot(index="parent_asin", columns="attr_id", values="chosen_num")

# 최종 속성 칼럼 만들기(문자형은 텍스트, 숫자형은 숫자)
attr_wide = pd.DataFrame(index=wide_text.index.union(wide_num.index))
for a in wanted:
    if a in ["price_num","volume_ml"]:
        # 숫자 우선, 텍스트는 무시
        if a in wide_num.columns:
            attr_wide[a] = pd.to_numeric(wide_num[a], errors="coerce")
        else:
            attr_wide[a] = np.nan
    else:
        # 텍스트 사용
        if a in wide_text.columns:
            attr_wide[a] = wide_text[a].astype(object)
        else:
            attr_wide[a] = np.nan

attr_wide = attr_wide.reset_index().rename(columns={"index":"parent_asin"})

# ------------------ 2) 메타/리뷰 조인 ------------------
meta_min = meta_df_text[["parent_asin","title"]].drop_duplicates()
item_table = (
    meta_min
    .merge(attr_wide, on="parent_asin", how="left")
    .merge(review_agg_feats[["parent_asin"] + REVIEW_KEEP], on="parent_asin", how="left")
)

# ------------------ 3) 가격/용량 bin ------------------
def make_price_bin(series):
    s = pd.to_numeric(series, errors="coerce")
    if PRICE_BIN_SCHEME == "quantile":
        qs = s.dropna().quantile(np.linspace(0,1,PRICE_Q+1)).values
        # 경계 중복 방어
        qs = np.unique(qs)
        if len(qs) < 3:  # 데이터 부족 시 도메인 프리셋으로 대체
            edges = [0,10,20,35,60,100,np.inf]
            labels = ["<=10","10-20","20-35","35-60","60-100","100+"]
        else:
            edges = qs
            labels = [f"Q{i+1}" for i in range(len(edges)-1)]
    else:
        edges  = [0,10,20,35,60,100,np.inf]
        labels = ["<=10","10-20","20-35","35-60","60-100","100+"]
    return pd.cut(s, bins=edges, labels=labels, include_lowest=True, right=False)

def make_vol_bin(series):
    s = pd.to_numeric(series, errors="coerce")
    if VOLUME_BIN_SCHEME == "quantile":
        qs = s.dropna().quantile(np.linspace(0,1,VOL_Q+1)).values
        qs = np.unique(qs)
        if len(qs) < 3:
            edges  = [0,15,30,75,150,300,600,np.inf]
            labels = ["<15ml","15-30","30-75","75-150","150-300","300-600","600+"]
        else:
            edges = qs
            labels = [f"Q{i+1}" for i in range(len(edges)-1)]
    else:
        edges  = [0,15,30,75,150,300,600,np.inf]
        labels = ["<15ml","15-30","30-75","75-150","150-300","300-600","600+"]
    return pd.cut(s, bins=edges, labels=labels, include_lowest=True, right=False)

item_table["price_bin"]  = make_price_bin(item_table["price_num"])
item_table["volume_bin"] = make_vol_bin(item_table["volume_ml"])

# ------------------ 4) 커버리지/품질 리포트 ------------------
def cov(series):
    return int(series.notna().sum()), round(100*series.notna().mean(), 1)

covs = {}
for c in ["brand","product_type","target_audience","scent","volume_ml","price_num","review_signal"]:
    if c in item_table.columns:
        cnt, pct = cov(item_table[c])
        covs[c] = (cnt, pct)

report = []
report.append("=== CELL 8 REPORT — item_table 생성 ===")
report.append(f"- 총 상품수: {len(item_table):,}")
for k,(cnt,pct) in covs.items():
    report.append(f"- {k:<16} : {cnt:5d}개  ({pct:4.1f}%)")
# 가격/용량 bin 분포
for col, name in [("price_bin","[price_bin 분포]"), ("volume_bin","[volume_bin 분포]")]:
    if col in item_table.columns:
        dist = item_table[col].value_counts(dropna=False, normalize=True).sort_index()
        dist_str = " / ".join([f"{str(idx)}: {round(v*100,1)}%" for idx,v in dist.items()])
        report.append(name)
        report.append(f"- {dist_str}")

print("\n".join(report))

print("\n[item_table 미리보기 — 상위 8행]")
display(item_table.head(8))

# 최종 산출 유지
# item_table : parent_asin 단위 최종 테이블


=== CELL 8 REPORT — item_table 생성 ===
- 총 상품수: 12,402
- brand            : 11282개  (91.0%)
- product_type     :  3972개  (32.0%)
- target_audience  :  5430개  (43.8%)
- scent            :  1452개  (11.7%)
- volume_ml        :  3030개  (24.4%)
- price_num        :  2249개  (18.1%)
- review_signal    : 12402개  (100.0%)
[price_bin 분포]
- <=10: 5.0% / 10-20: 5.7% / 20-35: 4.1% / 35-60: 2.0% / 60-100: 0.9% / 100+: 0.4% / nan: 81.9%
[volume_bin 분포]
- <15ml: 3.4% / 15-30: 2.5% / 30-75: 4.3% / 75-150: 5.4% / 150-300: 5.0% / 300-600: 2.6% / 600+: 1.3% / nan: 75.6%

[item_table 미리보기 — 상위 8행]


/tmp/ipykernel_18119/1622973753.py:90: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_pick_best_attr)


,parent_asin,title,brand,product_type,target_audience,scent,volume_ml,price_num,review_signal,avg_rating,rating_cnt,verified_ratio,helpful_sum,first_review_date,last_review_date,days_since_last_review,price_bin,volume_bin
0,B076WQZGPM,"Yes to Tomatoes Detoxifying Charcoal Cleanser (Pack of 2) with Charcoal Powder, Tomato Fruit Extract, and Gingko Biloba Leaf Extract, 5 fl. oz.",yes to,cleanser,acne-prone,NaN,295.74,NaN,0.458619,5.000000,1,1.0,0,2018-05-19 23:00:21.775000+00:00,2018-05-19 23:00:21.775000+00:00,3108,NaN,150-300
1,B07G9GWFSM,Lurrose 100Pcs Full Cover Fake Toenails Artificial Transparent Nail Tips Nail Art for DIY,lurrose,NaN,NaN,NaN,NaN,6.99,0.373705,2.000000,1,0.0,0,2021-03-26 15:07:49.988000+00:00,2021-03-26 15:07:49.988000+00:00,2066,<=10,NaN
2,B08DNQTTQK,50 Pieces False Eyelash Packaging Box Empty Eyelash Box Plastic Eyelash Storage Case with Glitter Paper and Clear Tray for Women Girls Eyelash (Holographic),maitys,NaN,women,NaN,NaN,NaN,0.486226,4.333333,3,1.0,16,2020-09-24 21:23:07.518000+00:00,2021-06-18 04:45:54.076000+00:00,1982,NaN,NaN
3,B01ERJEGS6,Gold extatic Musk EDT 90ml,balmain,NaN,NaN,NaN,90.00,86.95,0.416799,3.500000,2,1.0,0,2017-11-05 01:11:04.808000+00:00,2019-09-04 19:22:17.774000+00:00,2635,60-100,75-150
4,B08P7LXKP7,4 Pieces Satin Bonnet Adjustable Sleep Cap Double Layer Printed Hair Bonnet Large Reversible Sleeping Silk Night Hair Cap for Women Natural Curly Hair (Flor...,geyoga,NaN,curly,floral,NaN,NaN,0.505269,4.250000,4,1.0,0,2021-04-10 13:21:51.261000+00:00,2021-08-17 18:10:40.542000+00:00,1922,NaN,NaN
5,B01555WAOS,Avon Big & False Lash Mascara - Brown/Black by Big & False Lash,avon,NaN,NaN,NaN,NaN,NaN,0.344065,2.000000,2,1.0,0,2016-01-09 23:38:09+00:00,2016-03-18 16:47:45+00:00,3900,NaN,NaN
6,B06XJZ7955,Brand New Headrang Face line Contour V-line Machine,NaN,NaN,NaN,NaN,NaN,79.50,0.393688,3.250000,4,1.0,0,2017-12-17 19:58:37.119000+00:00,2019-01-04 07:11:43.108000+00:00,2878,60-100,NaN
7,B07FVZVQKV,"Loose Powder, 1.5oz Bottle Loose Face Powder, Translucent Tone Loose Face Powder, for Setting Makeup or as Foundation, Lightweight, Long Lasting",getmore7,NaN,NaN,NaN,44.36,NaN,0.501700,4.250000,4,1.0,1,2018-12-19 20:39:46.615000+00:00,2020-02-12 15:29:21.739000+00:00,2474,NaN,30-75


In [20]:
# ==========================================================
# CELL 8 (safe) / item_table 생성 (문자열 가드 적용, 재실행 안전)
# 목적:
#  - 프론트/스코어링에 쓸 단일 상품 테이블 구성
#  - .str 에 의한 AttributeError 방지(모든 문자열 컬럼 가드)
# 산출: item_table
# ==========================================================
import numpy as np
import pandas as pd
from IPython.display import display

# ---------- 유틸: 문자열/수치 가드 ----------
def s_guard(s: pd.Series) -> pd.Series:
    """문자열 연산 전 안전 캐스팅(빈값은 "")"""
    if s is None:
        return pd.Series([], dtype="object")
    return s.astype(object).where(~s.isna(), "").astype(str)

def n_guard(s: pd.Series) -> pd.Series:
    """수치 연산 전 안전 캐스팅"""
    return pd.to_numeric(s, errors="coerce")

def ensure_bins(df: pd.DataFrame) -> pd.DataFrame:
    # price_bin
    if "price_num" in df.columns and "price_bin" not in df.columns:
        s = n_guard(df["price_num"])
        edges  = [0,10,20,35,60,100,np.inf]
        labels = ["<=10","10-20","20-35","35-60","60-100","100+"]
        df["price_bin"] = pd.cut(s, bins=edges, labels=labels, include_lowest=True, right=False)
    # volume_bin
    if "volume_ml" in df.columns and "volume_bin" not in df.columns:
        s = n_guard(df["volume_ml"])
        edges  = [0,15,30,75,150,300,600,np.inf]
        labels = ["<15ml","15-30","30-75","75-150","150-300","300-600","600+"]
        df["volume_bin"] = pd.cut(s, bins=edges, labels=labels, include_lowest=True, right=False)
    return df

# ---------- 1) 가장 최신 아티팩트가 있으면 그것으로 빌드 ----------
if "item_features_step3" in globals():
    base = item_features_step3.copy()
    # 문자열 가드
    for c in ["parent_asin","title","brand_view","product_type_std","product_type_std_filled",
              "target_audience_set","scent_set"]:
        if c in base.columns:
            base[c] = s_guard(base[c])
    # 수치 가드
    for c in ["price_num","volume_ml","review_signal","avg_rating","rating_cnt",
              "verified_ratio","helpful_sum","days_since_last_review",
              "price_norm_0_1","volume_norm_0_1","pop_log_norm"]:
        if c in base.columns:
            base[c] = n_guard(base[c])
    # bin 보장
    base = ensure_bins(base)

    # 최종 선택/리네임(존재하는 것만)
    cols = [
        "parent_asin","title","brand_view",
        "product_type_std_filled","target_audience_set","scent_set",
        "price_num","price_bin","price_norm_0_1",
        "volume_ml","volume_bin","volume_norm_0_1",
        "avg_rating","rating_cnt","verified_ratio","helpful_sum",
        "review_signal","pop_log_norm"
    ]
    keep = [c for c in cols if c in base.columns]
    item_table = base[keep].copy()

else:
    # ---------- 2) 이전 단계 산출물을 조합해 빌드 (fallback) ----------
    # 2.1 메타
    if "meta_df_text" in globals():
        meta = meta_df_text.copy()
    else:
        raise RuntimeError("item_features_step3 또는 meta_df_text 가 필요합니다.")
    # 필요한 기본 컬럼만
    meta_cols = [c for c in ["parent_asin","title","brand","price_num"] if c in meta.columns]
    meta = meta[meta_cols].drop_duplicates(subset=["parent_asin"]) if "parent_asin" in meta_cols else meta.copy()
    # 문자열 가드
    for c in ["parent_asin","title","brand"]:
        if c in meta.columns:
            meta[c] = s_guard(meta[c])
    if "price_num" in meta.columns:
        meta["price_num"] = n_guard(meta["price_num"])

    # 2.2 속성(롱→와이드)
    if "item_attr_long" not in globals():
        raise RuntimeError("item_attr_long 이 없습니다. 6.x 셀을 먼저 실행하세요.")
    al = item_attr_long.copy()
    # 가드
    for c in ["parent_asin","attr_id","value_text","unit","support_sources"]:
        if c in al.columns:
            al[c] = s_guard(al[c])
    for c in ["value_num","confidence","confidence_adj"]:
        if c in al.columns:
            al[c] = n_guard(al[c])

    # attr_id 별 대표값 선택(가중 기준: hard_pass, confidence_adj, value_num 유효성)
    if "hard_pass" not in al.columns:
        al["hard_pass"] = False
    al["conf_sort"] = n_guard(al.get("confidence_adj", al.get("confidence", pd.Series(index=al.index))))
    al["value_num_ok"] = al["value_num"].notna().astype(int)

    al_sorted = al.sort_values(
        by=["parent_asin","attr_id","hard_pass","conf_sort","value_num_ok"],
        ascending=[True, True, False, False, False]
    )
    al_pick = al_sorted.drop_duplicates(subset=["parent_asin","attr_id"], keep="first")

    # wide pivot
    wide_parts = {}
    for aid, colname in [("brand","brand_view"),
                         ("product_type","product_type_std_filled"),
                         ("target_audience","target_audience_set"),
                         ("scent","scent_set")]:
        sub = al_pick[al_pick["attr_id"]==aid]
        if colname in ["target_audience_set","scent_set"]:
            # 세트형: 동일 parent_asin에서 여러 값 있을 수 있으니 세미콜론 조인
            grp = sub.groupby("parent_asin")["value_text"].apply(lambda x: ";".join(sorted(set([y for y in x if y]))))
        else:
            # 단일 텍스트 우선, 없으면 value_num
            vt = sub.set_index("parent_asin")["value_text"]
            vn = sub.set_index("parent_asin")["value_num"]
            chosen = vt.where(vt.astype(bool), vn.astype(str))
            grp = chosen
        wide_parts[colname] = s_guard(grp)

    # (제출 후 수정) 위 목록에 volume_ml이 없어서, CELL 8에서 만든 volume_ml(재실행 기준 25.0%)이
    # 이 셀에서 item_table을 다시 만들 때 사라졌다. 숫자 속성이므로 value_num을 따로 붙인다.
    vol_part = n_guard(al_pick[al_pick["attr_id"]=="volume_ml"].set_index("parent_asin")["value_num"])

    # 2.3 리뷰 집계(있으면 조인)
    review_cols = ["parent_asin","avg_rating","rating_cnt","verified_ratio","helpful_sum","review_signal"]
    review = None
    if "review_agg" in globals():
        review = review_agg.copy()
    elif "review_table" in globals():
        review = review_table.copy()
    if review is not None:
        for c in review_cols:
            if c in review.columns and c != "parent_asin":
                review[c] = n_guard(review[c])
        review = review[[c for c in review_cols if c in review.columns]].drop_duplicates(subset=["parent_asin"])

    # 2.4 조인
    item_table = meta.set_index("parent_asin")
    for name, part in wide_parts.items():
        item_table[name] = part.reindex(item_table.index)
    item_table["volume_ml"] = vol_part.reindex(item_table.index)
    item_table = item_table.reset_index()

    if review is not None:
        item_table = item_table.merge(review, on="parent_asin", how="left")

    # 숫자 가드 & bin
    for c in ["price_num","volume_ml","avg_rating","rating_cnt","verified_ratio","helpful_sum","review_signal"]:
        if c in item_table.columns:
            item_table[c] = n_guard(item_table[c])
    item_table = ensure_bins(item_table)

# ---------- 리포트 ----------
def cov(series):
    return f"{int(series.notna().sum()):,}개 / {round(100*series.notna().mean(),1)}%"

print("=== CELL 8 REPORT — item_table 생성 (safe) ===")
print(f"- 총 상품수: {len(item_table):,}")
for c in ["brand_view","product_type_std_filled","target_audience_set","scent_set",
          "price_num","volume_ml","review_signal"]:
    if c in item_table.columns:
        print(f"- {c:18s}: {cov(item_table[c])}")

# 분포
if "price_bin" in item_table.columns:
    print("\n[price_bin 분포]")
    display(item_table["price_bin"].value_counts(dropna=False).rename_axis("bin").to_frame("count"))
if "volume_bin" in item_table.columns:
    print("\n[volume_bin 분포]")
    display(item_table["volume_bin"].value_counts(dropna=False).rename_axis("bin").to_frame("count"))

print("\n[item_table 미리보기 — 상위 8]")
view_cols = [c for c in [
    "parent_asin","title","brand_view","product_type_std_filled",
    "target_audience_set","scent_set",
    "price_num","price_bin","volume_ml","volume_bin",
    "avg_rating","rating_cnt","verified_ratio","helpful_sum",
    "review_signal","pop_log_norm"
] if c in item_table.columns]
display(item_table[view_cols].head(8))


=== CELL 8 REPORT — item_table 생성 (safe) ===
- 총 상품수: 12,402
- brand_view        : 11,282개 / 91.0%
- product_type_std_filled: 3,972개 / 32.0%
- target_audience_set: 5,430개 / 43.8%
- scent_set         : 1,452개 / 11.7%
- price_num         : 2,249개 / 18.1%
- volume_ml         : 3,030개 / 24.4%

[price_bin 분포]


,count
bin,
NaN,10153
10-20,702
<=10,626
20-35,504
35-60,251
60-100,111
100+,55



[volume_bin 분포]


,count
bin,
NaN,9372
75-150,668
150-300,614
30-75,536
<15ml,423
300-600,326
15-30,305
600+,158



[item_table 미리보기 — 상위 8]


,parent_asin,title,brand_view,product_type_std_filled,target_audience_set,scent_set,price_num,price_bin,volume_ml,volume_bin,avg_rating,rating_cnt,verified_ratio,helpful_sum
0,B076WQZGPM,"Yes to Tomatoes Detoxifying Charcoal Cleanser (Pack of 2) with Charcoal Powder, Tomato Fruit Extract, and Gingko Biloba Leaf Extract, 5 fl. oz.",yes to,cleanser,acne-prone,NaN,NaN,NaN,295.74,150-300,5.000000,1,1.0,0
1,B07G9GWFSM,Lurrose 100Pcs Full Cover Fake Toenails Artificial Transparent Nail Tips Nail Art for DIY,lurrose,NaN,NaN,NaN,6.99,<=10,NaN,NaN,2.000000,1,0.0,0
2,B08DNQTTQK,50 Pieces False Eyelash Packaging Box Empty Eyelash Box Plastic Eyelash Storage Case with Glitter Paper and Clear Tray for Women Girls Eyelash (Holographic),maitys,NaN,women,NaN,NaN,NaN,NaN,NaN,4.333333,3,1.0,16
3,B01ERJEGS6,Gold extatic Musk EDT 90ml,balmain,NaN,NaN,NaN,86.95,60-100,90.00,75-150,3.500000,2,1.0,0
4,B08P7LXKP7,4 Pieces Satin Bonnet Adjustable Sleep Cap Double Layer Printed Hair Bonnet Large Reversible Sleeping Silk Night Hair Cap for Women Natural Curly Hair (Flor...,geyoga,NaN,curly,floral,NaN,NaN,NaN,NaN,4.250000,4,1.0,0
5,B01555WAOS,Avon Big & False Lash Mascara - Brown/Black by Big & False Lash,avon,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.000000,2,1.0,0
6,B06XJZ7955,Brand New Headrang Face line Contour V-line Machine,NaN,NaN,NaN,NaN,79.50,60-100,NaN,NaN,3.250000,4,1.0,0
7,B07FVZVQKV,"Loose Powder, 1.5oz Bottle Loose Face Powder, Translucent Tone Loose Face Powder, for Setting Makeup or as Foundation, Lightweight, Long Lasting",getmore7,NaN,NaN,NaN,NaN,NaN,44.36,30-75,4.250000,4,1.0,1


In [21]:
# ==========================================================
# CELL 8.1c (safe) / 브랜드 정제 보강 v2 + 데모 서브셋 + 점수 보강
# - review_signal 없으면 자동 재계산(베이지안 평균 + 인기 + helpful)
# - .str 호출 전 문자열 가드 적용
# 산출: item_table_enh, item_table_demo (review_signal 보장)
# ==========================================================
import re, unicodedata
import numpy as np
import pandas as pd
from IPython.display import display

# ---------- 유틸 ----------
def s_guard(s: pd.Series) -> pd.Series:
    return s.astype(object).where(~s.isna(), "").astype(str)

def n_guard(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce")

def _strip_accents(text: str) -> str:
    text = unicodedata.normalize("NFKD", text)
    return "".join(ch for ch in text if not unicodedata.combining(ch))

def clean_brand_token(s) -> str:
    if s is None or (isinstance(s, float) and np.isnan(s)): return ""
    t = str(s).strip()
    if not t: return ""
    t = _strip_accents(t).lower()
    t = re.sub(r"&", " and ", t)
    t = re.sub(r"\b(co|co\.|inc|inc\.|ltd|ltd\.|llc|corp|corporation|company|"
               r"official|store|shop|beauty|cosmetics|skincare|care|usa|us|korea|japan|brand)\b", " ", t)
    t = t.replace("’","").replace("'","")
    t = re.sub(r"[^a-z0-9\s\+]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return {"l oreal":"loreal","kiehl s":"kiehls","3 m":"3m"}.get(t, t)

def normalize_brand(s) -> str:
    tok = clean_brand_token(s)
    if not tok: return ""
    parts = []
    for w in tok.split():
        parts.append("and" if w=="and" else w.title())
    return " ".join(parts)

def ensure_review_signal(df: pd.DataFrame) -> pd.DataFrame:
    """review_signal이 없거나 전부 결측이면 베이지안+인기+helpful로 재계산"""
    need = ("review_signal" not in df.columns) or (df["review_signal"].notna().sum()==0)
    if not need:
        return df
    df = df.copy()
    df["avg_rating"] = n_guard(df.get("avg_rating", pd.Series(index=df.index)))
    df["rating_cnt"] = n_guard(df.get("rating_cnt", pd.Series(index=df.index)))
    mu = float(np.nanmean(df["avg_rating"])) if np.isfinite(n_guard(df.get("avg_rating", pd.Series(dtype=float))).mean()) else 4.079
    alpha = 10.0
    bayes_mean = (df["avg_rating"]*df["rating_cnt"] + mu*alpha) / (df["rating_cnt"] + alpha)
    bayes_norm = ((bayes_mean - 1.0) / 4.0).clip(0,1)

    pop = np.log1p(df["rating_cnt"])
    if np.isfinite(pop.max()) and pop.max()>pop.min():
        pop_norm = (pop - pop.min()) / (pop.max() - pop.min())
    else:
        pop_norm = pop*0

    helpful = n_guard(df.get("helpful_sum", pd.Series(index=df.index))).fillna(0)
    hmax = helpful.max()
    helpful_norm = helpful / (hmax if (hmax and np.isfinite(hmax)) else 1.0)

    df["review_signal"] = (0.6*bayes_norm + 0.3*pop_norm + 0.1*helpful_norm).fillna(0)
    return df

def safe_show(table, cols, n=5, title=None, sort_by=None, ascending=False):
    t = table.copy()
    if sort_by and sort_by in t.columns:
        t = t.sort_values(sort_by, ascending=ascending)
    if title: print(f"\n[{title}]")
    keep = [c for c in cols if c in t.columns]
    display(t[keep].head(n))

# ---------- 입력 확인 ----------
if "item_table" not in globals():
    raise RuntimeError("item_table 가 없습니다. CELL 8(safe) 실행 후 진행하세요.")

df = item_table.copy()

# ---------- 문자열/수치 가드 ----------
for c in ["parent_asin","title","brand_view","brand","product_type_std_filled",
          "target_audience_set","scent_set"]:
    if c not in df.columns: df[c] = np.nan
    df[c] = s_guard(df[c])

for c in ["price_num","volume_ml","avg_rating","rating_cnt","verified_ratio",
          "helpful_sum","review_signal"]:
    if c in df.columns:
        df[c] = n_guard(df[c])

# ---------- 브랜드 정제 v2 ----------
# brand_view 우선, 없으면 brand로 보충
df["brand_view"] = np.where(df["brand_view"].str.strip()=="", df["brand"], df["brand_view"])
df["brand_clean_v2"] = df["brand_view"].map(normalize_brand).replace("", np.nan)
df["brand_view"] = df["brand_clean_v2"].fillna(df["brand_view"]).replace("", "Unknown")

# ---------- product_type 보강 ----------
if df["product_type_std_filled"].eq("").all():
    # 없는 경우 product_type로 대체 or other
    if "product_type" in df.columns:
        df["product_type_std_filled"] = s_guard(df["product_type"]).replace("", "other")
    else:
        df["product_type_std_filled"] = "other"

# ---------- review_signal 보장 ----------
df = ensure_review_signal(df)

# ---------- 데모 서브셋 ----------
CORE_PT = {
    "cleanser","serum","essence","moisturizer","mask","sunscreen",
    "shampoo","conditioner","treatment","oil","powder","makeup remover",
    "soap","toner","cream","gel","balm"
}
df["pt"] = df["product_type_std_filled"].str.lower()
item_table_demo = df[(df["pt"].isin(CORE_PT)) & (df["brand_view"].str.strip()!="")].copy()

# ---------- 산출 테이블 ----------
cols_enh = [c for c in [
    "parent_asin","title","brand","brand_view","brand_clean_v2",
    "product_type_std_filled","target_audience_set","scent_set",
    "volume_ml","price_num","avg_rating","rating_cnt","helpful_sum","review_signal"
] if c in df.columns]
item_table_enh = df[cols_enh].copy()

# ---------- 리포트 ----------
print("=== CELL 8.1c REPORT — brand 정제 보강 & 데모 v3 (safe) ===")
cov_brand = df["brand_clean_v2"].notna().mean()*100
print(f"- brand_clean_v2 커버리지: {df['brand_clean_v2'].notna().sum():,} / {len(df):,} ({cov_brand:.1f}%)")
print(f"- demo_ready_v3 상품수: {len(item_table_demo):,} / {len(df):,} ({100*len(item_table_demo)/len(df):.1f}%)")

print("\n[item_table_enh 미리보기 — 상위 5]")
safe_show(
    item_table_enh,
    ["parent_asin","title","brand","brand_view","brand_clean_v2",
     "product_type_std_filled","target_audience_set","scent_set",
     "volume_ml","price_num","avg_rating","rating_cnt","helpful_sum","review_signal"],
    n=5, title="item_table_enh 미리보기"
)

print("\n[item_table_demo v3 — 상위 5 (by review_signal)]")
safe_show(
    item_table_demo,
    ["parent_asin","brand_view","brand_clean_v2","product_type_std_filled",
     "volume_ml","price_num","review_signal","title"],
    n=5, title="item_table_demo v3 — 상위 5 (review_signal)",
    sort_by="review_signal", ascending=False
)


=== CELL 8.1c REPORT — brand 정제 보강 & 데모 v3 (safe) ===
- brand_clean_v2 커버리지: 11,277 / 12,402 (90.9%)
- demo_ready_v3 상품수: 3,603 / 12,402 (29.1%)

[item_table_enh 미리보기 — 상위 5]

[item_table_enh 미리보기]


,parent_asin,title,brand,brand_view,brand_clean_v2,product_type_std_filled,target_audience_set,scent_set,volume_ml,price_num,avg_rating,rating_cnt,helpful_sum,review_signal
0,B076WQZGPM,"Yes to Tomatoes Detoxifying Charcoal Cleanser (Pack of 2) with Charcoal Powder, Tomato Fruit Extract, and Gingko Biloba Leaf Extract, 5 fl. oz.",Yes To,Yes To,Yes To,cleanser,acne-prone,,295.74,NaN,5.000000,1,0,0.441439
1,B07G9GWFSM,Lurrose 100Pcs Full Cover Fake Toenails Artificial Transparent Nail Tips Nail Art for DIY,Lurrose,Lurrose,Lurrose,,,,NaN,6.99,2.000000,1,0,0.400530
2,B08DNQTTQK,50 Pieces False Eyelash Packaging Box Empty Eyelash Box Plastic Eyelash Storage Case with Glitter Paper and Clear Tray for Women Girls Eyelash (Holographic),Maitys,Maitys,Maitys,,women,,NaN,NaN,4.333333,3,16,0.475244
3,B01ERJEGS6,Gold extatic Musk EDT 90ml,Balmain,Balmain,Balmain,,,,90.00,86.95,3.500000,2,0,0.435771
4,B08P7LXKP7,4 Pieces Satin Bonnet Adjustable Sleep Cap Double Layer Printed Hair Bonnet Large Reversible Sleeping Silk Night Hair Cap for Women Natural Curly Hair (Flor...,Geyoga,Geyoga,Geyoga,,curly,floral,NaN,NaN,4.250000,4,0,0.485350



[item_table_demo v3 — 상위 5 (by review_signal)]

[item_table_demo v3 — 상위 5 (review_signal)]


,parent_asin,brand_view,brand_clean_v2,product_type_std_filled,volume_ml,price_num,review_signal,title
9105,B07R1K1SBD,Gk Hair,Gk Hair,shampoo,300.0,NaN,0.837395,GK HAIR Global Keratin Moisturizing Shampoo and Conditioner Sets (10.1 Fl Oz/300ml) for Color Treated Hair - Daily Use Cleansing Dry to Normal Sulfate Parab...
4221,B0888TZ9W6,Venture Wipes,Venture Wipes,oil,NaN,14.95,0.803995,"Large Body Wipes for Adults Bathing. Biodegradable with Aloe, Vitamin E and Tea Tree Oil. After Workout Shower Wipes for Men and Women. Backpacking Essentia..."
4259,B08R5DJGZQ,Warartisan,Warartisan,mask,NaN,NaN,0.781271,Foot Peel Mask 3 Pack Exfoliating Foot Mask for Callus and Dry Dead Skin Repair Rough Heels Get Baby Soft Smooth Touch Feet for Man Woman ( Lavender Rose Al...
6984,B09F3LF24J,Diaregno,Diaregno,gel,NaN,12.99,0.774076,"DIAREGNO Teeth Whitening Pen (2 Pens) - 20+ Uses, Effective ＆ Painless, No Sensitivity - Beautiful White Smile - Natural Mint Flavor"
9916,B08CKCV9HD,Qimeisi,Qimeisi,gel,15.0,NaN,0.768001,"Poly Nail Gel Kit Nail Extension Gel Builder, Poly Nail Gel Enhancement System For Nail Art, 6 Crystal Basic Colors Poly Gel 15mL/Each"


In [22]:
# ==========================================================
# CELL 9.5 (safe) / 수치 피처 정리 + 윈저라이징/정규화
# 목적:
#   - 프런트/랭킹에서 사용할 수 있는 기본 numeric feature들을 정리
#   - price/volume/popularity(rating_cnt) 에 대해 winsor + 0~1 정규화
# 산출:
#   - item_features_step2 DataFrame
#   - 간단 품질 리포트(커버리지/분포)
# 전제:
#   - item_table_enh 또는 item_table 이 존재
# ==========================================================
import numpy as np
import pandas as pd
from IPython.display import display

# ---------- 유틸 ----------
def n_guard(s: pd.Series) -> pd.Series:
    """수치형 안전 캐스팅"""
    return pd.to_numeric(s, errors="coerce")

def normalize_numeric_col(df: pd.DataFrame,
                          col: str,
                          winsor_col: str,
                          norm_col: str,
                          q_low: float = 0.01,
                          q_high: float = 0.99):
    """
    단일 수치 컬럼에 대해:
      - NaN 포함한 상태에서 유효값만 q_low~q_high 분위수 계산
      - [lo, hi] 구간으로 clip한 winsor_col 생성
      - [lo, hi] 기준 0~1 정규화한 norm_col 생성
    """
    x = n_guard(df.get(col, pd.Series(index=df.index)))
    df[col] = x

    valid = x.dropna()
    if valid.empty:
        df[winsor_col] = np.nan
        df[norm_col] = np.nan
        return {
            "col": col,
            "cov": 0.0,
            "q01": np.nan,
            "q99": np.nan,
            "min_w": np.nan,
            "max_w": np.nan,
            "p50_norm": np.nan,
            "p90_norm": np.nan,
            "max_norm": np.nan,
        }

    lo = float(valid.quantile(q_low))
    hi = float(valid.quantile(q_high))
    clipped = x.clip(lo, hi)

    if hi > lo:
        norm = (clipped - lo) / (hi - lo)
    else:
        norm = clipped * 0.0

    df[winsor_col] = clipped
    df[norm_col] = norm

    return {
        "col": col,
        "cov": float(x.notna().mean()),
        "q01": lo,
        "q99": hi,
        "min_w": float(clipped.min()),
        "max_w": float(clipped.max()),
        "p50_norm": float(norm.quantile(0.5)),
        "p90_norm": float(norm.quantile(0.9)),
        "max_norm": float(norm.max()),
    }

# ---------- 1) 소스 선택 ----------
if "item_table_enh" in globals():
    src = item_table_enh.copy()
    src_name = "item_table_enh"
elif "item_table" in globals():
    src = item_table.copy()
    src_name = "item_table"
else:
    raise RuntimeError("item_table_enh / item_table 이 없습니다. CELL 8 / 8.1c 이후를 먼저 실행하세요.")

print(f"[INFO] 9.5 source: {src_name} (rows={len(src):,})")

# id + 텍스트 정보는 그대로 유지
base_cols = [c for c in [
    "parent_asin","title","brand_view","brand_clean_v2",
    "product_type_std_filled","target_audience_set","scent_set"
] if c in src.columns]

df = src[base_cols].copy()

# 수치 컬럼 확보 (없으면 NaN 컬럼 생성)
for col in ["price_num","volume_ml","rating_cnt","review_signal"]:
    if col not in src.columns:
        df[col] = np.nan
    else:
        df[col] = n_guard(src[col])

# ---------- 2) rating_cnt 로그 피처 ----------
df["rating_cnt_log"] = np.log1p(df["rating_cnt"])

# ---------- 3) 각 컬럼 정규화 ----------
stats = []

# price
stats.append(
    normalize_numeric_col(df,
                          col="price_num",
                          winsor_col="price_winsor",
                          norm_col="price_norm_0_1")
)

# volume (없으면 전부 NaN이지만 에러는 안 남)
stats.append(
    normalize_numeric_col(df,
                          col="volume_ml",
                          winsor_col="volume_winsor",
                          norm_col="volume_norm_0_1")
)

# popularity (rating_cnt_log)
stats.append(
    normalize_numeric_col(df,
                          col="rating_cnt_log",
                          winsor_col="pop_log_winsor",
                          norm_col="pop_log_norm")
)

stats_df = pd.DataFrame(stats)

# ---------- 4) 산출 테이블 정리 ----------
keep_cols = [c for c in [
    "parent_asin","title","brand_view","brand_clean_v2",
    "product_type_std_filled","target_audience_set","scent_set",
    "price_num","price_winsor","price_norm_0_1",
    "volume_ml","volume_winsor","volume_norm_0_1",
    "rating_cnt","rating_cnt_log","pop_log_winsor","pop_log_norm",
    "review_signal"
] if c in df.columns]

item_features_step2 = df[keep_cols].copy()

# ---------- 5) 리포트 ----------
print("=== CELL 9.5 REPORT — 수치 정규화/스케일링 (safe) ===")
for row in stats:
    col = row["col"]
    print(f"- {col}: raw_cov={row['cov']*100:.1f}% "
          f"/ q01~q99={row['q01']:.4g}~{row['q99']:.4g} "
          f"/ norm[p50,p90,max]={row['p50_norm']:.4g},{row['p90_norm']:.4g},{row['max_norm']:.4g}")

print("\n[item_features_step2 미리보기 — 상위 8]")
display(item_features_step2.head(8))


[INFO] 9.5 source: item_table_enh (rows=12,402)
=== CELL 9.5 REPORT — 수치 정규화/스케일링 (safe) ===
- price_num: raw_cov=18.1% / q01~q99=3.257~180 / norm[p50,p90,max]=0.07771,0.2645,1
- volume_ml: raw_cov=24.4% / q01~q99=1.77~1479 / norm[p50,p90,max]=0.06688,0.3192,1
- rating_cnt_log: raw_cov=100.0% / q01~q99=0.6931~4.605 / norm[p50,p90,max]=0.1036,0.4974,1

[item_features_step2 미리보기 — 상위 8]


,parent_asin,title,brand_view,brand_clean_v2,product_type_std_filled,target_audience_set,scent_set,price_num,price_winsor,price_norm_0_1,volume_ml,volume_winsor,volume_norm_0_1,rating_cnt,rating_cnt_log,pop_log_winsor,pop_log_norm,review_signal
0,B076WQZGPM,"Yes to Tomatoes Detoxifying Charcoal Cleanser (Pack of 2) with Charcoal Powder, Tomato Fruit Extract, and Gingko Biloba Leaf Extract, 5 fl. oz.",Yes To,Yes To,cleanser,acne-prone,,NaN,NaN,NaN,295.74,295.74,0.199045,1,0.693147,0.693147,0.000000,0.441439
1,B07G9GWFSM,Lurrose 100Pcs Full Cover Fake Toenails Artificial Transparent Nail Tips Nail Art for DIY,Lurrose,Lurrose,,,,6.99,6.99,0.021123,NaN,NaN,NaN,1,0.693147,0.693147,0.000000,0.400530
2,B08DNQTTQK,50 Pieces False Eyelash Packaging Box Empty Eyelash Box Plastic Eyelash Storage Case with Glitter Paper and Clear Tray for Women Girls Eyelash (Holographic),Maitys,Maitys,,women,,NaN,NaN,NaN,NaN,NaN,NaN,3,1.386294,1.386294,0.177188,0.475244
3,B01ERJEGS6,Gold extatic Musk EDT 90ml,Balmain,Balmain,,,,86.95,86.95,0.473607,90.00,90.00,0.059740,2,1.098612,1.098612,0.103649,0.435771
4,B08P7LXKP7,4 Pieces Satin Bonnet Adjustable Sleep Cap Double Layer Printed Hair Bonnet Large Reversible Sleeping Silk Night Hair Cap for Women Natural Curly Hair (Flor...,Geyoga,Geyoga,,curly,floral,NaN,NaN,NaN,NaN,NaN,NaN,4,1.609438,1.609438,0.234230,0.485350
5,B01555WAOS,Avon Big & False Lash Mascara - Brown/Black by Big & False Lash,Avon,Avon,,,,NaN,NaN,NaN,NaN,NaN,NaN,2,1.098612,1.098612,0.103649,0.398271
6,B06XJZ7955,Brand New Headrang Face line Contour V-line Machine,Unknown,NaN,,,,79.50,79.50,0.431448,NaN,NaN,NaN,4,1.609438,1.609438,0.234230,0.442492
7,B07FVZVQKV,"Loose Powder, 1.5oz Bottle Loose Face Powder, Translucent Tone Loose Face Powder, for Setting Makeup or as Foundation, Lightweight, Long Lasting",Getmore7,Getmore7,,,,NaN,NaN,NaN,44.36,44.36,0.028837,4,1.609438,1.609438,0.234230,0.485391


In [23]:
# ==========================================================
# CELL 9.6 (safe) / 수치 정규화 품질 리포트
# 목적:
#   - CELL 9.5(safe)에서 만든 item_features_step2의 스케일링 상태 점검
#   - price / volume / rating_cnt_log(popularity) 분포를 요약해서 확인
# 전제:
#   - CELL 9.5(safe) 실행 완료 → item_features_step2 존재
# 산출:
#   - 콘솔 리포트 + head(8) 프리뷰
# ==========================================================
import numpy as np
import pandas as pd
from IPython.display import display

if "item_features_step2" not in globals():
    raise RuntimeError("item_features_step2 가 없습니다. CELL 9.5(safe)를 먼저 실행하세요.")

df = item_features_step2.copy()

def _summary(col_raw: pd.Series,
             col_norm: pd.Series | None,
             name: str) -> str:
    """단일 피처의 커버리지/분포/정규화 상태 요약"""
    s = pd.to_numeric(col_raw, errors="coerce")
    cov = float(s.notna().mean())

    valid = s.dropna()
    if valid.empty:
        return f"- {name}: raw_cov={cov*100:.1f}% (유효값 없음)"

    q01, q50, q90, q99 = valid.quantile([0.01, 0.5, 0.9, 0.99])

    norm_info = None
    if col_norm is not None:
        n = pd.to_numeric(col_norm, errors="coerce")
        n_valid = n.dropna()
        if not n_valid.empty:
            norm_info = (
                float(n_valid.quantile(0.5)),
                float(n_valid.quantile(0.9)),
                float(n_valid.max()),
            )

    if norm_info is not None:
        p50_n, p90_n, max_n = norm_info
        return (
            f"- {name}: raw_cov={cov*100:.1f}% "
            f"/ q01~q99={q01:.4g}~{q99:.4g} "
            f"/ norm[p50,p90,max]={p50_n:.4g},{p90_n:.4g},{max_n:.4g}"
        )
    else:
        return (
            f"- {name}: raw_cov={cov*100:.1f}% "
            f"/ q01~q99={q01:.4g}~{q99:.4g} (norm 없음)"
        )

lines: list[str] = []

if "price_num" in df.columns:
    lines.append(
        _summary(
            df["price_num"],
            df.get("price_norm_0_1"),
            "price_num",
        )
    )

if "volume_ml" in df.columns:
    lines.append(
        _summary(
            df["volume_ml"],
            df.get("volume_norm_0_1"),
            "volume_ml",
        )
    )

if "rating_cnt_log" in df.columns:
    lines.append(
        _summary(
            df["rating_cnt_log"],
            df.get("pop_log_norm"),
            "rating_cnt_log(pop)",
        )
    )

print("=== CELL 9.6 REPORT — 수치 정규화 품질 요약 ===")
print("\n".join(lines))

print("\n[item_features_step2 미리보기 — 상위 8]")
preview_cols = [c for c in [
    "parent_asin","title","brand_view","product_type_std_filled",
    "price_num","price_norm_0_1",
    "volume_ml","volume_norm_0_1",
    "rating_cnt","rating_cnt_log","pop_log_norm",
    "review_signal",
] if c in df.columns]

display(df[preview_cols].head(8))


=== CELL 9.6 REPORT — 수치 정규화 품질 요약 ===
- price_num: raw_cov=18.1% / q01~q99=3.257~180 / norm[p50,p90,max]=0.07771,0.2645,1
- volume_ml: raw_cov=24.4% / q01~q99=1.77~1479 / norm[p50,p90,max]=0.06688,0.3192,1
- rating_cnt_log(pop): raw_cov=100.0% / q01~q99=0.6931~4.605 / norm[p50,p90,max]=0.1036,0.4974,1

[item_features_step2 미리보기 — 상위 8]


,parent_asin,title,brand_view,product_type_std_filled,price_num,price_norm_0_1,volume_ml,volume_norm_0_1,rating_cnt,rating_cnt_log,pop_log_norm,review_signal
0,B076WQZGPM,"Yes to Tomatoes Detoxifying Charcoal Cleanser (Pack of 2) with Charcoal Powder, Tomato Fruit Extract, and Gingko Biloba Leaf Extract, 5 fl. oz.",Yes To,cleanser,NaN,NaN,295.74,0.199045,1,0.693147,0.000000,0.441439
1,B07G9GWFSM,Lurrose 100Pcs Full Cover Fake Toenails Artificial Transparent Nail Tips Nail Art for DIY,Lurrose,,6.99,0.021123,NaN,NaN,1,0.693147,0.000000,0.400530
2,B08DNQTTQK,50 Pieces False Eyelash Packaging Box Empty Eyelash Box Plastic Eyelash Storage Case with Glitter Paper and Clear Tray for Women Girls Eyelash (Holographic),Maitys,,NaN,NaN,NaN,NaN,3,1.386294,0.177188,0.475244
3,B01ERJEGS6,Gold extatic Musk EDT 90ml,Balmain,,86.95,0.473607,90.00,0.059740,2,1.098612,0.103649,0.435771
4,B08P7LXKP7,4 Pieces Satin Bonnet Adjustable Sleep Cap Double Layer Printed Hair Bonnet Large Reversible Sleeping Silk Night Hair Cap for Women Natural Curly Hair (Flor...,Geyoga,,NaN,NaN,NaN,NaN,4,1.609438,0.234230,0.485350
5,B01555WAOS,Avon Big & False Lash Mascara - Brown/Black by Big & False Lash,Avon,,NaN,NaN,NaN,NaN,2,1.098612,0.103649,0.398271
6,B06XJZ7955,Brand New Headrang Face line Contour V-line Machine,Unknown,,79.50,0.431448,NaN,NaN,4,1.609438,0.234230,0.442492
7,B07FVZVQKV,"Loose Powder, 1.5oz Bottle Loose Face Powder, Translucent Tone Loose Face Powder, for Setting Makeup or as Foundation, Lightweight, Long Lasting",Getmore7,,NaN,NaN,44.36,0.028837,4,1.609438,0.234230,0.485391


In [24]:
# ==========================================================
# CELL 9.7 / 품질 게이트·결측 보완 + 유효성 점검 리포트  [완전 실행형]
# 목적:
#  - 최소 사용 가능(minimal OK) 조건 플래그 생성: title 존재, parent_asin 존재, brand_view 대체
#  - product_type_std 결측시 'other'로 채움(스코어링은 conf=0.0 그대로 유지)
#  - brand_view 결측은 'Unknown'으로 대체(표시용), 원본은 별도 보존
#  - parent_asin 기준 중복 정리(우선순위: review_signal desc → rating_cnt desc → price_num asc)
#  - 기본 bin(price_bin/volume_bin) 보장
# 산출: item_features_step3
# ==========================================================
import numpy as np
import pandas as pd
from IPython.display import display

if "item_features_step2" not in globals():
    raise RuntimeError("item_features_step2 가 없습니다. CELL 9.6을 먼저 실행하세요.")

df = item_features_step2.copy()

# ---------- 0) 기본 정리 ----------
# 안전 캐스팅
for col in ["parent_asin","title","brand_view","product_type_std","target_audience_set","scent_set"]:
    if col not in df.columns:
        df[col] = np.nan
    df[col] = df[col].astype(object)

# 숫자 보정
for col in ["price_num","volume_ml","review_signal","avg_rating","rating_cnt","helpful_sum","days_since_last_review",
            "price_norm_0_1","volume_norm_0_1","pop_log_norm","rating_cnt_log","helpful_log"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# ---------- 1) 최소 사용 가능 조건 플래그 ----------
df["brand_view_orig"] = df["brand_view"]
df["brand_view"] = df["brand_view"].fillna("Unknown").astype(str).str.strip()
df.loc[df["brand_view"]=="", "brand_view"] = "Unknown"

df["has_title"]       = df["title"].notna() & (df["title"].astype(str).str.strip()!="")
df["has_parent_asin"] = df["parent_asin"].notna() & (df["parent_asin"].astype(str).str.strip()!="")
df["minimal_ok"]      = df["has_title"] & df["has_parent_asin"]

# ---------- 2) product_type_std 결측 대체 ----------
df["product_type_std_filled"] = df["product_type_std"].where(df["product_type_std"].astype(str).str.strip()!="", "other")
df["filled_pt_other_flag"]    = (df["product_type_std"].astype(str).str.strip()=="")  # True면 other로 채운 행

# ---------- 3) price_bin / volume_bin 보장 ----------
if "price_bin" not in df.columns:
    s = df["price_num"]
    edges  = [0,10,20,35,60,100,np.inf]
    labels = ["<=10","10-20","20-35","35-60","60-100","100+"]
    df["price_bin"] = pd.cut(s, bins=edges, labels=labels, include_lowest=True, right=False)

if "volume_bin" not in df.columns:
    s = df["volume_ml"]
    edges  = [0,15,30,75,150,300,600,np.inf]
    labels = ["<15ml","15-30","30-75","75-150","150-300","300-600","600+"]
    df["volume_bin"] = pd.cut(s, bins=edges, labels=labels, include_lowest=True, right=False)

# ---------- 4) parent_asin 중복 정리 ----------
before_n = len(df)
dup_cnt  = int(df["parent_asin"].duplicated(keep=False).sum()) if "parent_asin" in df.columns else 0

sort_cols = []
if "review_signal" in df.columns: sort_cols.append(("review_signal", False))
if "rating_cnt"   in df.columns: sort_cols.append(("rating_cnt", False))
if "price_num"    in df.columns: sort_cols.append(("price_num", True))

if sort_cols:
    df = df.sort_values([c for c,_ in sort_cols], ascending=[a for _,a in sort_cols])

# drop_duplicates는 첫 번째(정렬 후) 것을 남기므로 위 정렬 우선순위가 유효
df = df.drop_duplicates(subset=["parent_asin"], keep="first")
after_n  = len(df)
removed  = before_n - after_n

# ---------- 5) 출력 테이블 확정 ----------
item_features_step3 = df.copy()

# ---------- 6) 리포트 ----------
def cov(series):
    return f"{int(series.notna().sum()):,}개 / {round(100*series.notna().mean(),1)}%"

print("=== CELL 9.7 REPORT — 품질 게이트·결측 보완 ===")
print(f"- 총 행수(정리 전 → 후): {before_n:,} → {after_n:,} (중복 제거 {removed:,}행, 중복표시 {dup_cnt:,}행)")
print(f"- minimal_ok(제목+ASIN) 충족: {int(item_features_step3['minimal_ok'].sum()):,}개 / {round(100*item_features_step3['minimal_ok'].mean(),1)}%")
print(f"- brand_view 결측 대체(Unknown) 건수: {(item_features_step3['brand_view']=='Unknown').sum():,}개")
print(f"- product_type_std 결측 대체(other) 건수: {int(item_features_step3['filled_pt_other_flag'].sum()):,}개")
if "review_signal" in item_features_step3.columns:
    print(f"- review_signal 커버리지: {cov(item_features_step3['review_signal'])}")
if "price_num" in item_features_step3.columns:
    print(f"- price_num 커버리지: {cov(item_features_step3['price_num'])}")
if "volume_ml" in item_features_step3.columns:
    print(f"- volume_ml 커버리지: {cov(item_features_step3['volume_ml'])}")

# 분포 요약
print("\n[product_type_std_filled 분포 상위 15]")
display(item_features_step3["product_type_std_filled"].value_counts().head(15))

print("\n[price_bin 분포]")
if "price_bin" in item_features_step3.columns:
    display(item_features_step3["price_bin"].value_counts(dropna=False).rename_axis("bin").to_frame("count"))

print("\n[volume_bin 분포]")
if "volume_bin" in item_features_step3.columns:
    display(item_features_step3["volume_bin"].value_counts(dropna=False).rename_axis("bin").to_frame("count"))

print("\n[item_features_step3 미리보기 — 상위 8]")
view_cols = [
    "parent_asin","title","brand_view","product_type_std","product_type_std_filled",
    "target_audience_set","scent_set","price_num","price_norm_0_1","price_bin",
    "volume_ml","volume_norm_0_1","volume_bin",
    "review_signal","rating_cnt","pop_log_norm",
    "minimal_ok","filled_pt_other_flag"
]
keep = [c for c in view_cols if c in item_features_step3.columns]
display(item_features_step3[keep].head(8))


=== CELL 9.7 REPORT — 품질 게이트·결측 보완 ===
- 총 행수(정리 전 → 후): 12,402 → 12,402 (중복 제거 0행, 중복표시 0행)
- minimal_ok(제목+ASIN) 충족: 12,400개 / 100.0%
- brand_view 결측 대체(Unknown) 건수: 1,120개
- product_type_std 결측 대체(other) 건수: 0개
- review_signal 커버리지: 12,402개 / 100.0%
- price_num 커버리지: 2,249개 / 18.1%
- volume_ml 커버리지: 3,030개 / 24.4%

[product_type_std_filled 분포 상위 15]


,count
product_type_std_filled,



[price_bin 분포]


,count
bin,
NaN,10153
10-20,702
<=10,626
20-35,504
35-60,251
60-100,111
100+,55



[volume_bin 분포]


,count
bin,
NaN,9372
75-150,668
150-300,614
30-75,536
<15ml,423
300-600,326
15-30,305
600+,158



[item_features_step3 미리보기 — 상위 8]


,parent_asin,title,brand_view,product_type_std,product_type_std_filled,target_audience_set,scent_set,price_num,price_norm_0_1,price_bin,volume_ml,volume_norm_0_1,volume_bin,review_signal,rating_cnt,pop_log_norm,minimal_ok,filled_pt_other_flag
3801,B09X9BG4FC,Makone Crystal Crowns and Tiaras with Comb Headband for Girl or Women Birthday Party Wedding Prom Bridal Christmas Valentine… (03 Pink),Makone,NaN,NaN,women,,9.99,0.038100,<=10,NaN,NaN,NaN,0.847937,1374,1.0,True,False
9105,B07R1K1SBD,GK HAIR Global Keratin Moisturizing Shampoo and Conditioner Sets (10.1 Fl Oz/300ml) for Color Treated Hair - Daily Use Cleansing Dry to Normal Sulfate Parab...,Gk Hair,NaN,NaN,dry,,NaN,NaN,NaN,300.0,0.20193,300-600,0.837395,347,1.0,True,False
8516,B09FFQT1KK,Sun Labs Self Tanning Mitt Applicator - 1 Sunless Tanning Mitt for Sun Tanning Lotion,Sun Laboratories By Giesee,NaN,NaN,unisex,,6.00,0.015521,<=10,NaN,NaN,NaN,0.828702,522,1.0,True,False
2067,B09C2ZW2GL,Tigi Bedhead Hard Head Hairspray (6 Pack),Tigi,NaN,NaN,,citrus,NaN,NaN,NaN,NaN,NaN,NaN,0.817866,959,1.0,True,False
9792,B09K9JHDNX,"Happy Birthday Box for Women - Unique Birthday Gifts for Women, Best Gift Baskets for Women Birthday for Best Friend, Birthday Gifts for Sister, Birthday Gi...",Sodilly,NaN,NaN,women,,39.99,0.207866,35-60,NaN,NaN,NaN,0.814903,115,1.0,True,False
6578,B08F4Q8ZBR,YMHPRIDE 4 Pieces Stretch Jersey Turban Head Wrap Knit Headwraps Urban Hair Scarf Solid Color Extra Long Hair Scarf Urban African Head Wrap Head Band Ultra ...,Ymhpride,NaN,NaN,women,,25.99,0.128642,20-35,NaN,NaN,NaN,0.810079,561,1.0,True,False
4221,B0888TZ9W6,"Large Body Wipes for Adults Bathing. Biodegradable with Aloe, Vitamin E and Tea Tree Oil. After Workout Shower Wipes for Men and Women. Backpacking Essentia...",Venture Wipes,NaN,NaN,normal,tea tree,14.95,0.066168,10-20,NaN,NaN,NaN,0.803995,248,1.0,True,False
4970,B09LHS6RGF,Hairginkgo Mannequin Head 100% Real Hair Manikin Head Styling Hairdresser Training Head Cosmetology Doll Head for Dyeing Cutting Braiding Practice with Clam...,Hairginkgo,NaN,NaN,kids,,27.99,0.139960,20-35,NaN,NaN,NaN,0.792468,881,1.0,True,False


In [25]:
# ==========================================================
# CELL 10.0 / 속성 온톨로지(attr_ontology) 생성  [완전 실행형]
# 목적:
#  - 프런트에서 속성 매칭/동의어 처리를 위해 facet별 표준 라벨과 alias(동의어) 사전을 CSV로 제공
# 입력: item_features_step3 (CELL 9.7 산출)
# 산출: attr_ontology.csv  (컬럼: facet,value,canonical,aliases,group,default_weight,pattern)
# ==========================================================
import os
import re
import numpy as np
import pandas as pd
from IPython.display import display

if "item_features_step3" not in globals():
    raise RuntimeError("item_features_step3 가 없습니다. CELL 9.7을 먼저 실행하세요.")

# ---------- 1) 온톨로지 정의(작고 견고하게) ----------
# facet: product_type_std / target_audience / scent
# value=canonical 기본값(표준 라벨), aliases=동의어 파이프(|) 구분, group=상위 그룹, default_weight=속성 중요도(0~1)

PT_ROWS = [
    # skincare
    ("product_type_std","cleanser", "cleanser",
     "cleanser|face wash|facial wash|foaming wash|foam wash|gel cleanser|cream cleanser|micellar gel",
     "skincare", 1.0),
    ("product_type_std","toner","toner",
     "toner|skin softener|skin conditioner|lotion toner",
     "skincare", 0.9),
    ("product_type_std","serum","serum",
     "serum|ampoule|ampule|booster serum|essence serum",
     "skincare", 1.0),
    ("product_type_std","essence","essence",
     "essence|treatment essence|first essence",
     "skincare", 0.9),
    ("product_type_std","moisturizer","moisturizer",
     "moisturizer|moisturiser|lotion|face cream|gel cream|balm|emulsion",
     "skincare", 0.9),
    ("product_type_std","mask","mask",
     "mask|sheet mask|sleeping pack|sleeping mask|mud mask|clay mask",
     "skincare", 0.8),
    ("product_type_std","sunscreen","sunscreen",
     "sunscreen|sun screen|sunblock|spf|uv defense|uva|uvb",
     "skincare", 1.0),

    # hair/body
    ("product_type_std","shampoo","shampoo",
     "shampoo|scalp shampoo|clarifying shampoo|anti-dandruff shampoo",
     "hair", 0.9),
    ("product_type_std","conditioner","conditioner",
     "conditioner|leave-in conditioner|rinse-out conditioner|co-wash",
     "hair", 0.9),
    ("product_type_std","treatment","treatment",
     "treatment|leave-in|hair pack|hair mask|bond repair",
     "hair", 0.9),
    ("product_type_std","soap","soap",
     "soap|bar soap|body bar|beauty bar",
     "body", 0.7),
    ("product_type_std","oil","oil",
     "face oil|facial oil|hair oil|cleansing oil|body oil",
     "skincare", 0.8),

    # makeup/other
    ("product_type_std","powder","powder",
     "powder|loose powder|setting powder|finishing powder",
     "makeup", 0.7),
    ("product_type_std","makeup remover","makeup remover",
     "makeup remover|cleansing oil|micellar water|eye makeup remover",
     "makeup", 0.8),

    # 안전망
    ("product_type_std","other","other",
     "accessory|headband|towel|wipes|tool|applicator|case|pouch|comb|razor",
     "other", 0.4),
]

TARGET_ROWS = [
    ("target_audience","acne-prone","acne-prone",
     "acne|acne-prone|blemish|blemish-prone|anti-acne",
     "skin_type", 1.0),
    ("target_audience","dry","dry",
     "dry|dehydrated|very dry|extra dry",
     "skin_type", 0.9),
    ("target_audience","oily","oily",
     "oily|greasy|oil control|sebum control",
     "skin_type", 0.9),
    ("target_audience","sensitive","sensitive",
     "sensitive|hypoallergenic|for sensitive skin",
     "skin_type", 1.0),
    ("target_audience","combination","combination",
     "combination|combo",
     "skin_type", 0.8),
    ("target_audience","normal","normal",
     "normal|balanced",
     "skin_type", 0.6),

    ("target_audience","men","men",
     "men|male|for men|men's",
     "demographic", 0.6),
    ("target_audience","women","women",
     "women|female|for women|women's",
     "demographic", 0.6),
    ("target_audience","kids","kids",
     "kids|children|child|baby|infant|toddler",
     "demographic", 0.8),

    ("target_audience","curly","curly",
     "curly|curl|coily|kinky",
     "hair_type", 0.8),
    ("target_audience","color-treated","color-treated",
     "color-treated|colored hair|dyed hair|color care",
     "hair_type", 0.7),
    ("target_audience","aging","aging",
     "aging|mature|anti-aging|anti aging|wrinkle|fine lines",
     "skin_concern", 0.8),
]

SCENT_ROWS = [
    ("scent","unscented","unscented",
     "unscented|fragrance free|fragrance-free|no scent|no fragrance|odorless",
     "neutral", 0.6),
    ("scent","green tea","green tea",
     "green tea|matcha",
     "botanical", 0.5),
    ("scent","tea tree","tea tree",
     "tea tree|melaleuca",
     "botanical", 0.6),
    ("scent","lavender","lavender",
     "lavender",
     "botanical", 0.5),
    ("scent","rose","rose",
     "rose|rosa",
     "botanical", 0.5),
    ("scent","vanilla","vanilla",
     "vanilla|vanillin",
     "gourmand", 0.4),
    ("scent","coconut","coconut",
     "coconut|cocos nucifera",
     "gourmand", 0.4),
    ("scent","citrus","citrus",
     "citrus|orange|lemon|lime|grapefruit|bergamot|mandarin|yuzu",
     "fruit", 0.5),
    ("scent","mint","mint",
     "mint|peppermint|spearmint|menthol",
     "herb", 0.5),
    ("scent","aloe","aloe",
     "aloe|aloe vera",
     "botanical", 0.4),
    ("scent","apple","apple",
     "apple|pomme|malus",
     "fruit", 0.4),
    ("scent","berry","berry",
     "berry|berries|strawberry|blueberry|raspberry|cranberry|acai",
     "fruit", 0.4),
    ("scent","charcoal","charcoal",
     "charcoal|activated charcoal|bamboo charcoal",
     "other", 0.4),
]

rows = PT_ROWS + TARGET_ROWS + SCENT_ROWS
ontology_df = pd.DataFrame(rows, columns=["facet","value","canonical","aliases","group","default_weight"])

# pattern(정규식) 추가: aliases 파이프를 그대로 사용하되, 소문자 비교 권장
ontology_df["pattern"] = ontology_df["aliases"]

# ---------- 2) CSV 저장 ----------
save_path = os.path.join(BASE_DIR, "attr_ontology.csv")
ontology_df.to_csv(save_path, index=False)
print(f"[OK] attr_ontology.csv 저장: {save_path}  (rows={len(ontology_df):,})")

# ---------- 3) 커버리지/유효성 간단 점검 ----------
df = item_features_step3.copy()

def _split_set_col(s):
    if pd.isna(s): return []
    if isinstance(s, str):
        return [x.strip() for x in s.split(";") if x.strip()]
    return []

# (a) product_type_std_filled 커버리지 (other 제외 분포)
pt_series = df.get("product_type_std_filled", pd.Series(index=df.index)).astype(str)
pt_non_other = pt_series[pt_series!="other"]
pt_cov = round(100 * (pt_non_other.notna().sum()/len(df)), 1)

# (b) target_audience_set 토큰
toks_target = df.get("target_audience_set", pd.Series(index=df.index)).apply(_split_set_col)
all_target_tokens = pd.Series([t for lst in toks_target for t in lst])
target_cov = round(100 * (toks_target.apply(lambda x: len(x)>0).mean()), 1) if len(df)>0 else 0.0

# (c) scent_set 토큰
toks_scent = df.get("scent_set", pd.Series(index=df.index)).apply(_split_set_col)
all_scent_tokens = pd.Series([t for lst in toks_scent for t in lst])
scent_cov = round(100 * (toks_scent.apply(lambda x: len(x)>0).mean()), 1) if len(df)>0 else 0.0

# (d) 온톨로지 기준 라벨 목록
pt_canon = set(ontology_df.loc[ontology_df["facet"]=="product_type_std","canonical"])
tg_canon = set(ontology_df.loc[ontology_df["facet"]=="target_audience","canonical"])
sc_canon = set(ontology_df.loc[ontology_df["facet"]=="scent","canonical"])

# (e) 데이터의 실제 라벨이 온톨로지에 포함되는지(간단 교차)
pt_in_onto = pt_non_other.isin(pt_canon).mean() if len(pt_non_other)>0 else np.nan
tg_in_onto = all_target_tokens.isin(tg_canon).mean() if len(all_target_tokens)>0 else np.nan
sc_in_onto = all_scent_tokens.isin(sc_canon).mean() if len(all_scent_tokens)>0 else np.nan

print("\n=== CELL 10.0 REPORT — 온톨로지 생성/점검 ===")
print(f"- product_type_std_filled (other 제외) 커버리지: {pt_cov}%  | 온톨로지 포함비율: {round(100*pt_in_onto,1) if pd.notna(pt_in_onto) else 'NA'}%")
print(f"- target_audience_set 커버리지: {target_cov}%            | 온톨로지 포함비율: {round(100*tg_in_onto,1) if pd.notna(tg_in_onto) else 'NA'}%")
print(f"- scent_set 커버리지: {scent_cov}%                      | 온톨로지 포함비율: {round(100*sc_in_onto,1) if pd.notna(sc_in_onto) else 'NA'}%")

print("\n[attr_ontology 미리보기 — 상위 12]")
display(ontology_df.head(12))

# (선택) facet별 카운트
print("\n[facet별 항목 수]")
display(ontology_df["facet"].value_counts().rename_axis("facet").to_frame("count"))


[OK] attr_ontology.csv 저장: /content/attr_ontology.csv  (rows=40)

=== CELL 10.0 REPORT — 온톨로지 생성/점검 ===
- product_type_std_filled (other 제외) 커버리지: 100.0%  | 온톨로지 포함비율: 0.0%
- target_audience_set 커버리지: 43.8%            | 온톨로지 포함비율: 89.0%
- scent_set 커버리지: 11.7%                      | 온톨로지 포함비율: 70.4%

[attr_ontology 미리보기 — 상위 12]


,facet,value,canonical,aliases,group,default_weight,pattern
0,product_type_std,cleanser,cleanser,cleanser|face wash|facial wash|foaming wash|foam wash|gel cleanser|cream cleanser|micellar gel,skincare,1.0,cleanser|face wash|facial wash|foaming wash|foam wash|gel cleanser|cream cleanser|micellar gel
1,product_type_std,toner,toner,toner|skin softener|skin conditioner|lotion toner,skincare,0.9,toner|skin softener|skin conditioner|lotion toner
2,product_type_std,serum,serum,serum|ampoule|ampule|booster serum|essence serum,skincare,1.0,serum|ampoule|ampule|booster serum|essence serum
3,product_type_std,essence,essence,essence|treatment essence|first essence,skincare,0.9,essence|treatment essence|first essence
4,product_type_std,moisturizer,moisturizer,moisturizer|moisturiser|lotion|face cream|gel cream|balm|emulsion,skincare,0.9,moisturizer|moisturiser|lotion|face cream|gel cream|balm|emulsion
5,product_type_std,mask,mask,mask|sheet mask|sleeping pack|sleeping mask|mud mask|clay mask,skincare,0.8,mask|sheet mask|sleeping pack|sleeping mask|mud mask|clay mask
6,product_type_std,sunscreen,sunscreen,sunscreen|sun screen|sunblock|spf|uv defense|uva|uvb,skincare,1.0,sunscreen|sun screen|sunblock|spf|uv defense|uva|uvb
7,product_type_std,shampoo,shampoo,shampoo|scalp shampoo|clarifying shampoo|anti-dandruff shampoo,hair,0.9,shampoo|scalp shampoo|clarifying shampoo|anti-dandruff shampoo
8,product_type_std,conditioner,conditioner,conditioner|leave-in conditioner|rinse-out conditioner|co-wash,hair,0.9,conditioner|leave-in conditioner|rinse-out conditioner|co-wash
9,product_type_std,treatment,treatment,treatment|leave-in|hair pack|hair mask|bond repair,hair,0.9,treatment|leave-in|hair pack|hair mask|bond repair



[facet별 항목 수]


,count
facet,
product_type_std,15
scent,13
target_audience,12


In [26]:
# ==========================================================
# CELL 11 / item_scoring_table 구축 + 속성별 점수 컬럼 생성
# 목적:
#   - 프론트 개발자가 바로 사용할 수 있는 "추천용 CSV" 구조 만들기
#   - 각 상품별로 review_score / popularity_score / price/volume 점수 제공
#   - 나중에 프론트에서 가중합으로 최종 추천 점수 계산 가능
# 산출:
#   - item_scoring_table DataFrame
# ==========================================================
import numpy as np
import pandas as pd
from IPython.display import display

# ---------- 0) 선행 산출물 체크 ----------
if "item_features_step2" not in globals():
    raise RuntimeError("item_features_step2 가 없습니다. CELL 9.5(safe)를 먼저 실행하세요.")

df = item_features_step2.copy()

# ---------- 1) 문자열/수치 가드 ----------
def s_guard(s: pd.Series) -> pd.Series:
    return s.astype(object).where(~s.isna(), "").astype(str)

def n_guard(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce")

# 문자열 컬럼 보정
for c in ["parent_asin","title","brand_view","brand_clean_v2",
          "product_type_std_filled","target_audience_set","scent_set"]:
    if c not in df.columns:
        df[c] = ""
    df[c] = s_guard(df[c])

# product_type_std_filled 없거나 비어 있으면 other로 채우기
if "product_type_std_filled" not in df.columns:
    df["product_type_std_filled"] = "other"
df["product_type_std_filled"] = (
    df["product_type_std_filled"]
      .fillna("")
      .astype(str)
      .replace("", "other")
)

# 수치 컬럼 보정
for c in [
    "price_num","price_winsor","price_norm_0_1",
    "volume_ml","volume_winsor","volume_norm_0_1",
    "rating_cnt","rating_cnt_log","pop_log_winsor","pop_log_norm",
    "review_signal"
]:
    if c in df.columns:
        df[c] = n_guard(df[c])

# ---------- 2) price_bin / volume_bin 재생성 ----------
# 가격 구간 (프론트 필터용, 이전과 동일 정책)
if "price_num" in df.columns:
    edges  = [0,10,20,35,60,100,np.inf]
    labels = ["<=10","10-20","20-35","35-60","60-100","100+"]
    df["price_bin"] = pd.cut(
        df["price_num"],
        bins=edges,
        labels=labels,
        include_lowest=True,
        right=False
    )
else:
    df["price_bin"] = np.nan

# 용량 구간
if "volume_ml" in df.columns:
    edges  = [0,15,30,75,150,300,600,np.inf]
    labels = ["<15ml","15-30","30-75","75-150","150-300","300-600","600+"]
    df["volume_bin"] = pd.cut(
        df["volume_ml"],
        bins=edges,
        labels=labels,
        include_lowest=True,
        right=False
    )
else:
    df["volume_bin"] = np.nan

# ---------- 3) 속성별 점수 컬럼 생성 ----------
# (1) 리뷰 품질 점수: review_signal 그대로 사용 (0~1)
if "review_signal" in df.columns:
    df["review_score"] = df["review_signal"].fillna(0).clip(0, 1)
else:
    df["review_score"] = 0.0

# (2) 인기 점수: pop_log_norm (0~1)
if "pop_log_norm" in df.columns:
    df["popularity_score"] = df["pop_log_norm"].fillna(0).clip(0, 1)
else:
    df["popularity_score"] = 0.0

# (3) 가격 점수:
#   - price_norm_0_1: 비쌀수록 1에 가까움
#   - cheap_score: 쌀수록 점수 높게 (1 - price_norm)
if "price_norm_0_1" in df.columns:
    df["cheap_score"] = np.where(
        df["price_norm_0_1"].notna(),
        1.0 - df["price_norm_0_1"].clip(0, 1),
        np.nan
    )
else:
    df["cheap_score"] = np.nan

# (4) 용량 점수:
#   - volume_norm_0_1: 용량 클수록 1
#   - small_volume_score: 소용량 선호 시 사용 (1 - volume_norm)
if "volume_norm_0_1" in df.columns:
    df["big_volume_score"] = df["volume_norm_0_1"].clip(0, 1)
    df["small_volume_score"] = np.where(
        df["volume_norm_0_1"].notna(),
        1.0 - df["volume_norm_0_1"].clip(0, 1),
        np.nan
    )
else:
    df["big_volume_score"] = np.nan
    df["small_volume_score"] = np.nan

# ---------- 4) 커버리지 플래그 ----------
df["has_price"]   = df["price_num"].notna()
df["has_volume"]  = df["volume_ml"].notna()
df["has_review"]  = df["review_score"].notna() & (df["review_score"] > 0)
df["has_target"]  = df["target_audience_set"].str.strip().ne("")
df["has_scent"]   = df["scent_set"].str.strip().ne("")
df["has_brand"]   = df["brand_view"].str.strip().ne("")

# 추천 노출 최소 조건 (예시): 브랜드 + 제품타입 + 리뷰점수
df["minimal_ok"] = (
    df["has_brand"] &
    df["product_type_std_filled"].str.strip().ne("") &
    df["review_score"].notna()
)

# ---------- 5) 데모용 "기본 점수" (default_score_demo) ----------
# 기본 아이디어:
#   score = 0.6 * review_score
#         + 0.2 * popularity_score
#         + 0.1 * cheap_score
#         + 0.1 * big_volume_score
#   단, 값이 NaN인 항목은 그 항목의 weight를 빼고 나머지 weight로 리스케일

w_review = 0.6
w_pop    = 0.2
w_price  = 0.1
w_vol    = 0.1

components = {
    "review_score":   (df["review_score"],      w_review),
    "popularity_score": (df["popularity_score"], w_pop),
    "cheap_score":    (df["cheap_score"],       w_price),
    "big_volume_score": (df["big_volume_score"], w_vol),
}

weighted_sum = pd.Series(0.0, index=df.index)
weight_sum   = pd.Series(0.0, index=df.index)

for col_name, (series, w) in components.items():
    s = pd.to_numeric(series, errors="coerce")
    mask = s.notna()
    weighted_sum += s.fillna(0.0) * w
    weight_sum   += mask.astype(float) * w

df["default_score_demo"] = np.where(
    weight_sum > 0,
    weighted_sum / weight_sum,
    np.nan
)

# ---------- 6) 최종 item_scoring_table 구성 ----------
cols_for_scoring = [
    # 기본 식별자 / 표시용 텍스트
    "parent_asin","title","brand_view","brand_clean_v2",
    "product_type_std_filled","target_audience_set","scent_set",
    # 가격/용량 정보
    "price_num","price_bin","price_norm_0_1","cheap_score",
    "volume_ml","volume_bin","volume_norm_0_1","big_volume_score","small_volume_score",
    # 리뷰/인기
    "avg_rating" if "avg_rating" in df.columns else None,
    "rating_cnt","pop_log_norm","review_score","popularity_score",
    # 커버리지 플래그
    "has_price","has_volume","has_review","has_target","has_scent","has_brand","minimal_ok",
    # 데모용 종합 점수
    "default_score_demo",
]

# None 제거
cols_for_scoring = [c for c in cols_for_scoring if c in df.columns]

item_scoring_table = df[cols_for_scoring].copy()

# ---------- 7) 리포트 ----------
print("=== CELL 11 REPORT — item_scoring_table 구축 ===")
print(f"- 총 상품 수: {len(item_scoring_table):,}")
print(f"- minimal_ok=True 비율: {item_scoring_table['minimal_ok'].mean()*100:.1f}%")

if "default_score_demo" in item_scoring_table.columns:
    s = pd.to_numeric(item_scoring_table["default_score_demo"], errors="coerce").dropna()
    if not s.empty:
        print(f"- default_score_demo 분포: mean={s.mean():.3f} / p50={s.quantile(0.5):.3f} / p90={s.quantile(0.9):.3f} / max={s.max():.3f}")
    else:
        print("- default_score_demo: 유효값이 없습니다.")

print("\n[item_scoring_table 미리보기 — 상위 8]")
display(item_scoring_table.head(8))


=== CELL 11 REPORT — item_scoring_table 구축 ===
- 총 상품 수: 12,402
- minimal_ok=True 비율: 100.0%
- default_score_demo 분포: mean=0.393 / p50=0.371 / p90=0.523 / max=0.894

[item_scoring_table 미리보기 — 상위 8]


,parent_asin,title,brand_view,brand_clean_v2,product_type_std_filled,target_audience_set,scent_set,price_num,price_bin,price_norm_0_1,cheap_score,volume_ml,volume_bin,volume_norm_0_1,big_volume_score,small_volume_score,rating_cnt,pop_log_norm,review_score,popularity_score,has_price,has_volume,has_review,has_target,has_scent,has_brand,minimal_ok,default_score_demo
0,B076WQZGPM,"Yes to Tomatoes Detoxifying Charcoal Cleanser (Pack of 2) with Charcoal Powder, Tomato Fruit Extract, and Gingko Biloba Leaf Extract, 5 fl. oz.",Yes To,Yes To,cleanser,acne-prone,,NaN,NaN,NaN,NaN,295.74,150-300,0.199045,0.199045,0.800955,1,0.000000,0.441439,0.000000,False,True,True,True,False,True,True,0.316409
1,B07G9GWFSM,Lurrose 100Pcs Full Cover Fake Toenails Artificial Transparent Nail Tips Nail Art for DIY,Lurrose,Lurrose,other,,,6.99,<=10,0.021123,0.978877,NaN,NaN,NaN,NaN,NaN,1,0.000000,0.400530,0.000000,True,False,True,False,False,True,True,0.375784
2,B08DNQTTQK,50 Pieces False Eyelash Packaging Box Empty Eyelash Box Plastic Eyelash Storage Case with Glitter Paper and Clear Tray for Women Girls Eyelash (Holographic),Maitys,Maitys,other,women,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,0.177188,0.475244,0.177188,False,False,True,True,False,True,True,0.400730
3,B01ERJEGS6,Gold extatic Musk EDT 90ml,Balmain,Balmain,other,,,86.95,60-100,0.473607,0.526393,90.00,75-150,0.059740,0.059740,0.940260,2,0.103649,0.435771,0.103649,True,True,True,False,False,True,True,0.340806
4,B08P7LXKP7,4 Pieces Satin Bonnet Adjustable Sleep Cap Double Layer Printed Hair Bonnet Large Reversible Sleeping Silk Night Hair Cap for Women Natural Curly Hair (Flor...,Geyoga,Geyoga,other,curly,floral,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,0.234230,0.485350,0.234230,False,False,True,True,True,True,True,0.422570
5,B01555WAOS,Avon Big & False Lash Mascara - Brown/Black by Big & False Lash,Avon,Avon,other,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,0.103649,0.398271,0.103649,False,False,True,False,False,True,True,0.324616
6,B06XJZ7955,Brand New Headrang Face line Contour V-line Machine,Unknown,,other,,,79.50,60-100,0.431448,0.568552,NaN,NaN,NaN,NaN,NaN,4,0.234230,0.442492,0.234230,True,False,True,False,False,True,True,0.410219
7,B07FVZVQKV,"Loose Powder, 1.5oz Bottle Loose Face Powder, Translucent Tone Loose Face Powder, for Setting Makeup or as Foundation, Lightweight, Long Lasting",Getmore7,Getmore7,other,,,NaN,NaN,NaN,NaN,44.36,30-75,0.028837,0.028837,0.971163,4,0.234230,0.485391,0.234230,False,True,True,False,False,True,True,0.378849


In [27]:
# ==========================================================
# CELL 12 / CSV 내보내기 + 컬럼 설명 리포트
# 목적:
#   - 프론트엔드 개발자가 바로 사용할 수 있는 추천용 CSV 파일 생성
#   - 전체 상품 / minimal_ok 상품 각각 저장
#   - 어떤 컬럼이 어떤 의미인지 간단 설명 리포트 출력
# 전제:
#   - CELL 11 실행 완료 → item_scoring_table 존재
# 산출:
#   - {BASE_DIR}/beauty_item_scoring_all.csv
#   - {BASE_DIR}/beauty_item_scoring_minimal_ok.csv
# ==========================================================
import os
import numpy as np
import pandas as pd
from IPython.display import display

if "item_scoring_table" not in globals():
    raise RuntimeError("item_scoring_table 이 없습니다. CELL 11까지 먼저 실행하세요.")

df = item_scoring_table.copy()

# ---------- 1) 저장 경로/파일명 설정 ----------
OUTPUT_DIR = BASE_DIR
os.makedirs(OUTPUT_DIR, exist_ok=True)

all_path = os.path.join(OUTPUT_DIR, "beauty_item_scoring_all.csv")
ok_path  = os.path.join(OUTPUT_DIR, "beauty_item_scoring_minimal_ok.csv")

# ---------- 2) minimal_ok 필터 ----------
if "minimal_ok" in df.columns:
    df_ok = df[df["minimal_ok"]].copy()
else:
    # minimal_ok 없으면 전체를 그대로 사용
    df_ok = df.copy()

# ---------- 3) CSV 저장 ----------
df.to_csv(all_path, index=False)
df_ok.to_csv(ok_path, index=False)

print("=== CELL 12 REPORT — CSV 내보내기 완료 ===")
print(f"- 전체 상품 CSV : {all_path}  (rows={len(df):,})")
print(f"- minimal_ok CSV: {ok_path}  (rows={len(df_ok):,})")

# ---------- 4) 컬럼 그룹별 설명 정리 ----------
meta_cols = [c for c in [
    "parent_asin","title","brand_view","brand_clean_v2",
    "product_type_std_filled","target_audience_set","scent_set"
] if c in df.columns]

price_cols = [c for c in [
    "price_num","price_bin","price_norm_0_1","cheap_score"
] if c in df.columns]

volume_cols = [c for c in [
    "volume_ml","volume_bin","volume_norm_0_1","big_volume_score","small_volume_score"
] if c in df.columns]

review_cols = [c for c in [
    "avg_rating","rating_cnt","pop_log_norm","review_score","popularity_score"
] if c in df.columns]

flag_cols = [c for c in [
    "has_price","has_volume","has_review","has_target","has_scent","has_brand","minimal_ok"
] if c in df.columns]

score_cols = [c for c in [
    "default_score_demo"
] if c in df.columns]

print("\n[컬럼 그룹 요약]")
print("- 메타 정보(meta):", ", ".join(meta_cols) or "(없음)")
print("- 가격 관련(price):", ", ".join(price_cols) or "(없음)")
print("- 용량 관련(volume):", ", ".join(volume_cols) or "(없음)")
print("- 리뷰/인기(review):", ", ".join(review_cols) or "(없음)")
print("- 커버리지 플래그(flags):", ", ".join(flag_cols) or "(없음)")
print("- 종합 점수(score):", ", ".join(score_cols) or "(없음)")

# ---------- 5) 예시 몇 줄 보여주기 ----------
print("\n[item_scoring_table 샘플 5행]")
sample_cols = meta_cols + price_cols + volume_cols + review_cols + score_cols
sample_cols = list(dict.fromkeys(sample_cols))  # 중복 제거
display(df[sample_cols].head(5))


=== CELL 12 REPORT — CSV 내보내기 완료 ===
- 전체 상품 CSV : /content/beauty_item_scoring_all.csv  (rows=12,402)
- minimal_ok CSV: /content/beauty_item_scoring_minimal_ok.csv  (rows=12,402)

[컬럼 그룹 요약]
- 메타 정보(meta): parent_asin, title, brand_view, brand_clean_v2, product_type_std_filled, target_audience_set, scent_set
- 가격 관련(price): price_num, price_bin, price_norm_0_1, cheap_score
- 용량 관련(volume): volume_ml, volume_bin, volume_norm_0_1, big_volume_score, small_volume_score
- 리뷰/인기(review): rating_cnt, pop_log_norm, review_score, popularity_score
- 커버리지 플래그(flags): has_price, has_volume, has_review, has_target, has_scent, has_brand, minimal_ok
- 종합 점수(score): default_score_demo

[item_scoring_table 샘플 5행]


,parent_asin,title,brand_view,brand_clean_v2,product_type_std_filled,target_audience_set,scent_set,price_num,price_bin,price_norm_0_1,cheap_score,volume_ml,volume_bin,volume_norm_0_1,big_volume_score,small_volume_score,rating_cnt,pop_log_norm,review_score,popularity_score,default_score_demo
0,B076WQZGPM,"Yes to Tomatoes Detoxifying Charcoal Cleanser (Pack of 2) with Charcoal Powder, Tomato Fruit Extract, and Gingko Biloba Leaf Extract, 5 fl. oz.",Yes To,Yes To,cleanser,acne-prone,,NaN,NaN,NaN,NaN,295.74,150-300,0.199045,0.199045,0.800955,1,0.000000,0.441439,0.000000,0.316409
1,B07G9GWFSM,Lurrose 100Pcs Full Cover Fake Toenails Artificial Transparent Nail Tips Nail Art for DIY,Lurrose,Lurrose,other,,,6.99,<=10,0.021123,0.978877,NaN,NaN,NaN,NaN,NaN,1,0.000000,0.400530,0.000000,0.375784
2,B08DNQTTQK,50 Pieces False Eyelash Packaging Box Empty Eyelash Box Plastic Eyelash Storage Case with Glitter Paper and Clear Tray for Women Girls Eyelash (Holographic),Maitys,Maitys,other,women,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,0.177188,0.475244,0.177188,0.400730
3,B01ERJEGS6,Gold extatic Musk EDT 90ml,Balmain,Balmain,other,,,86.95,60-100,0.473607,0.526393,90.00,75-150,0.059740,0.059740,0.940260,2,0.103649,0.435771,0.103649,0.340806
4,B08P7LXKP7,4 Pieces Satin Bonnet Adjustable Sleep Cap Double Layer Printed Hair Bonnet Large Reversible Sleeping Silk Night Hair Cap for Women Natural Curly Hair (Flor...,Geyoga,Geyoga,other,curly,floral,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,0.234230,0.485350,0.234230,0.422570
